**MobileNetV3-Large — ONNX CPU Optimization Experiments**
This notebook evaluates ONNX export, classification accuracy, CPU inference performance, prediction consistency, and deployment packaging.

**Validated result**: 96.94% validation accuracy (793/818); paired CPU benchmark: PyTorch 16.792 ms/image vs ONNX 7.077 ms/image (2.37× speedup). These measurements were obtained on Colab CPU, not the instructor’s final evaluation hardware.

**Setup**: Mount Google Drive in Colab before running the backup restoration cell. The backup is expected at /content/drive/MyDrive/Flower_Competition/FlowerCompetition_FullBackup.zip.

**1. Restore Project Backup**
Restore the previously trained MobileNetV3-Large checkpoints and supporting project files from Google Drive. No model retraining is required.

In [ ]:
from pathlib import Path
import zipfile

backup_path = Path(
    "/content/drive/MyDrive/Flower_Competition/FlowerCompetition_FullBackup.zip"
)

assert backup_path.is_file(), "Backup ZIP not found!"

print("Backup found:", backup_path.name)
print("Backup size:", round(backup_path.stat().st_size / 1024**2, 2), "MiB")

with zipfile.ZipFile(backup_path, "r") as z:
    print("Files in ZIP:", len(z.namelist()))
    z.extractall("/content")

print("Backup extraction completed.")

Backup found: FlowerCompetition_FullBackup.zip
Backup size: 701.9 MiB
Files in ZIP: 8522
Backup extraction completed.


**2. Checkpoint and Project File Verification**
Verify that the trained MobileNetV3-Large checkpoints, class mappings, and inference scripts have been restored successfully. This step checks file availability without modifying or retraining the models.

In [ ]:
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
DATA_DIR = PROJECT / "data"

required_files = [
    "mobilenetv3_finetuned_best.pth",
    "mobilenetv3_fp16.pth",
    "mobilenetv3_int8_experimental.pth",
    "class_to_idx.json",
    "evaluate.py",
    "predict.py",
]

print("Project directory exists:", PROJECT.is_dir())
print("\n=== Restored Files ===")

for filename in required_files:
    matches = list(PROJECT.rglob(filename)) if PROJECT.is_dir() else []

    if matches:
        for path in matches:
            size = path.stat().st_size / (1024**2)
            print(f"FOUND | {filename} | {size:.2f} MiB")
            print(f"      {path}")
    else:
        print(f"MISSING | {filename}")

print("\nData directory exists:", DATA_DIR.is_dir())

Project directory exists: True

=== Restored Files ===
FOUND | mobilenetv3_finetuned_best.pth | 16.73 MiB
      /content/FlowerCompetition/mobilenetv3_finetuned_best.pth
FOUND | mobilenetv3_fp16.pth | 8.42 MiB
      /content/FlowerCompetition/mobilenetv3_fp16.pth
FOUND | mobilenetv3_int8_experimental.pth | 4.61 MiB
      /content/FlowerCompetition/mobilenetv3_int8_experimental.pth
FOUND | class_to_idx.json | 0.00 MiB
      /content/FlowerCompetition/class_to_idx.json
FOUND | class_to_idx.json | 0.00 MiB
      /content/FlowerCompetition/deployment_onnx/class_to_idx.json
FOUND | evaluate.py | 0.01 MiB
      /content/FlowerCompetition/evaluate.py
FOUND | predict.py | 0.01 MiB
      /content/FlowerCompetition/predict.py

Data directory exists: True


**3. CPU Benchmark Preparation**
Inspect the existing evaluation script and dataset structure before benchmarking CPU inference performance. The competition evaluates inference latency on a CPU-only system, so all comparisons must use consistent preprocessing, input resolution, batch size, and timing conditions.

In [ ]:
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")

print("=== evaluate.py ===")
print((PROJECT / "evaluate.py").read_text(encoding="utf-8"))

print("\n=== Data Directory ===")
for item in sorted((PROJECT / "data").iterdir()):
    print(item.name, "|", "DIR" if item.is_dir() else "FILE")

=== evaluate.py ===

import argparse
import json
import math
import statistics
import time
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v3_large


def load_model(checkpoint_path, device):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Checkpoint bulunamadı: {checkpoint_path}"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    architecture = checkpoint.get(
        "architecture", "mobilenet_v3_large"
    )

    if architecture != "mobilenet_v3_large":
        raise ValueError("Desteklenmeyen model mimarisi")

    class_to_idx = checkpoint["class_to_idx"]
    num_classes = len(class_to_idx)

    if sorted(class_to_idx.values()) != list(range(num_classes)):


**4. Baseline CPU Performance Evaluation**
Evaluate the original MobileNetV3-Large FP32 checkpoint on the validation dataset using CPU inference. Record classification accuracy, inference latency, and checkpoint size as baseline metrics for subsequent optimization experiments.

The reported latency measures model forward execution only and excludes image loading and preprocessing.

In [ ]:
import sys
import torch
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
sys.path.insert(0, str(PROJECT))

from evaluate import evaluate

# Force CPU evaluation, even if a GPU is available.
original_cuda_check = torch.cuda.is_available

try:
    torch.cuda.is_available = lambda: False

    baseline_results = evaluate(
        checkpoint_path=PROJECT / "mobilenetv3_finetuned_best.pth",
        data_dir=PROJECT / "data" / "valid",
        output_path=PROJECT / "results" / "cpu_fp32_baseline.json",
        batch_size=32,
        use_tta=False
    )
finally:
    torch.cuda.is_available = original_cuda_check

print("\n=== CPU BASELINE SUMMARY ===")
print("Device:", baseline_results["device"])
print("Accuracy:", baseline_results["accuracy_percent"])
print("Mean latency:", baseline_results["inference"]["mean_ms"], "ms/image")
print("Model size:", baseline_results["model_size_mib"], "MiB")
print("CPU threads:", torch.get_num_threads())


=== MODEL EVALUATION V3 ===
Checkpoint: /content/FlowerCompetition/mobilenetv3_finetuned_best.pth
Precision: FP32
TTA: Disabled
Accuracy: 96.94%
Correct: 793/818
Macro Precision: 0.9674
Macro Recall: 0.9620
Macro F1: 0.9607
Model size: 16.73 MiB
Mean inference: 16.478 ms
Median inference: 16.131 ms
Std inference: 1.197 ms
P95 inference: 18.805 ms
Median run mean: 16.436 ms
Device: cpu

Top 5 confusions:
  42 -> 43: 2
  18 -> 83: 1
  39 -> 4: 1
  4 -> 3: 1
  4 -> 88: 1

Results saved: /content/FlowerCompetition/results/cpu_fp32_baseline.json

=== CPU BASELINE SUMMARY ===
Device: cpu
Accuracy: 96.9437652811736
Mean latency: 16.477695053329928 ms/image
Model size: 16.731240272521973 MiB
CPU threads: 1


5. ONNX and ONNX Runtime Installation
Install the dependencies required to export the trained MobileNetV3-Large model to ONNX and evaluate its CPU inference performance.

The original PyTorch checkpoint and baseline results remain unchanged.

In [ ]:
%pip install -q onnx onnxruntime onnxscript

import onnx
import onnxruntime as ort
import onnxscript

print("=== Installation Verification ===")
print("ONNX:", onnx.__version__)
print("ONNX Runtime:", ort.__version__)
print("ONNX Script:", onnxscript.__version__)
print("Available providers:", ort.get_available_providers())

=== Installation Verification ===
ONNX: 1.23.2
ONNX Runtime: 1.31.0
ONNX Script: 0.7.2
Available providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


**6. MobileNetV3-Large ONNX Export**
Export the trained FP32 MobileNetV3-Large checkpoint to ONNX format for CPU inference optimization.

The exported model uses a fixed input shape of (1, 3, 224, 224), matching single-image inference. The original PyTorch checkpoint remains unchanged.

ONNX output equivalence and validation accuracy will be verified before performance benchmarking.

In [ ]:
import torch
import onnx
from pathlib import Path
from evaluate import load_model

PROJECT = Path("/content/FlowerCompetition")
ONNX_DIR = PROJECT / "onnx"
ONNX_DIR.mkdir(exist_ok=True)

checkpoint_path = PROJECT / "mobilenetv3_finetuned_best.pth"
onnx_path = ONNX_DIR / "mobilenetv3_fp32.onnx"

device = torch.device("cpu")

model, class_to_idx, use_fp16 = load_model(
    checkpoint_path,
    device
)

model.eval()

dummy_input = torch.randn(
    1, 3, 224, 224,
    dtype=torch.float32
)

torch.onnx.export(
    model,
    (dummy_input,),
    str(onnx_path),
    input_names=["input"],
    output_names=["logits"],
    opset_version=18,
    dynamo=True
)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

print("=== ONNX Export ===")
print("Export successful:", onnx_path.exists())
print("ONNX model:", onnx_path)
print("ONNX size:", round(onnx_path.stat().st_size / 1024**2, 2), "MiB")
print("ONNX validation: PASSED")
print("Number of classes:", len(class_to_idx))

[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
=== ONNX Export ===
Export successful: True
ONNX model: /content/FlowerCompetition/onnx/mobilenetv3_fp32.onnx
ONNX size: 0.32 MiB
ONNX validation: PASSED
Number of classes: 102


**7. ONNX Model Integrity and Size Verification**
Inspect the exported ONNX model and its external weight files. Verify that ONNX Runtime can load the model and calculate the total storage size required for inference.

This step does not modify the original PyTorch checkpoint or perform model training.

In [ ]:
from pathlib import Path
import onnxruntime as ort

onnx_path = Path(
    "/content/FlowerCompetition/onnx/mobilenetv3_fp32.onnx"
)

print("=== ONNX Files ===")

total_bytes = 0

for file in sorted(onnx_path.parent.iterdir()):
    if file.is_file():
        size = file.stat().st_size
        total_bytes += size
        print(f"{file.name}: {size / 1024**2:.3f} MiB")

print(f"\nTotal directory size: {total_bytes / 1024**2:.3f} MiB")

session = ort.InferenceSession(
    str(onnx_path),
    providers=["CPUExecutionProvider"]
)

print("\n=== ONNX Runtime Verification ===")
print("Model loaded: SUCCESS")
print("Input:", [(x.name, x.shape, x.type) for x in session.get_inputs()])
print("Output:", [(x.name, x.shape, x.type) for x in session.get_outputs()])
print("Provider:", session.get_providers())

=== ONNX Files ===
mobilenetv3_fp32.onnx: 0.325 MiB
mobilenetv3_fp32.onnx.data: 16.480 MiB

Total directory size: 16.805 MiB

=== ONNX Runtime Verification ===
Model loaded: SUCCESS
Input: [('input', [1, 3, 224, 224], 'tensor(float)')]
Output: [('logits', [1, 102], 'tensor(float)')]
Provider: ['CPUExecutionProvider']


8. PyTorch–ONNX Prediction Equivalence
Compare the original PyTorch FP32 model with the exported ONNX model using validation images.

Measure the maximum absolute difference between model outputs and verify prediction agreement. This preliminary test checks conversion consistency before full validation accuracy and CPU latency benchmarking.

In [ ]:
import numpy as np
import torch
import onnxruntime as ort

from pathlib import Path
from torchvision import datasets
from evaluate import load_model, get_transform

PROJECT = Path("/content/FlowerCompetition")

# Load original PyTorch model on CPU
model, class_to_idx, _ = load_model(
    PROJECT / "mobilenetv3_finetuned_best.pth",
    torch.device("cpu")
)

# Load ONNX model on CPU
session = ort.InferenceSession(
    str(PROJECT / "onnx/mobilenetv3_fp32.onnx"),
    providers=["CPUExecutionProvider"]
)

# Load validation images using identical preprocessing
dataset = datasets.ImageFolder(
    str(PROJECT / "data/valid"),
    transform=get_transform()
)

num_samples = min(100, len(dataset))

max_differences = []
matching_predictions = 0

with torch.inference_mode():
    for i in range(num_samples):
        image, _ = dataset[i]
        image = image.unsqueeze(0)

        pytorch_logits = model(image).numpy()

        onnx_logits = session.run(
            ["logits"],
            {"input": image.numpy()}
        )[0]

        max_differences.append(
            np.max(np.abs(pytorch_logits - onnx_logits))
        )

        if np.argmax(pytorch_logits) == np.argmax(onnx_logits):
            matching_predictions += 1

print("=== PyTorch vs ONNX ===")
print("Images compared:", num_samples)
print("Matching predictions:", matching_predictions)
print("Prediction agreement:",
      f"{100 * matching_predictions / num_samples:.2f}%")
print("Maximum absolute logit difference:",
      f"{max(max_differences):.8f}")
print("Mean maximum logit difference:",
      f"{np.mean(max_differences):.8f}")

=== PyTorch vs ONNX ===
Images compared: 100
Matching predictions: 100
Prediction agreement: 100.00%
Maximum absolute logit difference: 0.00005722
Mean maximum logit difference: 0.00002662


**9. Full ONNX Validation and CPU Benchmark**
Evaluate the exported ONNX model on all 818 validation images and compare its CPU inference latency with the PyTorch FP32 baseline.

Use batch size 1, one CPU inference thread, 30 warm-up iterations, and three timing repetitions. Timing excludes image loading and preprocessing.

The results are Colab CPU measurements and do not represent the instructor's final hardware benchmark.

In [ ]:
import json
import time
import statistics
import numpy as np
import torch
import onnxruntime as ort

from pathlib import Path
from torchvision import datasets
from evaluate import get_transform

PROJECT = Path("/content/FlowerCompetition")

# Match the PyTorch baseline's CPU thread count
torch.set_num_threads(1)

options = ort.SessionOptions()
options.intra_op_num_threads = 1
options.inter_op_num_threads = 1
options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

session = ort.InferenceSession(
    str(PROJECT / "onnx/mobilenetv3_fp32.onnx"),
    sess_options=options,
    providers=["CPUExecutionProvider"]
)

dataset = datasets.ImageFolder(
    str(PROJECT / "data/valid"),
    transform=get_transform()
)

# Restore checkpoint class mapping
class_to_idx = json.loads(
    (PROJECT / "class_to_idx.json").read_text(encoding="utf-8")
)

images = []
labels = []

for image, label in dataset:
    images.append(
        np.ascontiguousarray(image.unsqueeze(0).numpy())
    )
    labels.append(class_to_idx[dataset.classes[label]])

# Full validation accuracy
predictions = []

for image in images:
    logits = session.run(["logits"], {"input": image})[0]
    predictions.append(int(np.argmax(logits)))

correct = sum(
    pred == label
    for pred, label in zip(predictions, labels)
)

accuracy = 100 * correct / len(labels)

# Warm-up
for i in range(30):
    session.run(["logits"], {"input": images[i]})

# Benchmark: same sample-selection rule as evaluate.py
indices = np.linspace(0, len(images) - 1, 100).astype(int)

run_means = []
all_times = []

for run in range(3):
    times = []

    for idx in indices:
        start = time.perf_counter()

        session.run(
            ["logits"],
            {"input": images[idx]}
        )

        elapsed_ms = (time.perf_counter() - start) * 1000
        times.append(elapsed_ms)

    run_means.append(statistics.mean(times))
    all_times.extend(times)

onnx_mean_ms = statistics.mean(all_times)

# Save results
results = {
    "model": "MobileNetV3-Large ONNX FP32",
    "device": "cpu",
    "cpu_threads": 1,
    "accuracy_percent": accuracy,
    "correct": correct,
    "total": len(labels),
    "mean_latency_ms": onnx_mean_ms,
    "median_latency_ms": statistics.median(all_times),
    "run_means_ms": run_means,
    "num_measurements": len(all_times),
    "batch_size": 1,
    "scope": "model_forward_only"
}

output_path = PROJECT / "results/onnx_fp32_cpu.json"
output_path.parent.mkdir(exist_ok=True)

output_path.write_text(
    json.dumps(results, indent=4),
    encoding="utf-8"
)

print("=== ONNX CPU EVALUATION ===")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Correct: {correct}/{len(labels)}")
print(f"Mean inference: {onnx_mean_ms:.3f} ms/image")
print(f"Median inference: {statistics.median(all_times):.3f} ms")
print("CPU threads: 1")
print("Results saved:", output_path)

print("\n=== BASELINE COMPARISON ===")
print("PyTorch FP32 accuracy: 96.94%")
print("PyTorch FP32 latency: 16.945 ms/image")
print(f"ONNX FP32 accuracy: {accuracy:.2f}%")
print(f"ONNX FP32 latency: {onnx_mean_ms:.3f} ms/image")

=== ONNX CPU EVALUATION ===
Accuracy: 96.94%
Correct: 793/818
Mean inference: 7.246 ms/image
Median inference: 7.151 ms
CPU threads: 1
Results saved: /content/FlowerCompetition/results/onnx_fp32_cpu.json

=== BASELINE COMPARISON ===
PyTorch FP32 accuracy: 96.94%
PyTorch FP32 latency: 16.945 ms/image
ONNX FP32 accuracy: 96.94%
ONNX FP32 latency: 7.246 ms/image


**10. Paired CPU Inference Benchmark**
Benchmark PyTorch FP32 and ONNX Runtime FP32 under identical CPU conditions using the same preprocessed validation images.

Use one CPU thread, batch size 1, warm-up iterations, and alternating measurement order to reduce timing bias. Compare mean and median latency without including image loading or preprocessing.

This experiment verifies whether the observed ONNX speed improvement is reproducible.

In [ ]:
import time
import statistics
import numpy as np
import torch
import onnxruntime as ort

from pathlib import Path
from evaluate import load_model

PROJECT = Path("/content/FlowerCompetition")
torch.set_num_threads(1)

# PyTorch model
pt_model, _, _ = load_model(
    PROJECT / "mobilenetv3_finetuned_best.pth",
    torch.device("cpu")
)

# ONNX Runtime model
options = ort.SessionOptions()
options.intra_op_num_threads = 1
options.inter_op_num_threads = 1
options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

ort_session = ort.InferenceSession(
    str(PROJECT / "onnx/mobilenetv3_fp32.onnx"),
    sess_options=options,
    providers=["CPUExecutionProvider"]
)

# Reuse the 100 preprocessed validation images
sample_indices = np.linspace(0, len(images) - 1, 100).astype(int)
samples = [images[i] for i in sample_indices]
pt_samples = [torch.from_numpy(x) for x in samples]

def run_pytorch(i):
    return pt_model(pt_samples[i])

def run_onnx(i):
    return ort_session.run(
        ["logits"], {"input": samples[i]}
    )

# Warm-up both runtimes
with torch.inference_mode():
    for i in range(30):
        run_pytorch(i % len(samples))
        run_onnx(i % len(samples))

timings = {"PyTorch": [], "ONNX": []}
run_means = {"PyTorch": [], "ONNX": []}

with torch.inference_mode():
    for repetition in range(4):
        order = (
            ["PyTorch", "ONNX"]
            if repetition % 2 == 0
            else ["ONNX", "PyTorch"]
        )

        for name in order:
            times = []

            for i in range(len(samples)):
                start = time.perf_counter()

                if name == "PyTorch":
                    run_pytorch(i)
                else:
                    run_onnx(i)

                elapsed = (time.perf_counter() - start) * 1000
                times.append(elapsed)

            timings[name].extend(times)
            run_means[name].append(statistics.mean(times))

print("=== PAIRED CPU BENCHMARK ===")

for name in ["PyTorch", "ONNX"]:
    print(f"\n{name}")
    print(f"Mean: {statistics.mean(timings[name]):.3f} ms")
    print(f"Median: {statistics.median(timings[name]):.3f} ms")
    print("Run means:", [round(x, 3) for x in run_means[name]])

speedup = (
    statistics.mean(timings["PyTorch"])
    / statistics.mean(timings["ONNX"])
)

print(f"\nONNX speedup: {speedup:.2f}x")

=== PAIRED CPU BENCHMARK ===

PyTorch
Mean: 18.321 ms
Median: 16.846 ms
Run means: [16.963, 18.648, 20.818, 16.855]

ONNX
Mean: 7.285 ms
Median: 7.201 ms
Run means: [7.266, 7.37, 7.242, 7.265]

ONNX speedup: 2.51x


**11. ONNX Prediction Interface**
Implement a reusable ONNX Runtime prediction function for single-image flower classification.

Apply the same preprocessing used during validation and convert the model's output index to its corresponding class name using the saved class mapping.

Keep the existing PyTorch prediction script unchanged until the ONNX inference interface has been verified.

In [ ]:
import json
import numpy as np
import onnxruntime as ort

from pathlib import Path
from PIL import Image
from evaluate import get_transform

PROJECT = Path("/content/FlowerCompetition")

# Load class mapping
class_to_idx = json.loads(
    (PROJECT / "class_to_idx.json").read_text(encoding="utf-8")
)

idx_to_class = {
    int(idx): class_name
    for class_name, idx in class_to_idx.items()
}

# Create CPU inference session
options = ort.SessionOptions()
options.intra_op_num_threads = 1
options.inter_op_num_threads = 1

onnx_session = ort.InferenceSession(
    str(PROJECT / "onnx/mobilenetv3_fp32.onnx"),
    sess_options=options,
    providers=["CPUExecutionProvider"]
)

transform = get_transform()

def predict_onnx(image_path):
    image = Image.open(image_path).convert("RGB")

    tensor = transform(image).unsqueeze(0)
    input_array = np.ascontiguousarray(tensor.numpy())

    logits = onnx_session.run(
        ["logits"],
        {"input": input_array}
    )[0]

    predicted_idx = int(np.argmax(logits, axis=1)[0])

    return {
        "class_index": predicted_idx,
        "class_name": idx_to_class[predicted_idx]
    }


# Test on a validation image
test_image = next(
    p for p in (PROJECT / "data/valid").rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

prediction = predict_onnx(test_image)

print("=== ONNX PREDICTION TEST ===")
print("Image:", test_image.name)
print("Predicted class index:", prediction["class_index"])
print("Predicted class name:", prediction["class_name"])
print("Expected class:", test_image.parent.name)
print(
    "Prediction correct:",
    prediction["class_name"] == test_image.parent.name
)

=== ONNX PREDICTION TEST ===
Image: image_03568.jpg
Predicted class index: 71
Predicted class name: 72
Expected class: 72
Prediction correct: True


**12. Standalone ONNX Prediction Script**
Create and independently test predict_onnx.py using the exported ONNX model and saved class mapping. The original PyTorch prediction script remains unchanged.

In [ ]:
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
script_path = PROJECT / "predict_onnx.py"

script_code = r'''
import argparse
import json
from pathlib import Path

import numpy as np
import onnxruntime as ort
from PIL import Image


def preprocess(image_path):
    from torchvision import transforms

    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0)

    return np.ascontiguousarray(tensor.numpy())


def predict(image_path, model_path, mapping_path):
    with open(mapping_path, "r", encoding="utf-8") as f:
        class_to_idx = json.load(f)

    idx_to_class = {
        int(idx): name
        for name, idx in class_to_idx.items()
    }

    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1

    session = ort.InferenceSession(
        str(model_path),
        sess_options=options,
        providers=["CPUExecutionProvider"]
    )

    image = preprocess(image_path)

    logits = session.run(
        None,
        {session.get_inputs()[0].name: image}
    )[0]

    predicted_idx = int(np.argmax(logits, axis=1)[0])

    return idx_to_class[predicted_idx]


if __name__ == "__main__":
    parser = argparse.ArgumentParser()

    parser.add_argument("--image", required=True)
    parser.add_argument(
        "--model",
        default="onnx/mobilenetv3_fp32.onnx"
    )
    parser.add_argument(
        "--mapping",
        default="class_to_idx.json"
    )

    args = parser.parse_args()

    base_dir = Path(__file__).resolve().parent

    model_path = Path(args.model)
    mapping_path = Path(args.mapping)

    if not model_path.is_absolute():
        model_path = base_dir / model_path

    if not mapping_path.is_absolute():
        mapping_path = base_dir / mapping_path

    result = predict(
        args.image,
        model_path,
        mapping_path
    )

    print(result)
'''

script_path.write_text(script_code, encoding="utf-8")

print("=== Standalone ONNX Script ===")
print("Created:", script_path)
print("File size:", script_path.stat().st_size, "bytes")

# Run script as an independent Python process
import subprocess

test_image = next(
    p for p in (PROJECT / "data/valid").rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

result = subprocess.run(
    [
        "python",
        str(script_path),
        "--image",
        str(test_image)
    ],
    capture_output=True,
    text=True
)

print("\n=== Independent Execution Test ===")
print("Return code:", result.returncode)
print("Prediction:", result.stdout.strip())

if result.returncode != 0:
    print("Error:", result.stderr)

=== Standalone ONNX Script ===
Created: /content/FlowerCompetition/predict_onnx.py
File size: 2049 bytes

=== Independent Execution Test ===
Return code: 0
Prediction: 72


**13. Standalone Preprocessing Consistency Test**
Verify that the standalone ONNX prediction script applies exactly the same image preprocessing as the validated evaluation pipeline.

Compare the resulting input tensors across multiple validation images and confirm that the predicted classes remain consistent.

This test helps prevent preprocessing mismatches during final deployment.

In [ ]:
import sys
import numpy as np
import torch
import onnxruntime as ort

from pathlib import Path
from evaluate import get_transform

PROJECT = Path("/content/FlowerCompetition")
sys.path.insert(0, str(PROJECT))

from predict_onnx import preprocess

validation_images = sorted(
    p for p in (PROJECT / "data/valid").rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

test_images = validation_images[:100]

reference_transform = get_transform()

max_differences = []
matching_predictions = 0

for image_path in test_images:
    # Original evaluation preprocessing
    from PIL import Image

    with Image.open(image_path) as img:
        reference = (
            reference_transform(img.convert("RGB"))
            .unsqueeze(0)
            .numpy()
        )

    # Standalone prediction preprocessing
    standalone = preprocess(image_path)

    max_differences.append(
        float(np.max(np.abs(reference - standalone)))
    )

    reference_pred = int(np.argmax(
        onnx_session.run(
            ["logits"],
            {"input": reference}
        )[0]
    ))

    standalone_pred = int(np.argmax(
        onnx_session.run(
            ["logits"],
            {"input": standalone}
        )[0]
    ))

    matching_predictions += (
        reference_pred == standalone_pred
    )

print("=== PREPROCESSING CONSISTENCY ===")
print("Images tested:", len(test_images))
print("Maximum tensor difference:", max(max_differences))
print(
    "Exact tensor matches:",
    sum(diff == 0 for diff in max_differences),
    "/", len(test_images)
)
print(
    "Matching predictions:",
    matching_predictions,
    "/", len(test_images)
)

if all(diff == 0 for diff in max_differences):
    print("STATUS: PASSED")
else:
    print("STATUS: REVIEW REQUIRED")

=== PREPROCESSING CONSISTENCY ===
Images tested: 100
Maximum tensor difference: 0.0
Exact tensor matches: 100 / 100
Matching predictions: 100 / 100
STATUS: PASSED


**14. ONNX Deployment Package**
Create a portable ZIP archive containing the validated ONNX model, external weight data, class mapping, and standalone prediction script.

Include dependency information and a short usage guide. Verify the archive contents before saving it to Google Drive.

The original PyTorch checkpoint and project files remain unchanged.

In [ ]:
from pathlib import Path
import zipfile
import shutil
import subprocess

PROJECT = Path("/content/FlowerCompetition")
PACKAGE_DIR = PROJECT / "deployment_onnx"
ZIP_PATH = PROJECT / "MobileNetV3_ONNX_Deployment.zip"

PACKAGE_DIR.mkdir(exist_ok=True)

files_to_copy = [
    "onnx/mobilenetv3_fp32.onnx",
    "onnx/mobilenetv3_fp32.onnx.data",
    "class_to_idx.json",
    "predict_onnx.py"
]

for relative_path in files_to_copy:
    source = PROJECT / relative_path
    destination = PACKAGE_DIR / relative_path

    if not source.is_file():
        raise FileNotFoundError(source)

    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

requirements = """numpy
onnxruntime
Pillow
torch
torchvision
"""

(PACKAGE_DIR / "requirements.txt").write_text(
    requirements,
    encoding="utf-8"
)

readme = """# MobileNetV3-Large ONNX Deployment

Flower classification model with 102 classes.

Validation accuracy: 96.94% (793/818)
CPU inference: approximately 7.08 ms/image
Benchmark: Google Colab CPU, one thread, batch size 1.

## Installation

pip install -r requirements.txt

## Prediction

python predict_onnx.py --image /path/to/flower.jpg

The script prints the predicted class label.

## Required files

- onnx/mobilenetv3_fp32.onnx
- onnx/mobilenetv3_fp32.onnx.data
- class_to_idx.json
- predict_onnx.py

Keep the ONNX model and its .data file together.
"""

(PACKAGE_DIR / "README.md").write_text(
    readme,
    encoding="utf-8"
)

# Create ZIP with relative paths
with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for file in sorted(PACKAGE_DIR.rglob("*")):
        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(PACKAGE_DIR)
            )

# Verify ZIP integrity and contents
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    corrupted = archive.testzip()
    contents = archive.namelist()

    if corrupted is not None:
        raise RuntimeError(f"Corrupted ZIP member: {corrupted}")

print("=== ONNX DEPLOYMENT PACKAGE ===")
print("ZIP:", ZIP_PATH)
print(f"ZIP size: {ZIP_PATH.stat().st_size / 1024**2:.2f} MiB")
print("ZIP integrity: PASSED")

print("\nIncluded files:")
for name in contents:
    print("-", name)

=== ONNX DEPLOYMENT PACKAGE ===
ZIP: /content/FlowerCompetition/MobileNetV3_ONNX_Deployment.zip
ZIP size: 15.35 MiB
ZIP integrity: PASSED

Included files:
- README.md
- class_to_idx.json
- onnx/mobilenetv3_fp32.onnx
- onnx/mobilenetv3_fp32.onnx.data
- predict_onnx.py
- requirements.txt


**15. Google Drive Backup**
Back up the validated ONNX deployment package to the existing Flower_Competition folder in Google Drive.

Verify the copied archive using SHA-256 checksums to ensure that the backup is identical to the original file.

Preserve the existing full project backup without modification.

In [ ]:
from pathlib import Path
import shutil
import hashlib

source = Path(
    "/content/FlowerCompetition/MobileNetV3_ONNX_Deployment.zip"
)

drive_dir = Path(
    "/content/drive/MyDrive/Flower_Competition"
)

destination = drive_dir / source.name

if not source.is_file():
    raise FileNotFoundError(source)

if not drive_dir.is_dir():
    raise RuntimeError(
        "Google Drive folder not found. "
        "Check that Drive is mounted."
    )

shutil.copy2(source, destination)

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024), b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

source_hash = sha256_file(source)
backup_hash = sha256_file(destination)

print("=== GOOGLE DRIVE BACKUP ===")
print("Source:", source)
print("Destination:", destination)
print(f"Size: {destination.stat().st_size / 1024**2:.2f} MiB")
print("SHA-256 match:", source_hash == backup_hash)

if source_hash == backup_hash:
    print("STATUS: BACKUP VERIFIED")
else:
    raise RuntimeError("Backup checksum mismatch!")

=== GOOGLE DRIVE BACKUP ===
Source: /content/FlowerCompetition/MobileNetV3_ONNX_Deployment.zip
Destination: /content/drive/MyDrive/Flower_Competition/MobileNetV3_ONNX_Deployment.zip
Size: 15.35 MiB
SHA-256 match: True
STATUS: BACKUP VERIFIED


In [ ]:
from pathlib import Path

project = Path("/content/FlowerCompetition")

for filename in ["evaluate.py", "predict.py"]:
    path = project / filename

    print(f"\n{'=' * 20} {filename} {'=' * 20}")

    if path.exists():
        print(path.read_text(encoding="utf-8"))
    else:
        print("FILE NOT FOUND")


==================== evaluate.py ====================

import argparse
import json
import math
import statistics
import time
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v3_large


def load_model(checkpoint_path, device):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Checkpoint bulunamadı: {checkpoint_path}"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    architecture = checkpoint.get(
        "architecture", "mobilenet_v3_large"
    )

    if architecture != "mobilenet_v3_large":
        raise ValueError("Desteklenmeyen model mimarisi")

    class_to_idx = checkpoint["class_to_idx"]
    num_classes = len(class_to_idx)

    if sorted(class_to_idx.valu

### 17. Model Checkpoint Inventory

Inspect the available PyTorch checkpoints and ONNX models before designing a shared evaluation and prediction interface.

Identify model architectures, checkpoint metadata, and file sizes without modifying any existing models or scripts.

In [ ]:
from pathlib import Path
import torch

PROJECT = Path("/content/FlowerCompetition")

print("=== MODEL INVENTORY ===")

for path in sorted(PROJECT.rglob("*")):
    if not path.is_file():
        continue

    if path.suffix.lower() not in {".pth", ".pt", ".onnx"}:
        continue

    print(f"\nFile: {path.relative_to(PROJECT)}")
    print(f"Size: {path.stat().st_size / 1024**2:.2f} MiB")

    if path.suffix.lower() in {".pth", ".pt"}:
        try:
            checkpoint = torch.load(
                path,
                map_location="cpu",
                weights_only=True
            )

            if isinstance(checkpoint, dict):
                print("Architecture:", checkpoint.get("architecture", "Not specified"))
                print("Precision:", checkpoint.get("precision", "Not specified"))

                mapping = checkpoint.get("class_to_idx")
                if isinstance(mapping, dict):
                    print("Classes:", len(mapping))

        except Exception as exc:
            print("Metadata unavailable:", str(exc)[:150])

    else:
        print("Format: ONNX")

=== MODEL INVENTORY ===


In [ ]:
from pathlib import Path

project = Path("/content/FlowerCompetition")

print("Project exists:", project.exists())
print("Current directory:", Path.cwd())

print("\n/content files:")
for p in Path("/content").iterdir():
    print("-", p.name)

print("\nDrive mounted:",
      Path("/content/drive/MyDrive").exists())

Project exists: False
Current directory: /content

/content files:
- .config
- drive
- sample_data

Drive mounted: True


In [ ]:
from pathlib import Path

project = Path("/content/FlowerCompetition")
drive = Path("/content/drive/MyDrive/Flower_Competition")

print("=== ENVIRONMENT CHECK ===")
print("Project exists:", project.is_dir())
print("Drive mounted:", Path("/content/drive/MyDrive").is_dir())
print("Full backup:", (drive / "FlowerCompetition_FullBackup.zip").is_file())
print("ONNX backup:", (drive / "MobileNetV3_ONNX_Deployment.zip").is_file())

=== ENVIRONMENT CHECK ===
Project exists: False
Drive mounted: True
Full backup: True
ONNX backup: True


### 18. Restore Project and ONNX Deployment

Restore the original FlowerCompetition project and the validated ONNX deployment files from Google Drive after the Colab runtime reset.

Verify that the PyTorch checkpoint, ONNX model, external weights, and inference scripts are available before integrating the EfficientNet-B0 model.

In [ ]:
from pathlib import Path
import zipfile

DRIVE = Path("/content/drive/MyDrive/Flower_Competition")
PROJECT = Path("/content/FlowerCompetition")

full_backup = DRIVE / "FlowerCompetition_FullBackup.zip"
onnx_backup = DRIVE / "MobileNetV3_ONNX_Deployment.zip"

def restore_zip(zip_path, destination):
    if not zip_path.is_file():
        raise FileNotFoundError(zip_path)

    with zipfile.ZipFile(zip_path) as archive:
        bad_file = archive.testzip()
        if bad_file:
            raise RuntimeError(f"Corrupted file: {bad_file}")

        # Reject unsafe archive paths
        base = destination.resolve()
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(base):
                raise RuntimeError(
                    f"Unsafe ZIP path: {member.filename}"
                )

        archive.extractall(destination)

    print("Restored:", zip_path.name)

print("=== RESTORING PROJECT ===")
restore_zip(full_backup, Path("/content"))

print("\n=== RESTORING ONNX FILES ===")
PROJECT.mkdir(parents=True, exist_ok=True)
restore_zip(onnx_backup, PROJECT)

print("\n=== FILE VERIFICATION ===")

required_files = [
    "mobilenetv3_finetuned_best.pth",
    "evaluate.py",
    "predict.py",
    "class_to_idx.json",
    "onnx/mobilenetv3_fp32.onnx",
    "onnx/mobilenetv3_fp32.onnx.data",
    "predict_onnx.py",
]

for filename in required_files:
    path = PROJECT / filename
    print(
        f"{filename}:",
        "FOUND" if path.is_file() else "MISSING"
    )

print("\nProject restored:", PROJECT.is_dir())

=== RESTORING PROJECT ===
Restored: FlowerCompetition_FullBackup.zip

=== RESTORING ONNX FILES ===
Restored: MobileNetV3_ONNX_Deployment.zip

=== FILE VERIFICATION ===
mobilenetv3_finetuned_best.pth: FOUND
evaluate.py: FOUND
predict.py: FOUND
class_to_idx.json: FOUND
onnx/mobilenetv3_fp32.onnx: FOUND
onnx/mobilenetv3_fp32.onnx.data: FOUND
predict_onnx.py: FOUND

Project restored: True


### 19. Post-Restoration Environment Verification

Verify the restored dataset structure, model checkpoints, and Python dependencies before proceeding with multi-model integration.

In [ ]:
from pathlib import Path
import importlib.util

PROJECT = Path("/content/FlowerCompetition")

print("=== DATASET VERIFICATION ===")

for split in ["train", "valid", "test"]:
    folder = PROJECT / "data" / split

    if folder.is_dir():
        classes = [p for p in folder.iterdir() if p.is_dir()]
        images = sum(
            1 for p in folder.rglob("*")
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        )

        print(f"{split}: {len(classes)} classes, {images} images")
    else:
        print(f"{split}: MISSING")

print("\n=== MODEL CHECKPOINTS ===")

for name in [
    "mobilenetv3_finetuned_best.pth",
    "mobilenetv3_fp16.pth",
    "mobilenetv3_int8_experimental.pth",
]:
    path = PROJECT / name

    if path.is_file():
        print(f"{name}: {path.stat().st_size / (1024**2):.2f} MiB")
    else:
        print(f"{name}: MISSING")

print("\n=== PYTHON DEPENDENCIES ===")

for module in [
    "torch",
    "torchvision",
    "numpy",
    "PIL",
    "onnxruntime",
]:
    available = importlib.util.find_spec(module) is not None
    print(f"{module}: {'AVAILABLE' if available else 'MISSING'}")

=== DATASET VERIFICATION ===
train: 102 classes, 6552 images
valid: 102 classes, 818 images
test: 102 classes, 819 images

=== MODEL CHECKPOINTS ===
mobilenetv3_finetuned_best.pth: 16.73 MiB
mobilenetv3_fp16.pth: 8.42 MiB
mobilenetv3_int8_experimental.pth: 4.61 MiB

=== PYTHON DEPENDENCIES ===
torch: AVAILABLE
torchvision: AVAILABLE
numpy: AVAILABLE
PIL: AVAILABLE
onnxruntime: MISSING


### 20. Install ONNX Runtime

Install the ONNX Runtime dependency required for optimized CPU inference after restoring the Colab environment.

In [ ]:
%pip install -q onnxruntime

import onnxruntime as ort

print("ONNX Runtime version:", ort.__version__)
print("Available providers:", ort.get_available_providers())

model_path = "/content/FlowerCompetition/onnx/mobilenetv3_fp32.onnx"

session = ort.InferenceSession(
    model_path,
    providers=["CPUExecutionProvider"]
)

print("\nONNX model loaded successfully!")
print("Input:", session.get_inputs()[0].shape)
print("Output:", session.get_outputs()[0].shape)

ONNX Runtime version: 1.31.1
Available providers: ['AzureExecutionProvider', 'CPUExecutionProvider']

ONNX model loaded successfully!
Input: [1, 3, 224, 224]
Output: [1, 102]


### 21. Backup Existing Evaluation and Prediction Scripts

Preserve the validated MobileNetV3 evaluation and prediction scripts before developing the shared multi-model inference interface.

In [ ]:
from pathlib import Path
import shutil

PROJECT = Path("/content/FlowerCompetition")
BACKUP_DIR = PROJECT / "script_backups"

BACKUP_DIR.mkdir(parents=True, exist_ok=True)

for filename in ["evaluate.py", "predict.py"]:
    source = PROJECT / filename
    destination = BACKUP_DIR / filename

    if not source.is_file():
        raise FileNotFoundError(f"Missing script: {source}")

    if destination.exists():
        print(f"Already backed up: {filename}")
    else:
        shutil.copy2(source, destination)
        print(f"Backup created: {destination}")

print("\nOriginal scripts preserved.")

Backup created: /content/FlowerCompetition/script_backups/evaluate.py
Backup created: /content/FlowerCompetition/script_backups/predict.py

Original scripts preserved.


### 22. Inspect Model Checkpoint Structure

Inspect the saved MobileNetV3 checkpoint metadata and state dictionary structure to prepare a reliable multi-model loading interface without modifying the original model.

In [ ]:
from pathlib import Path
import torch

PROJECT = Path("/content/FlowerCompetition")
checkpoint_path = PROJECT / "mobilenetv3_finetuned_best.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("=== CHECKPOINT STRUCTURE ===")

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("\nTop-level keys:")
    for key in checkpoint.keys():
        value = checkpoint[key]
        print(f"  {key}: {type(value).__name__}")

    print("\n=== MODEL METADATA ===")

    for key in [
        "model_name",
        "architecture",
        "num_classes",
        "class_to_idx",
        "epoch",
        "best_val_acc",
        "val_accuracy",
    ]:
        if key in checkpoint:
            value = checkpoint[key]

            if key == "class_to_idx":
                print(f"{key}: {len(value)} entries")
                print("First 5:", list(value.items())[:5])
            else:
                print(f"{key}: {value}")

    print("\n=== STATE DICTIONARY ===")

    state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint.get("state_dict")
    )

    if state_dict is not None:
        print("Total tensors:", len(state_dict))

        for name, tensor in list(state_dict.items())[:5]:
            print(name, tuple(tensor.shape))
    else:
        print("No nested state dictionary found.")

del checkpoint

=== CHECKPOINT STRUCTURE ===
Checkpoint type: dict

Top-level keys:
  model_state_dict: OrderedDict
  class_to_idx: dict
  epoch: int
  valid_accuracy: float
  architecture: str
  image_size: int

=== MODEL METADATA ===
architecture: mobilenet_v3_large
class_to_idx: 102 entries
First 5: [('1', 0), ('2', 1), ('3', 2), ('4', 3), ('5', 4)]
epoch: 3

=== STATE DICTIONARY ===
Total tensors: 312
features.0.0.weight (16, 3, 3, 3)
features.0.1.weight (16,)
features.0.1.bias (16,)
features.0.1.running_mean (16,)
features.0.1.running_var (16,)


### 23. Create a Shared Model Registry

Create a centralized model registry for the competition. The registry identifies each model's architecture, checkpoint format, class count, and inference backend.

Only validated models are enabled. Additional team models will be registered after their architecture, preprocessing, and class mappings are verified.

In [ ]:
from pathlib import Path
import json

PROJECT = Path("/content/FlowerCompetition")

registry = {
    "mobilenetv3_pytorch": {
        "architecture": "mobilenet_v3_large",
        "backend": "pytorch",
        "checkpoint": "mobilenetv3_finetuned_best.pth",
        "num_classes": 102,
        "image_size": 224,
        "preprocessing": "imagenet_resize256_centercrop224",
        "class_mapping": "checkpoint",
        "enabled": True
    },
    "mobilenetv3_onnx": {
        "architecture": "mobilenet_v3_large",
        "backend": "onnxruntime",
        "checkpoint": "onnx/mobilenetv3_fp32.onnx",
        "num_classes": 102,
        "image_size": 224,
        "preprocessing": "imagenet_resize256_centercrop224",
        "class_mapping": "class_to_idx.json",
        "enabled": True
    }
}

registry_path = PROJECT / "model_registry.json"

with open(registry_path, "w", encoding="utf-8") as f:
    json.dump(registry, f, indent=4, ensure_ascii=False)

print("=== SHARED MODEL REGISTRY ===")
print("Registry saved:", registry_path)

for model_name, config in registry.items():
    checkpoint = PROJECT / config["checkpoint"]

    print(f"\nModel: {model_name}")
    print("  Architecture:", config["architecture"])
    print("  Backend:", config["backend"])
    print("  Checkpoint exists:", checkpoint.is_file())
    print("  Enabled:", config["enabled"])

print("\nRegistered models:", len(registry))

=== SHARED MODEL REGISTRY ===
Registry saved: /content/FlowerCompetition/model_registry.json

Model: mobilenetv3_pytorch
  Architecture: mobilenet_v3_large
  Backend: pytorch
  Checkpoint exists: True
  Enabled: True

Model: mobilenetv3_onnx
  Architecture: mobilenet_v3_large
  Backend: onnxruntime
  Checkpoint exists: True
  Enabled: True

Registered models: 2


### 24. Create a Shared Model Loader

Implement a unified model-loading interface for the registered PyTorch and ONNX models.

The loader reads model configurations from the shared registry, validates checkpoint metadata and class mappings, and initializes the appropriate inference backend. Existing evaluation and prediction scripts remain unchanged.

In [ ]:
from pathlib import Path
import json
import textwrap

PROJECT = Path("/content/FlowerCompetition")

shared_loader_code = r'''
from pathlib import Path
import json

import torch
import torchvision.models as models
import onnxruntime as ort


PROJECT_DIR = Path(__file__).resolve().parent
REGISTRY_PATH = PROJECT_DIR / "model_registry.json"


def load_registry():
    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


def load_class_mapping(mapping_path):
    with open(mapping_path, "r", encoding="utf-8") as f:
        mapping = json.load(f)

    mapping = {str(k): int(v) for k, v in mapping.items()}

    if sorted(mapping.values()) != list(range(len(mapping))):
        raise ValueError("Invalid class index mapping.")

    return mapping


class SharedModel:
    def __init__(self, model_name, device="cpu"):
        registry = load_registry()

        if model_name not in registry:
            raise ValueError(f"Unknown model: {model_name}")

        self.name = model_name
        self.config = registry[model_name]

        if not self.config.get("enabled", False):
            raise ValueError(f"Model is disabled: {model_name}")

        self.backend = self.config["backend"]
        self.architecture = self.config["architecture"]
        self.num_classes = self.config["num_classes"]
        self.device = torch.device(device)

        checkpoint_path = PROJECT_DIR / self.config["checkpoint"]

        if not checkpoint_path.is_file():
            raise FileNotFoundError(checkpoint_path)

        if self.backend == "pytorch":
            self._load_pytorch(checkpoint_path)

        elif self.backend == "onnxruntime":
            self._load_onnx(checkpoint_path)

        else:
            raise ValueError(f"Unsupported backend: {self.backend}")

    def _load_pytorch(self, checkpoint_path):
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

        if self.architecture == "mobilenet_v3_large":
            model = models.mobilenet_v3_large(weights=None)
            model.classifier[3] = torch.nn.Linear(
                model.classifier[3].in_features,
                self.num_classes
            )
        else:
            raise ValueError(
                f"Unsupported architecture: {self.architecture}"
            )

        state_dict = checkpoint.get(
            "model_state_dict",
            checkpoint.get("state_dict")
        )

        if state_dict is None:
            raise ValueError("Checkpoint has no state dictionary.")

        checkpoint_arch = checkpoint.get("architecture")

        if checkpoint_arch and checkpoint_arch != self.architecture:
            raise ValueError("Checkpoint architecture mismatch.")

        self.class_to_idx = {
            str(k): int(v)
            for k, v in checkpoint["class_to_idx"].items()
        }

        if sorted(self.class_to_idx.values()) != list(
            range(self.num_classes)
        ):
            raise ValueError("Invalid checkpoint class mapping.")

        model.load_state_dict(state_dict, strict=True)
        self.model = model.to(self.device).eval()

    def _load_onnx(self, checkpoint_path):
        if self.device.type != "cpu":
            raise ValueError("ONNX deployment currently supports CPU only.")

        options = ort.SessionOptions()
        options.intra_op_num_threads = 1
        options.inter_op_num_threads = 1

        self.session = ort.InferenceSession(
            str(checkpoint_path),
            sess_options=options,
            providers=["CPUExecutionProvider"]
        )

        mapping_path = PROJECT_DIR / self.config["class_mapping"]
        self.class_to_idx = load_class_mapping(mapping_path)

        if len(self.class_to_idx) != self.num_classes:
            raise ValueError("ONNX class count mismatch.")

    @torch.inference_mode()
    def predict_logits(self, images):
        if self.backend == "pytorch":
            images = images.to(self.device)
            return self.model(images).cpu()

        if self.backend == "onnxruntime":
            inputs = images.detach().cpu().numpy().astype("float32")

            outputs = self.session.run(
                None,
                {self.session.get_inputs()[0].name: inputs}
            )

            return torch.from_numpy(outputs[0])

        raise RuntimeError("Unsupported inference backend.")
'''

loader_path = PROJECT / "shared_model_loader.py"

loader_path.write_text(
    textwrap.dedent(shared_loader_code).lstrip(),
    encoding="utf-8"
)

print("=== SHARED MODEL LOADER ===")
print("Created:", loader_path)
print("File size:", loader_path.stat().st_size, "bytes")

import sys
sys.path.insert(0, str(PROJECT))

from shared_model_loader import SharedModel

for model_name in ["mobilenetv3_pytorch", "mobilenetv3_onnx"]:
    model = SharedModel(model_name, device="cpu")

    print(f"\nModel: {model_name}")
    print("Backend:", model.backend)
    print("Architecture:", model.architecture)
    print("Classes:", len(model.class_to_idx))
    print("Loaded successfully!")

    del model

=== SHARED MODEL LOADER ===
Created: /content/FlowerCompetition/shared_model_loader.py
File size: 4303 bytes

Model: mobilenetv3_pytorch
Backend: pytorch
Architecture: mobilenet_v3_large
Classes: 102
Loaded successfully!

Model: mobilenetv3_onnx
Backend: onnxruntime
Architecture: mobilenet_v3_large
Classes: 102
Loaded successfully!


### 25. Verify PyTorch and ONNX Prediction Consistency

Compare the predictions of the original PyTorch MobileNetV3 model and its optimized ONNX version on 100 validation images.

Verify class mapping consistency, prediction agreement, and numerical differences between model outputs.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import torch

from PIL import Image
from torchvision import transforms

PROJECT = Path("/content/FlowerCompetition")
sys.path.insert(0, str(PROJECT))

from shared_model_loader import SharedModel

torch.set_num_threads(1)

# Load both model versions
pytorch_model = SharedModel("mobilenetv3_pytorch", device="cpu")
onnx_model = SharedModel("mobilenetv3_onnx", device="cpu")

# Verify class mapping consistency
assert (
    pytorch_model.class_to_idx == onnx_model.class_to_idx
), "Class mappings do not match!"

print("Class mappings: IDENTICAL")

# Use the same preprocessing as validation
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

valid_dir = PROJECT / "data" / "valid"

image_paths = sorted(
    p for p in valid_dir.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)[:100]

assert len(image_paths) == 100, (
    f"Expected 100 images, found {len(image_paths)}"
)

agreements = 0
max_logit_difference = 0.0
total_logit_difference = 0.0

for image_path in image_paths:
    with Image.open(image_path) as image:
        image_tensor = transform(
            image.convert("RGB")
        ).unsqueeze(0)

    pytorch_logits = pytorch_model.predict_logits(image_tensor)
    onnx_logits = onnx_model.predict_logits(image_tensor)

    pytorch_prediction = pytorch_logits.argmax(dim=1).item()
    onnx_prediction = onnx_logits.argmax(dim=1).item()

    if pytorch_prediction == onnx_prediction:
        agreements += 1

    difference = (
        pytorch_logits - onnx_logits
    ).abs()

    max_logit_difference = max(
        max_logit_difference,
        difference.max().item()
    )

    total_logit_difference += difference.mean().item()

print("\n=== PREDICTION CONSISTENCY ===")
print("Images tested:", len(image_paths))
print(
    "Prediction agreement:",
    f"{agreements}/{len(image_paths)}",
    f"({agreements / len(image_paths) * 100:.2f}%)"
)
print(
    "Maximum logit difference:",
    f"{max_logit_difference:.8f}"
)
print(
    "Mean logit difference:",
    f"{total_logit_difference / len(image_paths):.8f}"
)

if agreements == len(image_paths):
    print("\nPASS: PyTorch and ONNX predictions are consistent.")
else:
    print("\nWARNING: Prediction differences detected.")

Class mappings: IDENTICAL

=== PREDICTION CONSISTENCY ===
Images tested: 100
Prediction agreement: 100/100 (100.00%)
Maximum logit difference: 0.00006866
Mean logit difference: 0.00000815

PASS: PyTorch and ONNX predictions are consistent.


### 26. Create Shared Evaluation Script

Implement a unified evaluation script for registered PyTorch and ONNX models.

Evaluate classification accuracy, CPU inference latency, and model size using consistent preprocessing and class mappings. Save the evaluation results as JSON for reproducible model comparisons.

In [ ]:
from pathlib import Path
import textwrap

PROJECT = Path("/content/FlowerCompetition")

shared_evaluation_code = r'''
import argparse
import json
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder

from shared_model_loader import SharedModel, PROJECT_DIR


def get_transform():
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])


def evaluate_model(model_name, data_dir, output_path=None):
    torch.set_num_threads(1)

    model = SharedModel(model_name, device="cpu")

    dataset = ImageFolder(
        root=str(data_dir),
        transform=get_transform()
    )

    if dataset.class_to_idx != model.class_to_idx:
        raise ValueError(
            "Dataset and model class mappings do not match."
        )

    correct = 0
    total = 0
    latencies = []

    # Preload samples so disk I/O is excluded from timing.
    samples = []

    for image, label in dataset:
        samples.append((image.unsqueeze(0), label))

    # Warm up the model.
    for image, _ in samples[:min(30, len(samples))]:
        model.predict_logits(image)

    for image, label in samples:
        start = time.perf_counter()

        logits = model.predict_logits(image)

        latency_ms = (
            time.perf_counter() - start
        ) * 1000

        prediction = int(logits.argmax(dim=1).item())

        correct += int(prediction == label)
        total += 1
        latencies.append(latency_ms)

    checkpoint = PROJECT_DIR / model.config["checkpoint"]

    size_bytes = checkpoint.stat().st_size

    # Include external ONNX weights in model size.
    if model.backend == "onnxruntime":
        external_data = Path(str(checkpoint) + ".data")
        if external_data.is_file():
            size_bytes += external_data.stat().st_size

    results = {
        "model": model_name,
        "architecture": model.architecture,
        "backend": model.backend,
        "dataset": str(data_dir),
        "images": total,
        "correct": correct,
        "accuracy": correct / total if total else 0.0,
        "cpu_mean_latency_ms": float(np.mean(latencies)),
        "cpu_median_latency_ms": float(np.median(latencies)),
        "cpu_p95_latency_ms": float(np.percentile(latencies, 95)),
        "model_size_mib": size_bytes / (1024 ** 2),
        "cpu_threads": 1,
        "batch_size": 1
    }

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=4)

    return results


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--data-dir", required=True)
    parser.add_argument("--output", default=None)

    args = parser.parse_args()

    results = evaluate_model(
        model_name=args.model,
        data_dir=args.data_dir,
        output_path=args.output
    )

    print(json.dumps(results, indent=4))


if __name__ == "__main__":
    main()
'''

script_path = PROJECT / "evaluate_shared.py"

script_path.write_text(
    textwrap.dedent(shared_evaluation_code).lstrip(),
    encoding="utf-8"
)

print("=== SHARED EVALUATION SCRIPT ===")
print("Created:", script_path)

# Syntax verification without running evaluation.
import py_compile

py_compile.compile(
    str(script_path),
    doraise=True
)

print("Python syntax: VALID")
print("Original evaluate.py: UNCHANGED")

=== SHARED EVALUATION SCRIPT ===
Created: /content/FlowerCompetition/evaluate_shared.py
Python syntax: VALID
Original evaluate.py: UNCHANGED


### 27. Benchmark PyTorch and ONNX Models

Evaluate both registered MobileNetV3 implementations on the complete validation dataset using the shared evaluation pipeline.

Compare classification accuracy, single-thread CPU inference latency, and model size. Save both results as JSON files for subsequent model comparisons.

In [ ]:
from pathlib import Path
import sys
import json
import gc
import torch

PROJECT = Path("/content/FlowerCompetition")
sys.path.insert(0, str(PROJECT))

from evaluate_shared import evaluate_model

VALID_DIR = PROJECT / "data" / "valid"
RESULTS_DIR = PROJECT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

models_to_evaluate = [
    "mobilenetv3_pytorch",
    "mobilenetv3_onnx"
]

all_results = {}

for model_name in models_to_evaluate:
    print(f"\n=== EVALUATING {model_name} ===")

    result = evaluate_model(
        model_name=model_name,
        data_dir=VALID_DIR,
        output_path=RESULTS_DIR / f"{model_name}_shared.json"
    )

    all_results[model_name] = result

    print(f"Accuracy: {result['accuracy'] * 100:.2f}%")
    print(f"Correct: {result['correct']}/{result['images']}")
    print(f"CPU mean: {result['cpu_mean_latency_ms']:.3f} ms")
    print(f"CPU median: {result['cpu_median_latency_ms']:.3f} ms")
    print(f"CPU P95: {result['cpu_p95_latency_ms']:.3f} ms")
    print(f"Model size: {result['model_size_mib']:.2f} MiB")

    gc.collect()

print("\n=== COMPARISON SUMMARY ===")

pytorch_result = all_results["mobilenetv3_pytorch"]
onnx_result = all_results["mobilenetv3_onnx"]

print(
    "Accuracy match:",
    pytorch_result["correct"] == onnx_result["correct"]
)

print(
    "Expected accuracy:",
    "PASS" if (
        pytorch_result["correct"] == 793
        and onnx_result["correct"] == 793
    ) else "REVIEW REQUIRED"
)

speedup = (
    pytorch_result["cpu_mean_latency_ms"]
    / onnx_result["cpu_mean_latency_ms"]
)

print(f"ONNX speedup: {speedup:.2f}x")
print("Results saved to:", RESULTS_DIR)


=== EVALUATING mobilenetv3_pytorch ===
Accuracy: 96.94%
Correct: 793/818
CPU mean: 24.093 ms
CPU median: 23.452 ms
CPU P95: 29.215 ms
Model size: 16.73 MiB

=== EVALUATING mobilenetv3_onnx ===
Accuracy: 96.94%
Correct: 793/818
CPU mean: 7.090 ms
CPU median: 6.873 ms
CPU P95: 8.386 ms
Model size: 16.80 MiB

=== COMPARISON SUMMARY ===
Accuracy match: True
Expected accuracy: PASS
ONNX speedup: 3.40x
Results saved to: /content/FlowerCompetition/results


In [ ]:
from pathlib import Path
import importlib
import sys

PROJECT = Path("/content/FlowerCompetition")
script_path = PROJECT / "evaluate_shared.py"

source = script_path.read_text(encoding="utf-8")

old_code = '''    if dataset.class_to_idx != model.class_to_idx:
        raise ValueError(
            "Dataset and model class mappings do not match."
        )
'''

new_code = '''    # ImageFolder sorts folder names alphabetically.
    # Remap its labels using the checkpoint's class_to_idx.
    if set(dataset.classes) != set(model.class_to_idx):
        raise ValueError(
            "Dataset classes and model classes do not match."
        )

    dataset.target_transform = lambda label: model.class_to_idx[
        dataset.classes[label]
    ]
'''

if old_code not in source:
    if new_code in source:
        print("Class mapping fix already applied.")
    else:
        raise RuntimeError(
            "Expected code block not found. No changes made."
        )
else:
    source = source.replace(old_code, new_code, 1)
    script_path.write_text(source, encoding="utf-8")
    print("Class mapping fix applied successfully.")

# Verify the Python file is valid.
import py_compile
py_compile.compile(str(script_path), doraise=True)

# Reload the modified module.
sys.path.insert(0, str(PROJECT))
import evaluate_shared
importlib.reload(evaluate_shared)

print("Python syntax: VALID")
print("Evaluation module: RELOADED")
print("Original evaluate.py: UNCHANGED")

Class mapping fix applied successfully.
Python syntax: VALID
Evaluation module: RELOADED
Original evaluate.py: UNCHANGED


### 28. Create Shared Prediction Script

Implement a unified prediction interface for registered PyTorch and ONNX models.

The script supports single-image inference, configurable top-k predictions, and JSON output using consistent preprocessing and class mappings.

In [ ]:
from pathlib import Path
import textwrap
import py_compile

PROJECT = Path("/content/FlowerCompetition")

prediction_code = r'''
import argparse
import json
from pathlib import Path

import torch
from PIL import Image
from torchvision import transforms

from shared_model_loader import SharedModel


def get_transform():
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])


class SharedPredictor:
    def __init__(self, model_name):
        self.model = SharedModel(model_name, device="cpu")
        self.transform = get_transform()
        self.idx_to_class = {
            index: name
            for name, index in self.model.class_to_idx.items()
        }

    def predict(self, image_path, top_k=5):
        image_path = Path(image_path)

        if not image_path.is_file():
            raise FileNotFoundError(image_path)

        if not 1 <= top_k <= self.model.num_classes:
            raise ValueError("Invalid top_k value.")

        with Image.open(image_path) as image:
            tensor = self.transform(
                image.convert("RGB")
            ).unsqueeze(0)

        logits = self.model.predict_logits(tensor)
        probabilities = torch.softmax(logits, dim=1)[0]

        scores, indices = torch.topk(
            probabilities,
            k=top_k
        )

        predictions = []

        for score, index in zip(scores, indices):
            class_index = int(index.item())

            predictions.append({
                "class_name": self.idx_to_class[class_index],
                "class_index": class_index,
                "confidence": float(score.item())
            })

        return {
            "model": self.model.name,
            "image": str(image_path),
            "predictions": predictions
        }


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--model", required=True)
    parser.add_argument("--image", required=True)
    parser.add_argument("--top-k", type=int, default=5)
    parser.add_argument("--output", default=None)

    args = parser.parse_args()

    predictor = SharedPredictor(args.model)

    result = predictor.predict(
        image_path=args.image,
        top_k=args.top_k
    )

    if args.output:
        output_path = Path(args.output)
        output_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=4)

    print(json.dumps(result, indent=4))


if __name__ == "__main__":
    main()
'''

script_path = PROJECT / "predict_shared.py"

script_path.write_text(
    textwrap.dedent(prediction_code).lstrip(),
    encoding="utf-8"
)

py_compile.compile(str(script_path), doraise=True)

print("=== SHARED PREDICTION SCRIPT ===")
print("Created:", script_path)
print("Python syntax: VALID")
print("Original predict.py: UNCHANGED")

=== SHARED PREDICTION SCRIPT ===
Created: /content/FlowerCompetition/predict_shared.py
Python syntax: VALID
Original predict.py: UNCHANGED


### 29. Test Shared Prediction Interface

Test the shared prediction interface using the PyTorch and ONNX versions of MobileNetV3.

Compare the top-1 predicted class and top-5 probabilities on the same validation image, and verify that the prediction matches the ground-truth label.

In [ ]:
from pathlib import Path
import sys

PROJECT = Path("/content/FlowerCompetition")
sys.path.insert(0, str(PROJECT))

from predict_shared import SharedPredictor

# Select a validation image from a known class.
valid_dir = PROJECT / "data" / "valid"

image_paths = sorted(
    p for p in valid_dir.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

assert image_paths, "No validation images found."

image_path = image_paths[0]
true_class = image_path.parent.name

print("=== TEST IMAGE ===")
print("Image:", image_path.name)
print("Ground truth:", true_class)

results = {}

for model_name in [
    "mobilenetv3_pytorch",
    "mobilenetv3_onnx"
]:
    predictor = SharedPredictor(model_name)

    result = predictor.predict(
        image_path=image_path,
        top_k=5
    )

    results[model_name] = result

    print(f"\n=== {model_name} ===")

    for rank, prediction in enumerate(
        result["predictions"], start=1
    ):
        print(
            f"{rank}. Class {prediction['class_name']} "
            f"— {prediction['confidence'] * 100:.2f}%"
        )

    top1 = result["predictions"][0]["class_name"]
    print("Correct prediction:", top1 == true_class)

pytorch_top1 = results[
    "mobilenetv3_pytorch"
]["predictions"][0]["class_name"]

onnx_top1 = results[
    "mobilenetv3_onnx"
]["predictions"][0]["class_name"]

print("\n=== CONSISTENCY CHECK ===")
print("PyTorch prediction:", pytorch_top1)
print("ONNX prediction:", onnx_top1)
print("Predictions match:", pytorch_top1 == onnx_top1)

=== TEST IMAGE ===
Image: image_06739.jpg
Ground truth: 1

=== mobilenetv3_pytorch ===
1. Class 1 — 99.81%
2. Class 19 — 0.17%
3. Class 52 — 0.01%
4. Class 83 — 0.00%
5. Class 51 — 0.00%
Correct prediction: True

=== mobilenetv3_onnx ===
1. Class 1 — 99.81%
2. Class 19 — 0.17%
3. Class 52 — 0.01%
4. Class 83 — 0.00%
5. Class 51 — 0.00%
Correct prediction: True

=== CONSISTENCY CHECK ===
PyTorch prediction: 1
ONNX prediction: 1
Predictions match: True


### 30. Backup Shared Evaluation and Prediction System

Create and verify a Google Drive backup of the shared model registry, model loader, evaluation and prediction scripts, and validation results.

Preserve the original training and ONNX deployment backups without modifying them.

In [ ]:
from pathlib import Path
import zipfile
import hashlib

PROJECT = Path("/content/FlowerCompetition")
DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

DRIVE.mkdir(parents=True, exist_ok=True)

backup_path = DRIVE / "FlowerCompetition_SharedSystem_Backup.zip"

required_files = [
    "model_registry.json",
    "shared_model_loader.py",
    "evaluate_shared.py",
    "predict_shared.py",
    "results/mobilenetv3_pytorch_shared.json",
    "results/mobilenetv3_onnx_shared.json",
]

for filename in required_files:
    if not (PROJECT / filename).is_file():
        raise FileNotFoundError(
            f"Missing required file: {filename}"
        )

with zipfile.ZipFile(
    backup_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:

    for filename in required_files:
        archive.write(
            PROJECT / filename,
            arcname=filename
        )

with zipfile.ZipFile(backup_path) as archive:
    bad_file = archive.testzip()

    if bad_file is not None:
        raise RuntimeError(
            f"Backup verification failed: {bad_file}"
        )

    print("=== BACKUP CONTENTS ===")
    for filename in archive.namelist():
        print(" -", filename)

sha256 = hashlib.sha256()

with open(backup_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

print("\n=== BACKUP VERIFICATION ===")
print("Backup:", backup_path)
print("Size:", round(backup_path.stat().st_size / 1024, 2), "KiB")
print("ZIP integrity: PASS")
print("SHA256:", sha256.hexdigest())

=== BACKUP CONTENTS ===
 - model_registry.json
 - shared_model_loader.py
 - evaluate_shared.py
 - predict_shared.py
 - results/mobilenetv3_pytorch_shared.json
 - results/mobilenetv3_onnx_shared.json

=== BACKUP VERIFICATION ===
Backup: /content/drive/MyDrive/Flower_Competition/FlowerCompetition_SharedSystem_Backup.zip
Size: 5.05 KiB
ZIP integrity: PASS
SHA256: 06d6883e5fe276867de161bea83454122872b655b586b24437a38f4a07a960fe


# Part III — Accuracy Improvement

This section focuses on improving MobileNetV3 classification accuracy through validation error analysis and targeted experiments.

The original model checkpoint and optimized ONNX deployment are preserved as baselines.

### 31. Validation Error Analysis

Analyze the classification errors of the optimized MobileNetV3 ONNX model on the complete validation dataset.

Identify misclassified images, predicted classes, confidence scores, and the most frequently confused flower classes. Save the results for targeted accuracy improvement experiments.

In [ ]:
from pathlib import Path
from collections import Counter
import sys
import json

import torch
from PIL import Image
from torchvision import transforms

PROJECT = Path("/content/FlowerCompetition")
sys.path.insert(0, str(PROJECT))

from shared_model_loader import SharedModel

torch.set_num_threads(1)

model = SharedModel("mobilenetv3_onnx", device="cpu")

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

valid_dir = PROJECT / "data" / "valid"

image_paths = sorted(
    p for p in valid_dir.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

errors = []
correct = 0

for image_path in image_paths:
    true_class = image_path.parent.name

    with Image.open(image_path) as image:
        tensor = transform(
            image.convert("RGB")
        ).unsqueeze(0)

    logits = model.predict_logits(tensor)
    probabilities = torch.softmax(logits, dim=1)[0]

    confidence, predicted_index = probabilities.max(dim=0)

    predicted_index = int(predicted_index.item())

    idx_to_class = {
        index: name
        for name, index in model.class_to_idx.items()
    }

    predicted_class = idx_to_class[predicted_index]

    if predicted_class == true_class:
        correct += 1
    else:
        errors.append({
            "image": str(image_path),
            "true_class": true_class,
            "predicted_class": predicted_class,
            "confidence": round(
                float(confidence.item()), 6
            )
        })

accuracy = correct / len(image_paths)

print("=== VALIDATION ERROR ANALYSIS ===")
print("Total images:", len(image_paths))
print("Correct predictions:", correct)
print("Misclassified images:", len(errors))
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\n=== MOST FREQUENTLY MISCLASSIFIED CLASSES ===")

class_errors = Counter(
    error["true_class"] for error in errors
)

for class_name, count in class_errors.most_common(10):
    print(f"Class {class_name}: {count} errors")

print("\n=== MOST COMMON CONFUSIONS ===")

confusions = Counter(
    (error["true_class"], error["predicted_class"])
    for error in errors
)

for (true_class, predicted_class), count in confusions.most_common(10):
    print(
        f"True {true_class} -> Predicted {predicted_class}: "
        f"{count} images"
    )

print("\n=== MISCLASSIFIED IMAGES ===")

for error in errors:
    print(
        f"{Path(error['image']).name} | "
        f"True: {error['true_class']} | "
        f"Predicted: {error['predicted_class']} | "
        f"Confidence: {error['confidence'] * 100:.2f}%"
    )

# Save error analysis
results_dir = PROJECT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "mobilenetv3_validation_errors.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "total_images": len(image_paths),
            "correct": correct,
            "accuracy": accuracy,
            "errors": errors
        },
        f,
        indent=4
    )

print("\nError analysis saved:", output_path)

=== VALIDATION ERROR ANALYSIS ===
Total images: 818
Correct predictions: 793
Misclassified images: 25
Accuracy: 96.94%

=== MOST FREQUENTLY MISCLASSIFIED CLASSES ===
Class 43: 3 errors
Class 4: 2 errors
Class 42: 2 errors
Class 51: 2 errors
Class 91: 2 errors
Class 96: 2 errors
Class 18: 1 errors
Class 39: 1 errors
Class 48: 1 errors
Class 52: 1 errors

=== MOST COMMON CONFUSIONS ===
True 42 -> Predicted 43: 2 images
True 18 -> Predicted 83: 1 images
True 39 -> Predicted 4: 1 images
True 4 -> Predicted 3: 1 images
True 4 -> Predicted 88: 1 images
True 43 -> Predicted 68: 1 images
True 43 -> Predicted 15: 1 images
True 43 -> Predicted 85: 1 images
True 48 -> Predicted 65: 1 images
True 51 -> Predicted 45: 1 images

=== MISCLASSIFIED IMAGES ===
image_04299.jpg | True: 18 | Predicted: 83 | Confidence: 76.60%
image_07039.jpg | True: 39 | Predicted: 4 | Confidence: 58.59%
image_05677.jpg | True: 4 | Predicted: 3 | Confidence: 86.90%
image_05680.jpg | True: 4 | Predicted: 88 | Confidence: 26

### 32. Visualize Misclassified Images

Visualize all misclassified validation images with their ground-truth labels, predicted labels, and prediction confidence scores.

Inspect visual similarities, image quality, and potential sources of classification errors before selecting an accuracy improvement strategy.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from PIL import Image

PROJECT = Path("/content/FlowerCompetition")

errors_path = (
    PROJECT / "results" /
    "mobilenetv3_validation_errors.json"
)

with open(errors_path, "r", encoding="utf-8") as f:
    error_data = json.load(f)

errors = error_data["errors"]

cols = 5
rows = (len(errors) + cols - 1) // cols

fig, axes = plt.subplots(
    rows, cols,
    figsize=(18, rows * 3.5)
)

axes = axes.flatten()

for i, error in enumerate(errors):
    image_path = Path(error["image"])

    with Image.open(image_path) as img:
        image = img.convert("RGB").copy()

    axes[i].imshow(image)

    axes[i].set_title(
        f"True: {error['true_class']} | "
        f"Pred: {error['predicted_class']}\n"
        f"Confidence: {error['confidence'] * 100:.1f}%",
        fontsize=10
    )

    axes[i].axis("off")

for j in range(len(errors), len(axes)):
    axes[j].axis("off")

plt.suptitle(
    "MobileNetV3 — Misclassified Validation Images",
    fontsize=16
)

plt.tight_layout()

output_path = (
    PROJECT / "results" /
    "mobilenetv3_validation_errors.png"
)

plt.savefig(
    output_path,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

print("Total errors visualized:", len(errors))
print("Figure saved:", output_path)

Output hidden; open in https://colab.research.google.com to view.

### 33. Analyze TTA Impact on Classification Errors

Evaluate horizontal-flip test-time augmentation on the complete validation dataset.

Compare standard inference with TTA, identify corrected and newly introduced classification errors, and assess whether the accuracy improvement justifies the additional inference cost.

In [ ]:
from pathlib import Path
from collections import Counter
import sys
import json
import torch
from PIL import Image
from torchvision import transforms
from torchvision.transforms import functional as TF

PROJECT = Path("/content/FlowerCompetition")
sys.path.insert(0, str(PROJECT))

from shared_model_loader import SharedModel

torch.set_num_threads(1)

model = SharedModel("mobilenetv3_onnx", device="cpu")

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

idx_to_class = {
    index: name
    for name, index in model.class_to_idx.items()
}

valid_dir = PROJECT / "data" / "valid"

image_paths = sorted(
    p for p in valid_dir.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

baseline_correct = 0
tta_correct = 0

corrected_errors = []
new_errors = []

for image_path in image_paths:
    true_class = image_path.parent.name

    with Image.open(image_path) as image:
        tensor = transform(
            image.convert("RGB")
        ).unsqueeze(0)

    # Standard inference
    logits_original = model.predict_logits(tensor)

    # Horizontal-flip inference
    flipped_tensor = TF.hflip(tensor)
    logits_flipped = model.predict_logits(flipped_tensor)

    # Average logits
    logits_tta = (
        logits_original + logits_flipped
    ) / 2

    baseline_idx = int(
        logits_original.argmax(dim=1).item()
    )

    tta_idx = int(
        logits_tta.argmax(dim=1).item()
    )

    baseline_class = idx_to_class[baseline_idx]
    tta_class = idx_to_class[tta_idx]

    baseline_is_correct = baseline_class == true_class
    tta_is_correct = tta_class == true_class

    baseline_correct += int(baseline_is_correct)
    tta_correct += int(tta_is_correct)

    record = {
        "image": image_path.name,
        "true_class": true_class,
        "baseline_prediction": baseline_class,
        "tta_prediction": tta_class
    }

    if not baseline_is_correct and tta_is_correct:
        corrected_errors.append(record)

    elif baseline_is_correct and not tta_is_correct:
        new_errors.append(record)

total = len(image_paths)

print("=== TTA ACCURACY COMPARISON ===")
print(
    f"Baseline: {baseline_correct}/{total} "
    f"({100 * baseline_correct / total:.2f}%)"
)
print(
    f"TTA: {tta_correct}/{total} "
    f"({100 * tta_correct / total:.2f}%)"
)

print("\nErrors corrected by TTA:", len(corrected_errors))
print("New errors introduced by TTA:", len(new_errors))
print("Net correct predictions gained:", tta_correct - baseline_correct)

print("\n=== CORRECTED ERRORS ===")
for item in corrected_errors:
    print(item)

print("\n=== NEW ERRORS ===")
for item in new_errors:
    print(item)

results = {
    "baseline_correct": baseline_correct,
    "tta_correct": tta_correct,
    "total": total,
    "corrected_errors": corrected_errors,
    "new_errors": new_errors
}

output_path = PROJECT / "results" / "mobilenetv3_tta_analysis.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print("\nResults saved:", output_path)

=== TTA ACCURACY COMPARISON ===
Baseline: 793/818 (96.94%)
TTA: 794/818 (97.07%)

Errors corrected by TTA: 3
New errors introduced by TTA: 2
Net correct predictions gained: 1

=== CORRECTED ERRORS ===
{'image': 'image_05701.jpg', 'true_class': '42', 'baseline_prediction': '43', 'tta_prediction': '42'}
{'image': 'image_02443.jpg', 'true_class': '43', 'baseline_prediction': '85', 'tta_prediction': '43'}
{'image': 'image_03222.jpg', 'true_class': '65', 'baseline_prediction': '5', 'tta_prediction': '65'}

=== NEW ERRORS ===
{'image': 'image_06621.jpg', 'true_class': '3', 'baseline_prediction': '3', 'tta_prediction': '11'}
{'image': 'image_06443.jpg', 'true_class': '33', 'baseline_prediction': '33', 'tta_prediction': '30'}

Results saved: /content/FlowerCompetition/results/mobilenetv3_tta_analysis.json


### 34. Inspect Original Training Configuration

Inspect the original MobileNetV3 training notebook to identify the optimizer, learning rate, training schedule, frozen layers, and checkpoint selection strategy before planning further fine-tuning experiments.

In [ ]:
from pathlib import Path
import json

PROJECT = Path("/content/FlowerCompetition")

notebooks = sorted(
    list(PROJECT.rglob("*.ipynb"))
)

print("=== AVAILABLE TRAINING NOTEBOOKS ===")

for notebook in notebooks:
    print(notebook.relative_to(PROJECT))

if not notebooks:
    print(
        "\nNo training notebook found in the project folder."
        "\nWe can inspect the original mobilenet.ipynb separately."
    )
else:
    keywords = [
        "optimizer",
        "learning_rate",
        "lr=",
        "scheduler",
        "requires_grad",
        "epochs",
        "mobilenet_v3_large",
        "valid_accuracy",
    ]

    for notebook_path in notebooks:
        if "mobilenet" not in notebook_path.name.lower():
            continue

        print(f"\n=== {notebook_path.name} ===")

        with open(notebook_path, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        for index, cell in enumerate(notebook.get("cells", [])):
            if cell.get("cell_type") != "code":
                continue

            source = "".join(cell.get("source", []))

            if any(k in source.lower() for k in keywords):
                print(f"\n--- Cell {index} ---")
                print(source[:3500])

=== AVAILABLE TRAINING NOTEBOOKS ===

No training notebook found in the project folder.
We can inspect the original mobilenet.ipynb separately.


### 35. Locate Original MobileNetV3 Training Notebook

Search Google Drive for the original MobileNetV3 training notebook to recover its training configuration before planning further fine-tuning experiments.

In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")

print("=== SEARCHING FOR MOBILENET NOTEBOOKS ===")

notebooks = sorted(
    p for p in drive.rglob("*.ipynb")
    if "mobilenet" in p.name.lower()
)

if notebooks:
    for notebook in notebooks:
        print(notebook)
else:
    print("No MobileNet notebooks found in Google Drive.")

=== SEARCHING FOR MOBILENET NOTEBOOKS ===
/content/drive/MyDrive/Colab Notebooks/mobilenet_optimization_experiments.ipynb


### 36. Inspect Available MobileNetV3 Training Configuration

Inspect the available MobileNetV3 optimization notebook for training configuration details, including learning rate, optimizer, scheduler, frozen layers, and training epochs.

In [ ]:
from pathlib import Path
import json

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "mobilenet_optimization_experiments.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "optimizer",
    "learning_rate",
    "weight_decay",
    "scheduler",
    "requires_grad",
    "num_epochs",
    "epochs",
    "fine_tun",
    "mobilenet_v3_large",
]

print("=== TRAINING CONFIGURATION SEARCH ===")

matches = 0

for index, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    matched = [
        keyword for keyword in keywords
        if keyword in source.lower()
    ]

    if matched:
        matches += 1

        print(f"\n--- Cell {index} ---")
        print("Keywords:", ", ".join(matched))
        print(source[:2500])

print("\nMatching cells:", matches)

=== TRAINING CONFIGURATION SEARCH ===

--- Cell 47 ---
Keywords: mobilenet_v3_large
from pathlib import Path
import json

PROJECT = Path("/content/FlowerCompetition")

registry = {
    "mobilenetv3_pytorch": {
        "architecture": "mobilenet_v3_large",
        "backend": "pytorch",
        "checkpoint": "mobilenetv3_finetuned_best.pth",
        "num_classes": 102,
        "image_size": 224,
        "preprocessing": "imagenet_resize256_centercrop224",
        "class_mapping": "checkpoint",
        "enabled": True
    },
    "mobilenetv3_onnx": {
        "architecture": "mobilenet_v3_large",
        "backend": "onnxruntime",
        "checkpoint": "onnx/mobilenetv3_fp32.onnx",
        "num_classes": 102,
        "image_size": 224,
        "preprocessing": "imagenet_resize256_centercrop224",
        "class_mapping": "class_to_idx.json",
        "enabled": True
    }
}

registry_path = PROJECT / "model_registry.json"

with open(registry_path, "w", encoding="utf-8") as f:
    json.dump(re

### 37. Fine-Tuning Environment Check

Verify GPU availability and the original MobileNetV3 checkpoint before starting controlled fine-tuning experiments.

The baseline checkpoint will remain unchanged, and any improved model will be saved under a separate filename.

In [ ]:
from pathlib import Path
import torch

PROJECT = Path("/content/FlowerCompetition")
CHECKPOINT = PROJECT / "mobilenetv3_finetuned_best.pth"

print("=== FINE-TUNING ENVIRONMENT ===")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3), 2
        ),
        "GiB"
    )
else:
    print("GPU: NOT AVAILABLE")

print("\n=== PROJECT FILES ===")

for name in [
    "mobilenetv3_finetuned_best.pth",
    "class_to_idx.json"
]:
    print(name, (PROJECT / name).is_file())

for split in ["train", "valid"]:
    print(
        f"{split} dataset:",
        (PROJECT / "data" / split).is_dir()
    )

print("\nOriginal checkpoint size:",
      round(CHECKPOINT.stat().st_size / (1024 ** 2), 2),
      "MiB")

=== FINE-TUNING ENVIRONMENT ===
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GiB

=== PROJECT FILES ===
mobilenetv3_finetuned_best.pth True
class_to_idx.json True
train dataset: True
valid dataset: True

Original checkpoint size: 16.73 MiB


### 38. Prepare Fine-Tuning Data Loaders

Prepare reproducible training and validation data loaders for controlled MobileNetV3 fine-tuning.

Apply moderate data augmentation to the training split, preserve deterministic validation preprocessing, and use the original checkpoint's class mapping to ensure consistent evaluation.

In [ ]:
from pathlib import Path
import random
import numpy as np
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

PROJECT = Path("/content/FlowerCompetition")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
BATCH_SIZE = 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

checkpoint = torch.load(
    PROJECT / "mobilenetv3_finetuned_best.pth",
    map_location="cpu",
    weights_only=True
)

class_to_idx = {
    str(k): int(v)
    for k, v in checkpoint["class_to_idx"].items()
}

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.85, 1.0),
        ratio=(0.9, 1.1)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

valid_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = ImageFolder(
    PROJECT / "data" / "train",
    transform=train_transform
)

valid_dataset = ImageFolder(
    PROJECT / "data" / "valid",
    transform=valid_transform
)

# ImageFolder uses alphabetical ordering.
# Convert labels to the checkpoint's numerical class order.
for dataset in [train_dataset, valid_dataset]:
    assert set(dataset.classes) == set(class_to_idx), (
        "Dataset classes do not match checkpoint classes."
    )

    dataset.target_transform = (
        lambda label, ds=dataset:
        class_to_idx[ds.classes[label]]
    )

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("=== FINE-TUNING DATA LOADERS ===")
print("Device:", DEVICE)
print("Train images:", len(train_dataset))
print("Validation images:", len(valid_dataset))
print("Number of classes:", len(class_to_idx))
print("Batch size:", BATCH_SIZE)

images, labels = next(iter(train_loader))

print("\n=== FIRST TRAINING BATCH ===")
print("Image tensor shape:", tuple(images.shape))
print("Label tensor shape:", tuple(labels.shape))
print("Label range:", labels.min().item(), "-", labels.max().item())

assert len(train_dataset) == 6552
assert len(valid_dataset) == 818
assert len(class_to_idx) == 102
assert images.shape[1:] == (3, 224, 224)
assert labels.min() >= 0
assert labels.max() < 102

print("\nPASS: Data loaders are ready for fine-tuning.")

=== FINE-TUNING DATA LOADERS ===
Device: cuda
Train images: 6552
Validation images: 818
Number of classes: 102
Batch size: 32

=== FIRST TRAINING BATCH ===
Image tensor shape: (32, 3, 224, 224)
Label tensor shape: (32,)
Label range: 0 - 100

PASS: Data loaders are ready for fine-tuning.


### 39. Prepare MobileNetV3 for Controlled Fine-Tuning

Initialize MobileNetV3-Large from the best existing checkpoint.

Freeze the early feature extraction layers and fine-tune the final feature blocks and classifier using a low learning rate. Preserve the original checkpoint and save experimental models separately.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = PROJECT / "mobilenetv3_finetuned_best.pth"

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

# Rebuild the original architecture
model = models.mobilenet_v3_large(weights=None)

model.classifier[3] = nn.Linear(
    model.classifier[3].in_features,
    102
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# Freeze all parameters initially
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the last 3 feature blocks
for block in model.features[-3:]:
    for param in block.parameters():
        param.requires_grad = True

# Unfreeze the classifier
for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(DEVICE)

# Keep BatchNorm running statistics fixed during fine-tuning
def freeze_batchnorm_statistics(model):
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    [
        {
            "params": [
                p for p in model.features.parameters()
                if p.requires_grad
            ],
            "lr": 1e-5
        },
        {
            "params": [
                p for p in model.classifier.parameters()
                if p.requires_grad
            ],
            "lr": 3e-5
        }
    ],
    weight_decay=1e-4
)

EPOCHS = 5

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("=== FINE-TUNING CONFIGURATION ===")
print("Device:", DEVICE)
print("Architecture: MobileNetV3-Large")
print("Original checkpoint:", CHECKPOINT_PATH.name)
print("Original validation accuracy:",
      checkpoint.get("valid_accuracy"))
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_params / total_params:.2f}%"
)
print("Feature learning rate: 1e-5")
print("Classifier learning rate: 3e-5")
print("Optimizer: AdamW")
print("Scheduler: CosineAnnealingLR")
print("Planned epochs:", EPOCHS)

print("\nModel loaded and configured successfully.")
print("Training has NOT started.")

=== FINE-TUNING CONFIGURATION ===
Device: cuda
Architecture: MobileNetV3-Large
Original checkpoint: mobilenetv3_finetuned_best.pth
Original validation accuracy: 96.9437652811736
Total parameters: 4,332,694
Trainable parameters: 3,110,982
Trainable percentage: 71.80%
Feature learning rate: 1e-5
Classifier learning rate: 3e-5
Optimizer: AdamW
Scheduler: CosineAnnealingLR
Planned epochs: 5

Model loaded and configured successfully.
Training has NOT started.


### 40. Controlled MobileNetV3 Fine-Tuning

Fine-tune the pretrained MobileNetV3-Large model using a low learning rate and moderate augmentation.

Evaluate the model after each epoch, monitor validation accuracy and loss, and save a separate experimental checkpoint only when the model exceeds the original baseline accuracy.

Preserve the original checkpoint and apply early stopping when validation performance does not improve.

In [ ]:
import copy
import json
import time
import torch
from pathlib import Path
from tqdm.auto import tqdm

PROJECT = Path("/content/FlowerCompetition")
RESULTS_DIR = PROJECT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

BEST_PATH = PROJECT / "mobilenetv3_finetuned_v2_best.pth"
HISTORY_PATH = RESULTS_DIR / "mobilenetv3_finetuning_v2_history.json"

BASELINE_CORRECT = 793
BEST_CORRECT = BASELINE_CORRECT
BEST_VALID_LOSS = float("inf")

PATIENCE = 3
epochs_without_improvement = 0

history = []

def evaluate_finetuned_model(model, loader):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            predictions = outputs.argmax(dim=1)

            total_loss += loss.item() * labels.size(0)
            total_correct += (
                predictions == labels
            ).sum().item()

            total_samples += labels.size(0)

    return (
        total_loss / total_samples,
        total_correct,
        100 * total_correct / total_samples
    )


print("=== MOBILENETV3 FINE-TUNING V2 ===")
print("Baseline:", f"{BASELINE_CORRECT}/818")
print("Device:", DEVICE)
print("Maximum epochs:", EPOCHS)
print("Experimental checkpoint:", BEST_PATH.name)

for epoch in range(EPOCHS):
    start_time = time.time()

    model.train()

    # Freeze BatchNorm running statistics
    freeze_batchnorm_statistics(model)

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
        leave=False
    )

    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        predictions = outputs.argmax(dim=1)

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = 100 * train_correct / train_total

    valid_loss, valid_correct, valid_acc = (
        evaluate_finetuned_model(model, valid_loader)
    )

    scheduler.step()

    elapsed = time.time() - start_time

    record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_acc,
        "valid_correct": valid_correct,
        "epoch_seconds": elapsed
    }

    history.append(record)

    print(
        f"\nEpoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_acc:.2f}% "
        f"({valid_correct}/818) | "
        f"Time: {elapsed:.1f}s"
    )

    if valid_correct > BEST_CORRECT:
        BEST_CORRECT = valid_correct
        BEST_VALID_LOSS = valid_loss

        torch.save(
            {
                "model_state_dict": copy.deepcopy(
                    model.state_dict()
                ),
                "class_to_idx": class_to_idx,
                "epoch": epoch + 1,
                "valid_accuracy": valid_acc,
                "valid_correct": valid_correct,
                "architecture": "mobilenet_v3_large",
                "image_size": 224,
                "experiment": "controlled_finetuning_v2"
            },
            BEST_PATH
        )

        print(
            f"  NEW BEST: {valid_correct}/818 "
            f"— checkpoint saved."
        )

        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

        print(
            "  No improvement over the best "
            "validation correct count."
        )

    with open(HISTORY_PATH, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=4)

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\n=== FINE-TUNING SUMMARY ===")
print("Original baseline:", f"{BASELINE_CORRECT}/818")
print("Best experimental:", f"{BEST_CORRECT}/818")
print("Improvement:", BEST_CORRECT - BASELINE_CORRECT, "images")
print("History saved:", HISTORY_PATH)

if BEST_PATH.exists():
    print("Improved checkpoint:", BEST_PATH)
else:
    print("No improved checkpoint saved.")
    print("Keep the original MobileNetV3 model.")

=== MOBILENETV3 FINE-TUNING V2 ===
Baseline: 793/818
Device: cuda
Maximum epochs: 5
Experimental checkpoint: mobilenetv3_finetuned_v2_best.pth


Epoch 1/5:   0%|          | 0/205 [00:00<?, ?it/s]


Epoch 1/5 | Train Loss: 0.0422 | Train Acc: 98.78% | Valid Loss: 0.2121 | Valid Acc: 95.48% (781/818) | Time: 52.7s
  No improvement over the best validation correct count.


Epoch 2/5:   0%|          | 0/205 [00:00<?, ?it/s]


Epoch 2/5 | Train Loss: 0.0205 | Train Acc: 99.36% | Valid Loss: 0.2151 | Valid Acc: 94.99% (777/818) | Time: 59.8s
  No improvement over the best validation correct count.


Epoch 3/5:   0%|          | 0/205 [00:00<?, ?it/s]


Epoch 3/5 | Train Loss: 0.0172 | Train Acc: 99.54% | Valid Loss: 0.2100 | Valid Acc: 94.87% (776/818) | Time: 49.7s
  No improvement over the best validation correct count.

Early stopping triggered.

=== FINE-TUNING SUMMARY ===
Original baseline: 793/818
Best experimental: 793/818
Improvement: 0 images
History saved: /content/FlowerCompetition/results/mobilenetv3_finetuning_v2_history.json
No improved checkpoint saved.
Keep the original MobileNetV3 model.


### 41. Verify Original MobileNetV3 Checkpoint

Confirm that the original MobileNetV3 checkpoint remains intact after the unsuccessful fine-tuning experiment.

Evaluate the original model using the deterministic validation preprocessing and preserve the baseline for future optimization.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")

original_checkpoint = torch.load(
    PROJECT / "mobilenetv3_finetuned_best.pth",
    map_location="cpu",
    weights_only=True
)

original_model = models.mobilenet_v3_large(weights=None)

original_model.classifier[3] = nn.Linear(
    original_model.classifier[3].in_features,
    102
)

original_model.load_state_dict(
    original_checkpoint["model_state_dict"],
    strict=True
)

original_model = original_model.to(DEVICE)
original_model.eval()

correct = 0
total = 0

with torch.inference_mode():
    for images, labels in valid_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = original_model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total

print("=== ORIGINAL CHECKPOINT VERIFICATION ===")
print("Correct predictions:", f"{correct}/{total}")
print("Validation accuracy:", f"{accuracy:.2f}%")
print("Expected baseline: 793/818 (96.94%)")

assert total == 818

if correct == 793:
    print("\nPASS: Original baseline performance preserved.")
else:
    print("\nWARNING: Baseline differs. Investigate before continuing.")

=== ORIGINAL CHECKPOINT VERIFICATION ===
Correct predictions: 793/818
Validation accuracy: 96.94%
Expected baseline: 793/818 (96.94%)

PASS: Original baseline performance preserved.


### 42. Analyze Confidence-Based Selective TTA

Evaluate selective horizontal-flip test-time augmentation using different confidence thresholds.

Run a second inference pass only when the baseline prediction confidence falls below a specified threshold. Compare accuracy, the number of additional inference passes, and estimated computational cost against the original model and full TTA.

In [ ]:
import sys
import json
import numpy as np
import torch
from pathlib import Path
from torchvision.transforms.functional import hflip

PROJECT = Path("/content/FlowerCompetition")
sys.path.insert(0, str(PROJECT))

from shared_model_loader import SharedModel

onnx_model = SharedModel("mobilenetv3_onnx", device="cpu")

thresholds = [0.50, 0.70, 0.80, 0.90, 0.95, 0.99]

records = []

with torch.inference_mode():
    for images, labels in valid_loader:
        images = images.cpu()
        labels = labels.cpu()

        for image, label in zip(images, labels):
            batch = image.unsqueeze(0)

            baseline_logits = onnx_model.predict_logits(batch)
            flipped_logits = onnx_model.predict_logits(hflip(batch))

            baseline_probs = torch.softmax(
                baseline_logits, dim=1
            )

            confidence = baseline_probs.max().item()

            records.append({
                "true_label": int(label.item()),
                "confidence": confidence,
                "baseline_logits": baseline_logits.squeeze(0).numpy(),
                "flipped_logits": flipped_logits.squeeze(0).numpy()
            })

print("=== SELECTIVE TTA COMPARISON ===")

baseline_correct = sum(
    int(np.argmax(r["baseline_logits"]) == r["true_label"])
    for r in records
)

print(f"Baseline: {baseline_correct}/{len(records)}")

results = []

for threshold in thresholds:
    correct = 0
    additional_passes = 0

    for r in records:
        logits = r["baseline_logits"]

        if r["confidence"] < threshold:
            logits = (
                r["baseline_logits"] + r["flipped_logits"]
            ) / 2

            additional_passes += 1

        prediction = int(np.argmax(logits))

        correct += int(prediction == r["true_label"])

    accuracy = 100 * correct / len(records)
    inference_multiplier = 1 + additional_passes / len(records)

    results.append({
        "threshold": threshold,
        "correct": correct,
        "accuracy": accuracy,
        "additional_passes": additional_passes,
        "inference_multiplier": inference_multiplier
    })

    print(
        f"Threshold {threshold:.2f} | "
        f"Accuracy: {accuracy:.2f}% "
        f"({correct}/{len(records)}) | "
        f"Extra passes: {additional_passes} | "
        f"Compute multiplier: {inference_multiplier:.3f}x"
    )

output_path = (
    PROJECT / "results" /
    "mobilenetv3_selective_tta_analysis.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print("\nResults saved:", output_path)

=== SELECTIVE TTA COMPARISON ===
Baseline: 793/818
Threshold 0.50 | Accuracy: 96.82% (792/818) | Extra passes: 5 | Compute multiplier: 1.006x
Threshold 0.70 | Accuracy: 96.94% (793/818) | Extra passes: 28 | Compute multiplier: 1.034x
Threshold 0.80 | Accuracy: 97.07% (794/818) | Extra passes: 39 | Compute multiplier: 1.048x
Threshold 0.90 | Accuracy: 97.07% (794/818) | Extra passes: 57 | Compute multiplier: 1.070x
Threshold 0.95 | Accuracy: 97.07% (794/818) | Extra passes: 80 | Compute multiplier: 1.098x
Threshold 0.99 | Accuracy: 97.07% (794/818) | Extra passes: 132 | Compute multiplier: 1.161x

Results saved: /content/FlowerCompetition/results/mobilenetv3_selective_tta_analysis.json


### 43. Benchmark Selective TTA on CPU

Measure end-to-end model inference latency for baseline ONNX and confidence-based selective TTA using a confidence threshold of 0.80.

Compare accuracy, mean latency, median latency, P95 latency, and the number of additional inference passes. Keep preprocessing outside the timed section and use the same validation samples for both methods.

In [ ]:
import sys
import time
import json
import numpy as np
import torch

from pathlib import Path
from torchvision.transforms.functional import hflip

PROJECT = Path("/content/FlowerCompetition")
sys.path.insert(0, str(PROJECT))

from shared_model_loader import SharedModel

onnx_model = SharedModel("mobilenetv3_onnx", device="cpu")

THRESHOLD = 0.80

# Preload validation images to exclude disk I/O and
# preprocessing from the timed inference benchmark.
samples = []

for images, labels in valid_loader:
    for image, label in zip(images, labels):
        samples.append((
            image.unsqueeze(0).cpu(),
            int(label.item())
        ))

def predict_baseline(image):
    logits = onnx_model.predict_logits(image)
    return int(logits.argmax(dim=1).item()), False

def predict_selective_tta(image):
    logits = onnx_model.predict_logits(image)

    confidence = torch.softmax(
        logits, dim=1
    ).max().item()

    used_tta = confidence < THRESHOLD

    if used_tta:
        flipped_logits = onnx_model.predict_logits(
            hflip(image)
        )
        logits = (logits + flipped_logits) / 2

    return int(logits.argmax(dim=1).item()), used_tta

# Warm up ONNX Runtime
for image, _ in samples[:20]:
    predict_baseline(image)
    predict_selective_tta(image)

def benchmark(predict_fn):
    times = []
    correct = 0
    extra_passes = 0

    for image, label in samples:
        start = time.perf_counter()

        prediction, used_tta = predict_fn(image)

        elapsed_ms = (
            time.perf_counter() - start
        ) * 1000

        times.append(elapsed_ms)
        correct += int(prediction == label)
        extra_passes += int(used_tta)

    return {
        "correct": correct,
        "total": len(samples),
        "accuracy": 100 * correct / len(samples),
        "mean_ms": float(np.mean(times)),
        "median_ms": float(np.median(times)),
        "p95_ms": float(np.percentile(times, 95)),
        "extra_passes": extra_passes
    }

baseline = benchmark(predict_baseline)
selective = benchmark(predict_selective_tta)

print("=== CPU INFERENCE BENCHMARK ===")

for name, result in [
    ("Baseline ONNX", baseline),
    ("Selective TTA (0.80)", selective)
]:
    print(f"\n{name}")
    print(
        f"Accuracy: {result['accuracy']:.2f}% "
        f"({result['correct']}/{result['total']})"
    )
    print(f"Mean latency: {result['mean_ms']:.3f} ms")
    print(f"Median latency: {result['median_ms']:.3f} ms")
    print(f"P95 latency: {result['p95_ms']:.3f} ms")
    print("Extra passes:", result["extra_passes"])

print("\n=== COMPARISON ===")
print(
    "Mean latency ratio:",
    f"{selective['mean_ms'] / baseline['mean_ms']:.3f}x"
)
print(
    "Accuracy gain:",
    selective["correct"] - baseline["correct"],
    "image(s)"
)

output_path = (
    PROJECT / "results" /
    "mobilenetv3_selective_tta_benchmark.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "threshold": THRESHOLD,
            "baseline": baseline,
            "selective_tta": selective
        },
        f,
        indent=4
    )

print("\nResults saved:", output_path)

=== CPU INFERENCE BENCHMARK ===

Baseline ONNX
Accuracy: 96.94% (793/818)
Mean latency: 7.248 ms
Median latency: 6.885 ms
P95 latency: 10.777 ms
Extra passes: 0

Selective TTA (0.80)
Accuracy: 97.07% (794/818)
Mean latency: 7.465 ms
Median latency: 6.866 ms
P95 latency: 12.086 ms
Extra passes: 39

=== COMPARISON ===
Mean latency ratio: 1.030x
Accuracy gain: 1 image(s)

Results saved: /content/FlowerCompetition/results/mobilenetv3_selective_tta_benchmark.json


### 44. Save MobileNetV3 Candidate Comparison

Consolidate the baseline ONNX and selective TTA benchmark results into a reproducible candidate comparison report.

Preserve both inference strategies for final evaluation and team model integration.

In [ ]:
import json
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
RESULTS = PROJECT / "results"

benchmark_path = (
    RESULTS / "mobilenetv3_selective_tta_benchmark.json"
)

with open(benchmark_path, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

baseline = benchmark["baseline"]
selective = benchmark["selective_tta"]

report = {
    "model": "MobileNetV3-Large",
    "backend": "ONNX Runtime CPU",
    "validation_images": 818,
    "model_file": "onnx/mobilenetv3_fp32.onnx",
    "candidates": {
        "baseline_onnx": {
            **baseline,
            "tta_enabled": False
        },
        "selective_tta_080": {
            **selective,
            "tta_enabled": True,
            "confidence_threshold": 0.80,
            "augmentation": "horizontal_flip",
            "aggregation": "mean_logits"
        }
    },
    "notes": [
        "Original PyTorch checkpoint preserved.",
        "Fine-tuning V2 did not improve validation accuracy.",
        "Selective TTA threshold was selected using validation data.",
        "CPU latency measured in Google Colab, not competition hardware.",
        "Final strategy requires independent evaluation."
    ]
}

output_path = RESULTS / "mobilenetv3_final_candidates.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4)

print("=== FINAL MOBILENETV3 CANDIDATES ===")

for name, result in report["candidates"].items():
    print(f"\n{name}")
    print(
        f"Accuracy: {result['accuracy']:.2f}% "
        f"({result['correct']}/{result['total']})"
    )
    print(f"Mean CPU latency: {result['mean_ms']:.3f} ms")
    print(f"P95 CPU latency: {result['p95_ms']:.3f} ms")

print("\nReport saved:", output_path)
print("Original checkpoint preserved.")
print("ONNX model preserved.")

=== FINAL MOBILENETV3 CANDIDATES ===

baseline_onnx
Accuracy: 96.94% (793/818)
Mean CPU latency: 7.248 ms
P95 CPU latency: 10.777 ms

selective_tta_080
Accuracy: 97.07% (794/818)
Mean CPU latency: 7.465 ms
P95 CPU latency: 12.086 ms

Report saved: /content/FlowerCompetition/results/mobilenetv3_final_candidates.json
Original checkpoint preserved.
ONNX model preserved.


### 45. Add Optional Selective TTA Inference

Create a separate inference module supporting confidence-based horizontal-flip test-time augmentation for MobileNetV3 ONNX.

Preserve the original shared prediction pipeline. Apply TTA only when the baseline softmax confidence is below 0.80, and average the original and flipped logits before selecting the final class.

In [ ]:
from pathlib import Path
import sys
import textwrap

PROJECT = Path("/content/FlowerCompetition")

script = r'''
from pathlib import Path
import sys

import torch
from PIL import Image
from torchvision import transforms
from torchvision.transforms.functional import hflip

PROJECT_DIR = Path(__file__).resolve().parent
sys.path.insert(0, str(PROJECT_DIR))

from shared_model_loader import SharedModel


class SelectiveTTAPredictor:

    def __init__(self, threshold=0.80):
        self.model = SharedModel(
            "mobilenetv3_onnx",
            device="cpu"
        )

        self.threshold = float(threshold)

        self.idx_to_class = {
            int(idx): str(name)
            for name, idx in self.model.class_to_idx.items()
        }

        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def predict(self, image_path, top_k=5):

        image = Image.open(image_path).convert("RGB")
        tensor = self.transform(image).unsqueeze(0)

        with torch.inference_mode():

            baseline_logits = self.model.predict_logits(
                tensor
            )

            baseline_probs = torch.softmax(
                baseline_logits,
                dim=1
            )

            baseline_confidence = (
                baseline_probs.max().item()
            )

            used_tta = baseline_confidence < self.threshold

            final_logits = baseline_logits

            if used_tta:
                flipped_logits = self.model.predict_logits(
                    hflip(tensor)
                )

                final_logits = (
                    baseline_logits + flipped_logits
                ) / 2

            probabilities = torch.softmax(
                final_logits,
                dim=1
            )

            values, indices = torch.topk(
                probabilities,
                k=min(top_k, probabilities.shape[1]),
                dim=1
            )

        predictions = []

        for probability, index in zip(
            values[0].tolist(),
            indices[0].tolist()
        ):
            predictions.append({
                "class_id": self.idx_to_class[index],
                "confidence": float(probability)
            })

        return {
            "predictions": predictions,
            "used_tta": used_tta,
            "baseline_confidence": baseline_confidence,
            "threshold": self.threshold
        }


if __name__ == "__main__":

    if len(sys.argv) < 2:
        print(
            "Usage: python predict_selective_tta.py "
            "<image_path>"
        )
        sys.exit(1)

    predictor = SelectiveTTAPredictor()

    result = predictor.predict(sys.argv[1])

    print("TTA applied:", result["used_tta"])
    print(
        "Baseline confidence:",
        f"{result['baseline_confidence']:.4f}"
    )

    print("\nTop predictions:")

    for item in result["predictions"]:
        print(
            item["class_id"],
            f"{item['confidence']:.4f}"
        )
'''

script_path = PROJECT / "predict_selective_tta.py"

script_path.write_text(
    textwrap.dedent(script),
    encoding="utf-8"
)

print("=== SELECTIVE TTA MODULE ===")
print("Created:", script_path)
print("File size:", script_path.stat().st_size, "bytes")

# Verify Python syntax without running inference
compile(
    script_path.read_text(encoding="utf-8"),
    str(script_path),
    "exec"
)

print("PASS: Python syntax is valid.")
print("Original predict_shared.py was not modified.")

=== SELECTIVE TTA MODULE ===
Created: /content/FlowerCompetition/predict_selective_tta.py
File size: 3131 bytes
PASS: Python syntax is valid.
Original predict_shared.py was not modified.


### 46. Validate Selective TTA Prediction Module

Validate the standalone Selective TTA inference module using known validation examples.

Check whether the module reproduces the expected predictions, applies augmentation according to the confidence threshold, and returns valid top-k class probabilities.

In [ ]:
import sys
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
sys.path.insert(0, str(PROJECT))

from predict_selective_tta import SelectiveTTAPredictor

predictor = SelectiveTTAPredictor(threshold=0.80)

test_cases = [
    ("image_05701.jpg", "42"),
    ("image_02443.jpg", "43"),
    ("image_03222.jpg", "65"),
    ("image_06621.jpg", "3"),
    ("image_06443.jpg", "33"),
]

print("=== SELECTIVE TTA MODULE VALIDATION ===")

all_images = {
    path.name: path
    for path in (PROJECT / "data" / "valid").rglob("*.jpg")
}

for filename, expected_class in test_cases:

    if filename not in all_images:
        print(f"\nSKIP: {filename} not found")
        continue

    image_path = all_images[filename]

    result = predictor.predict(
        image_path,
        top_k=5
    )

    predicted_class = result["predictions"][0]["class_id"]
    confidence = result["predictions"][0]["confidence"]

    print(f"\nImage: {filename}")
    print("True class:", expected_class)
    print("Predicted class:", predicted_class)
    print("Confidence:", f"{confidence:.4f}")
    print("Baseline confidence:",
          f"{result['baseline_confidence']:.4f}")
    print("TTA applied:", result["used_tta"])
    print(
        "Correct:",
        predicted_class == expected_class
    )

    assert len(result["predictions"]) == 5
    assert 0 <= confidence <= 1

print("\nPASS: Selective TTA inference module executed.")

=== SELECTIVE TTA MODULE VALIDATION ===

Image: image_05701.jpg
True class: 42
Predicted class: 42
Confidence: 0.5815
Baseline confidence: 0.7900
TTA applied: True
Correct: True

Image: image_02443.jpg
True class: 43
Predicted class: 43
Confidence: 0.5627
Baseline confidence: 0.6516
TTA applied: True
Correct: True

Image: image_03222.jpg
True class: 65
Predicted class: 65
Confidence: 0.7800
Baseline confidence: 0.5312
TTA applied: True
Correct: True

Image: image_06621.jpg
True class: 3
Predicted class: 11
Confidence: 0.6957
Baseline confidence: 0.5039
TTA applied: True
Correct: False

Image: image_06443.jpg
True class: 33
Predicted class: 30
Confidence: 0.3535
Baseline confidence: 0.2974
TTA applied: True
Correct: False

PASS: Selective TTA inference module executed.


### 47. Backup Updated MobileNetV3 Experiments

Create a verified Google Drive backup containing the latest MobileNetV3 experiment results, shared inference scripts, model registry, and Selective TTA implementation.

Preserve previous backups and validate the new ZIP archive after creation.

In [ ]:
from pathlib import Path
import zipfile
import hashlib

PROJECT = Path("/content/FlowerCompetition")
DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

DRIVE.mkdir(parents=True, exist_ok=True)

backup_path = DRIVE / "MobileNetV3_Accuracy_Experiments_Backup.zip"

files_to_backup = [
    "model_registry.json",
    "class_to_idx.json",
    "shared_model_loader.py",
    "evaluate_shared.py",
    "predict_shared.py",
    "predict_selective_tta.py",
    "results/mobilenetv3_validation_errors.json",
    "results/mobilenetv3_validation_errors.png",
    "results/mobilenetv3_tta_analysis.json",
    "results/mobilenetv3_finetuning_v2_history.json",
    "results/mobilenetv3_selective_tta_analysis.json",
    "results/mobilenetv3_selective_tta_benchmark.json",
    "results/mobilenetv3_final_candidates.json",
]

missing = [
    name for name in files_to_backup
    if not (PROJECT / name).is_file()
]

if missing:
    print("Missing files:")
    for name in missing:
        print(" -", name)
    raise FileNotFoundError(
        "Backup cancelled because required files are missing."
    )

with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for name in files_to_backup:
        zf.write(
            PROJECT / name,
            arcname=name
        )

with zipfile.ZipFile(backup_path, "r") as zf:
    corrupted = zf.testzip()
    archived_files = zf.namelist()

assert corrupted is None
assert len(archived_files) == len(files_to_backup)

sha256 = hashlib.sha256()

with open(backup_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

print("=== MOBILE NET V3 EXPERIMENT BACKUP ===")
print("Backup:", backup_path)
print("Files archived:", len(archived_files))
print(
    "Size:",
    round(backup_path.stat().st_size / (1024 ** 2), 3),
    "MiB"
)
print("SHA256:", sha256.hexdigest())
print("ZIP integrity: PASS")
print("\nPrevious backups were not modified.")

=== MOBILE NET V3 EXPERIMENT BACKUP ===
Backup: /content/drive/MyDrive/Flower_Competition/MobileNetV3_Accuracy_Experiments_Backup.zip
Files archived: 13
Size: 8.085 MiB
SHA256: a509eff221f4066ad1c0f5c9a25e33920063a800817960fa6f959fe67164de51
ZIP integrity: PASS

Previous backups were not modified.


### 48. Inspect Available Team Model Checkpoints

Identify all available PyTorch model checkpoints in the project and Google Drive competition folder.

Report their paths, file sizes, and checkpoint metadata without modifying or loading model weights into an architecture.

In [ ]:
from pathlib import Path
import torch

PROJECT = Path("/content/FlowerCompetition")
DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

print("=== AVAILABLE TEAM CHECKPOINTS ===")

roots = [PROJECT, DRIVE]
seen = set()

for root in roots:
    if not root.exists():
        continue

    for path in sorted(root.rglob("*.pth")):
        if path.resolve() in seen:
            continue

        seen.add(path.resolve())

        print("\nFile:", path)
        print(
            "Size:",
            round(path.stat().st_size / (1024 ** 2), 2),
            "MiB"
        )

        try:
            checkpoint = torch.load(
                path,
                map_location="cpu",
                weights_only=True
            )

            if isinstance(checkpoint, dict):
                print("Checkpoint keys:", list(checkpoint.keys())[:15])

                for key in [
                    "architecture",
                    "epoch",
                    "valid_accuracy",
                    "class_to_idx",
                ]:
                    if key in checkpoint:
                        value = checkpoint[key]

                        if key == "class_to_idx":
                            print(
                                "Class mapping entries:",
                                len(value)
                            )
                        else:
                            print(key + ":", value)

            else:
                print("Checkpoint type:", type(checkpoint).__name__)

        except Exception as e:
            print("Metadata inspection failed:", str(e)[:200])

print("\nInspection complete.")


=== AVAILABLE TEAM CHECKPOINTS ===

File: /content/FlowerCompetition/mobilenetv3_finetuned_best.pth
Size: 16.73 MiB
Checkpoint keys: ['model_state_dict', 'class_to_idx', 'epoch', 'valid_accuracy', 'architecture', 'image_size']
architecture: mobilenet_v3_large
epoch: 3
valid_accuracy: 96.9437652811736
Class mapping entries: 102

File: /content/FlowerCompetition/mobilenetv3_fp16.pth
Size: 8.42 MiB
Checkpoint keys: ['model_state_dict', 'class_to_idx', 'architecture', 'image_size', 'precision']
architecture: mobilenet_v3_large
Class mapping entries: 102

File: /content/FlowerCompetition/mobilenetv3_head_best.pth
Size: 16.73 MiB
Checkpoint keys: ['model_state_dict', 'class_to_idx', 'epoch', 'valid_accuracy']
epoch: 5
valid_accuracy: 94.9877750611247
Class mapping entries: 102

File: /content/FlowerCompetition/mobilenetv3_int8_experimental.pth
Size: 4.61 MiB
Checkpoint keys: ['model_state_dict', 'architecture', 'precision', 'class_to_idx', 'num_classes', 'image_size', 'quantization_backend',

/usr/local/lib/python3.13/dist-packages/torch/_utils.py:476: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,


### 49. Locate Teammate Model Checkpoints in Google Drive

Search Google Drive for EfficientNet and other team model checkpoint files.

Identify available model files and their sizes before integrating them into the shared evaluation pipeline.

In [ ]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

extensions = {".pth", ".pt", ".onnx", ".ckpt"}

print("=== TEAM MODEL SEARCH ===")

found = []

for path in DRIVE.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in extensions:
        continue

    found.append(path)

if found:
    for path in sorted(found):
        size_mb = path.stat().st_size / (1024 ** 2)

        print("\nFile:", path)
        print(f"Size: {size_mb:.2f} MiB")
else:
    print("No model checkpoint files found in MyDrive.")

print("\nTotal model files found:", len(found))

=== TEAM MODEL SEARCH ===
No model checkpoint files found in MyDrive.

Total model files found: 0


### 50. Upload Teammate Model Checkpoint

Upload an EfficientNet checkpoint from the local computer to the project directory for architecture and metadata inspection.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

PROJECT = Path("/content/FlowerCompetition")
PROJECT.mkdir(parents=True, exist_ok=True)

print("Select the EfficientNet checkpoint (.pth or .pt).")

uploaded = files.upload()

for filename in uploaded:
    source = Path("/content") / filename
    destination = PROJECT / Path(filename).name

    if destination.exists():
        print("File already exists. Skipping:", destination)
        continue

    if source.resolve() != destination.resolve():
        shutil.move(str(source), str(destination))

    print("Uploaded:", destination)
    print(
        "Size:",
        round(destination.stat().st_size / (1024 ** 2), 2),
        "MiB"
    )

Select the EfficientNet checkpoint (.pth or .pt).


Saving efficientnet_best.pth to efficientnet_best.pth
Uploaded: /content/FlowerCompetition/efficientnet_best.pth
Size: 16.07 MiB


### 51. Inspect EfficientNet Checkpoint Structure

Inspect the teammate's EfficientNet checkpoint to identify its architecture, classifier dimensions, stored metadata, and class mapping.

Do not modify the checkpoint or assume that its class ordering matches the MobileNetV3 model.

In [ ]:
from pathlib import Path
import torch

PROJECT = Path("/content/FlowerCompetition")
CHECKPOINT_PATH = PROJECT / "efficientnet_best.pth"

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

print("=== EFFICIENTNET CHECKPOINT INSPECTION ===")
print("File:", CHECKPOINT_PATH.name)
print("Size:", round(CHECKPOINT_PATH.stat().st_size / 1024**2, 2), "MiB")
print("Checkpoint type:", type(checkpoint).__name__)

if not isinstance(checkpoint, dict):
    raise TypeError("Expected a dictionary checkpoint.")

print("\n=== TOP-LEVEL KEYS ===")
print(list(checkpoint.keys())[:30])

# Locate model weights
if "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

tensor_items = {
    k: v for k, v in state_dict.items()
    if isinstance(v, torch.Tensor)
}

print("\n=== MODEL WEIGHTS ===")
print("Number of tensors:", len(tensor_items))

print("\nFirst 12 tensor shapes:")
for key, value in list(tensor_items.items())[:12]:
    print(key, tuple(value.shape))

print("\n=== CLASSIFIER-RELATED WEIGHTS ===")
for key, value in tensor_items.items():
    if any(word in key.lower() for word in [
        "classifier", "_fc", "head", "fc.weight", "fc.bias"
    ]):
        print(key, tuple(value.shape))

print("\n=== STORED METADATA ===")
for key in [
    "architecture",
    "model_name",
    "epoch",
    "best_accuracy",
    "valid_accuracy",
    "val_accuracy",
    "image_size",
    "class_to_idx",
    "idx_to_class"
]:
    if key in checkpoint:
        value = checkpoint[key]

        if isinstance(value, dict):
            print(f"{key}: {len(value)} entries")
            print("Examples:", list(value.items())[:5])
        else:
            print(f"{key}: {value}")

print("\nInspection complete. Checkpoint was not modified.")

=== EFFICIENTNET CHECKPOINT INSPECTION ===
File: efficientnet_best.pth
Size: 16.07 MiB
Checkpoint type: OrderedDict

=== TOP-LEVEL KEYS ===
['features.0.0.weight', 'features.0.1.weight', 'features.0.1.bias', 'features.0.1.running_mean', 'features.0.1.running_var', 'features.0.1.num_batches_tracked', 'features.1.0.block.0.0.weight', 'features.1.0.block.0.1.weight', 'features.1.0.block.0.1.bias', 'features.1.0.block.0.1.running_mean', 'features.1.0.block.0.1.running_var', 'features.1.0.block.0.1.num_batches_tracked', 'features.1.0.block.1.fc1.weight', 'features.1.0.block.1.fc1.bias', 'features.1.0.block.1.fc2.weight', 'features.1.0.block.1.fc2.bias', 'features.1.0.block.2.0.weight', 'features.1.0.block.2.1.weight', 'features.1.0.block.2.1.bias', 'features.1.0.block.2.1.running_mean', 'features.1.0.block.2.1.running_var', 'features.1.0.block.2.1.num_batches_tracked', 'features.2.0.block.0.0.weight', 'features.2.0.block.0.1.weight', 'features.2.0.block.0.1.bias', 'features.2.0.block.0.1.ru

### 52. Verify EfficientNet-B0 Architecture

Reconstruct the torchvision EfficientNet-B0 architecture with 102 output classes and verify that the teammate's checkpoint loads with strict weight matching.

Inspect model parameter count and classifier dimensions without assuming a class mapping or preprocessing configuration.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
CHECKPOINT_PATH = PROJECT / "efficientnet_best.pth"

state_dict = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

# Reconstruct EfficientNet-B0
efficientnet_model = models.efficientnet_b0(weights=None)

efficientnet_model.classifier[1] = nn.Linear(
    efficientnet_model.classifier[1].in_features,
    102
)

# Require exact compatibility
efficientnet_model.load_state_dict(
    state_dict,
    strict=True
)

efficientnet_model.eval()

total_params = sum(
    p.numel() for p in efficientnet_model.parameters()
)

print("=== EFFICIENTNET-B0 VERIFICATION ===")
print("Architecture: torchvision EfficientNet-B0")
print("Checkpoint:", CHECKPOINT_PATH.name)
print("Strict weight loading: PASS")
print("Total parameters:", f"{total_params:,}")

print("\n=== CLASSIFIER ===")
print(efficientnet_model.classifier)

print("\n=== OUTPUT SHAPE TEST ===")

with torch.inference_mode():
    dummy_input = torch.zeros(1, 3, 224, 224)
    dummy_output = efficientnet_model(dummy_input)

print("Input shape:", tuple(dummy_input.shape))
print("Output shape:", tuple(dummy_output.shape))

assert dummy_output.shape == (1, 102)

print("\nPASS: Architecture and output dimensions verified.")
print("Class mapping and preprocessing are NOT yet verified.")

=== EFFICIENTNET-B0 VERIFICATION ===
Architecture: torchvision EfficientNet-B0
Checkpoint: efficientnet_best.pth
Strict weight loading: PASS
Total parameters: 4,138,210

=== CLASSIFIER ===
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=102, bias=True)
)

=== OUTPUT SHAPE TEST ===
Input shape: (1, 3, 224, 224)
Output shape: (1, 102)

PASS: Architecture and output dimensions verified.
Class mapping and preprocessing are NOT yet verified.


### 53. Diagnose EfficientNet-B0 Class Mapping

Compare alphabetical and numerical class mappings on the validation dataset using a provisional ImageNet preprocessing pipeline.

This diagnostic does not establish the original training configuration. Confirm the actual class mapping and preprocessing with the model's author before final benchmarking.

In [ ]:
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

efficientnet_model = efficientnet_model.to(DEVICE)
efficientnet_model.eval()

# Provisional preprocessing — must be confirmed with teammate
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = ImageFolder(
    PROJECT / "data" / "valid",
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Alphabetical: ImageFolder default
alphabetical_mapping = dataset.class_to_idx

# Numerical: class 1 -> index 0, class 102 -> index 101
numerical_mapping = {
    str(i): i - 1
    for i in range(1, 103)
}

alphabetical_correct = 0
numerical_correct = 0
total = 0

with torch.inference_mode():
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = efficientnet_model(images)
        predictions = outputs.argmax(dim=1)

        # Original ImageFolder labels: alphabetical
        alphabetical_correct += (
            predictions == labels
        ).sum().item()

        # Convert alphabetical labels to numerical labels
        numerical_labels = torch.tensor(
            [
                numerical_mapping[dataset.classes[label]]
                for label in labels.cpu().tolist()
            ],
            device=DEVICE
        )

        numerical_correct += (
            predictions == numerical_labels
        ).sum().item()

        total += labels.size(0)

print("=== EFFICIENTNET-B0 MAPPING DIAGNOSTIC ===")

print(
    f"Alphabetical mapping: "
    f"{alphabetical_correct}/{total} "
    f"({100 * alphabetical_correct / total:.2f}%)"
)

print(
    f"Numerical mapping: "
    f"{numerical_correct}/{total} "
    f"({100 * numerical_correct / total:.2f}%)"
)

print("\nPreprocessing: provisional Resize(256) + CenterCrop(224)")
print("Original training settings: not yet verified")

=== EFFICIENTNET-B0 MAPPING DIAGNOSTIC ===
Alphabetical mapping: 72/818 (8.80%)
Numerical mapping: 815/818 (99.63%)

Preprocessing: provisional Resize(256) + CenterCrop(224)
Original training settings: not yet verified


### 54. Analyze EfficientNet-B0 Validation Errors

Evaluate EfficientNet-B0 using the numerical class mapping and provisional ImageNet preprocessing.

Identify all misclassified validation images, their predicted classes, and prediction confidence. Save the results for comparison with MobileNetV3.

In [ ]:
import json
import torch
from pathlib import Path
from PIL import Image

PROJECT = Path("/content/FlowerCompetition")
RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

efficientnet_model.eval()

errors = []
correct = 0
total = 0

idx_to_class = {
    i - 1: str(i)
    for i in range(1, 103)
}

with torch.inference_mode():
    for image_path, _ in dataset.samples:

        # FIX: Convert string path to Path object
        image_path = Path(image_path)

        true_class = image_path.parent.name

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            tensor = transform(image).unsqueeze(0).to(DEVICE)

        logits = efficientnet_model(tensor)
        probabilities = torch.softmax(logits, dim=1)

        confidence, predicted_index = probabilities.max(dim=1)

        predicted_class = idx_to_class[
            int(predicted_index.item())
        ]

        total += 1

        if predicted_class == true_class:
            correct += 1
        else:
            errors.append({
                "image": str(image_path),
                "true_class": true_class,
                "predicted_class": predicted_class,
                "confidence": float(confidence.item())
            })

accuracy = 100 * correct / total

output_path = RESULTS / "efficientnet_b0_validation_errors.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "model": "efficientnet_b0",
            "mapping": "numerical",
            "preprocessing": "provisional_imagenet_resize256_centercrop224",
            "correct": correct,
            "total": total,
            "accuracy": accuracy,
            "errors": errors
        },
        f,
        indent=4
    )

print("=== EFFICIENTNET-B0 VALIDATION ANALYSIS ===")
print(f"Accuracy: {accuracy:.2f}% ({correct}/{total})")
print("Total errors:", len(errors))

print("\n=== MISCLASSIFIED IMAGES ===")

for error in errors:
    print(
        f"{Path(error['image']).name} | "
        f"True: {error['true_class']} | "
        f"Predicted: {error['predicted_class']} | "
        f"Confidence: {error['confidence']:.4f}"
    )

print("\nResults saved:", output_path)

assert total == 818
assert correct == 815, "Result differs from the mapping diagnostic."

=== EFFICIENTNET-B0 VALIDATION ANALYSIS ===
Accuracy: 99.63% (815/818)
Total errors: 3

=== MISCLASSIFIED IMAGES ===
image_05660.jpg | True: 4 | Predicted: 76 | Confidence: 0.1318
image_07303.jpg | True: 93 | Predicted: 94 | Confidence: 0.8675
image_07677.jpg | True: 96 | Predicted: 97 | Confidence: 0.7123

Results saved: /content/FlowerCompetition/results/efficientnet_b0_validation_errors.json


### 55. Benchmark EfficientNet-B0 CPU Inference

Measure EfficientNet-B0 single-image inference latency using PyTorch on CPU.

Use warm-up iterations and repeated measurements to report mean, median, and P95 latency. Exclude image loading and preprocessing for consistency with previous MobileNetV3 benchmarks.

In [ ]:
import time
import json
import numpy as np
import torch
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# Use CPU regardless of Colab GPU availability
cpu_model = efficientnet_model.to("cpu").eval()

# Control CPU threading for reproducibility
torch.set_num_threads(4)

# Preprocessed input, matching the previous image size
sample_input = torch.randn(1, 3, 224, 224)

WARMUP = 30
REPEATS = 200

print("=== EFFICIENTNET-B0 CPU BENCHMARK ===")
print("PyTorch version:", torch.__version__)
print("CPU threads:", torch.get_num_threads())
print("Input shape:", tuple(sample_input.shape))

with torch.inference_mode():

    # Warm-up
    for _ in range(WARMUP):
        _ = cpu_model(sample_input)

    latencies = []

    for _ in range(REPEATS):
        start = time.perf_counter()

        _ = cpu_model(sample_input)

        elapsed_ms = (
            time.perf_counter() - start
        ) * 1000

        latencies.append(elapsed_ms)

mean_ms = float(np.mean(latencies))
median_ms = float(np.median(latencies))
p95_ms = float(np.percentile(latencies, 95))

print("\n=== LATENCY RESULTS ===")
print(f"Mean:   {mean_ms:.3f} ms")
print(f"Median: {median_ms:.3f} ms")
print(f"P95:    {p95_ms:.3f} ms")

print("\n=== MODEL INFORMATION ===")
print("Architecture: EfficientNet-B0")
print("Backend: PyTorch CPU")
print("Checkpoint size: 16.07 MiB")
print("Validation accuracy: 99.63% (815/818)")

result = {
    "model": "efficientnet_b0",
    "backend": "pytorch_cpu",
    "cpu_threads": torch.get_num_threads(),
    "input_shape": [1, 3, 224, 224],
    "warmup_iterations": WARMUP,
    "measurement_iterations": REPEATS,
    "mean_latency_ms": mean_ms,
    "median_latency_ms": median_ms,
    "p95_latency_ms": p95_ms,
    "checkpoint_size_mib": 16.07,
    "validation_accuracy": 99.63,
    "validation_correct": 815,
    "validation_total": 818,
    "preprocessing_verified": False
}

output_path = RESULTS / "efficientnet_b0_cpu_benchmark.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4)

print("\nSaved:", output_path)

=== EFFICIENTNET-B0 CPU BENCHMARK ===
PyTorch version: 2.11.0+cu130
CPU threads: 4
Input shape: (1, 3, 224, 224)

=== LATENCY RESULTS ===
Mean:   71.175 ms
Median: 70.471 ms
P95:    80.265 ms

=== MODEL INFORMATION ===
Architecture: EfficientNet-B0
Backend: PyTorch CPU
Checkpoint size: 16.07 MiB
Validation accuracy: 99.63% (815/818)

Saved: /content/FlowerCompetition/results/efficientnet_b0_cpu_benchmark.json


### 56. Export EfficientNet-B0 to ONNX

Export the verified EfficientNet-B0 checkpoint to ONNX using a fixed batch size of one and 224×224 input resolution.

Keep the original PyTorch checkpoint unchanged. Inspect the exported ONNX graph, input/output shapes, and total file size before benchmarking.

In [ ]:
import sys
import subprocess
import importlib

print("=== REPAIR ONNX INSTALLATION ===")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "--force-reinstall",
    "--no-deps",
    "onnx==1.19.1"
])

# Reload ONNX from the installed package
for name in list(sys.modules):
    if name == "onnx" or name.startswith("onnx."):
        del sys.modules[name]

importlib.invalidate_caches()

import onnx

print("\n=== ONNX VERIFICATION ===")
print("ONNX version:", onnx.__version__)
print("ONNX location:", onnx.__file__)

required_functions = [
    "load",
    "load_model_from_string",
    "checker",
    "save"
]

for name in required_functions:
    available = hasattr(onnx, name)
    print(f"{name}: {'PASS' if available else 'MISSING'}")

assert all(
    hasattr(onnx, name)
    for name in required_functions
), "ONNX installation still has missing functions."

print("\nPASS: ONNX installation verified.")

=== REPAIR ONNX INSTALLATION ===

=== ONNX VERIFICATION ===
ONNX version: 1.19.1
ONNX location: /usr/local/lib/python3.13/dist-packages/onnx/__init__.py
load: PASS
load_model_from_string: PASS
checker: PASS
save: PASS

PASS: ONNX installation verified.


In [ ]:
from pathlib import Path
import torch
import onnx

PROJECT = Path("/content/FlowerCompetition")
ONNX_DIR = PROJECT / "onnx"
ONNX_DIR.mkdir(parents=True, exist_ok=True)

ONNX_PATH = ONNX_DIR / "efficientnet_b0_fp32.onnx"

# Use CPU for deterministic export
model = efficientnet_model.to("cpu").eval()

dummy_input = torch.randn(
    1, 3, 224, 224,
    dtype=torch.float32
)

print("=== EFFICIENTNET-B0 ONNX EXPORT ===")

with torch.inference_mode():
    torch.onnx.export(
        model,
        (dummy_input,),
        str(ONNX_PATH),
        input_names=["input"],
        output_names=["logits"],
        opset_version=17,
        dynamo=False,
        external_data=False
    )

print("Export completed.")

# Validate ONNX graph
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

print("ONNX graph validation: PASS")

print("\n=== INPUTS ===")
for inp in onnx_model.graph.input:
    shape = [
        d.dim_value
        for d in inp.type.tensor_type.shape.dim
    ]
    print(inp.name, shape)

print("\n=== OUTPUTS ===")
for out in onnx_model.graph.output:
    shape = [
        d.dim_value
        for d in out.type.tensor_type.shape.dim
    ]
    print(out.name, shape)

# Include any external weight files in total size
related_files = [
    p for p in ONNX_DIR.iterdir()
    if p.name == ONNX_PATH.name
    or p.name.startswith(ONNX_PATH.name + ".")
]

total_bytes = sum(
    p.stat().st_size
    for p in related_files
)

print("\n=== FILES ===")
for path in related_files:
    print(
        path.name,
        f"{path.stat().st_size / 1024**2:.2f} MiB"
    )

print(
    "\nTotal ONNX size:",
    f"{total_bytes / 1024**2:.2f} MiB"
)

print("\nOriginal PyTorch checkpoint preserved.")

=== EFFICIENTNET-B0 ONNX EXPORT ===


/tmp/ipykernel_1378/2326668761.py:22: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Export completed.
ONNX graph validation: PASS

=== INPUTS ===
input [1, 3, 224, 224]

=== OUTPUTS ===
logits [1, 102]

=== FILES ===
efficientnet_b0_fp32.onnx 15.78 MiB

Total ONNX size: 15.78 MiB

Original PyTorch checkpoint preserved.


### 57. Verify EfficientNet-B0 ONNX Prediction Consistency

Compare the exported ONNX model against the original PyTorch model on all 818 validation images.

Measure prediction agreement, maximum logit difference, and ONNX validation accuracy using the numerical class mapping and provisional preprocessing.

In [ ]:
import json
import numpy as np
import torch
import onnxruntime as ort
from pathlib import Path
from torch.utils.data import DataLoader

PROJECT = Path("/content/FlowerCompetition")
ONNX_PATH = PROJECT / "onnx" / "efficientnet_b0_fp32.onnx"
RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# Both models on CPU for a direct numerical comparison
pytorch_model = efficientnet_model.to("cpu").eval()

session = ort.InferenceSession(
    str(ONNX_PATH),
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

validation_loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

total = 0
matching_predictions = 0
onnx_correct = 0
pytorch_correct = 0
max_logit_difference = 0.0

numerical_mapping = {
    str(i): i - 1
    for i in range(1, 103)
}

with torch.inference_mode():
    for images, alphabetical_labels in validation_loader:

        images = images.cpu()

        pytorch_logits = pytorch_model(images).numpy()

        onnx_logits = session.run(
            [output_name],
            {input_name: images.numpy()}
        )[0]

        pytorch_pred = int(np.argmax(pytorch_logits, axis=1)[0])
        onnx_pred = int(np.argmax(onnx_logits, axis=1)[0])

        alphabetical_index = int(alphabetical_labels.item())
        class_name = dataset.classes[alphabetical_index]
        true_label = numerical_mapping[class_name]

        matching_predictions += int(pytorch_pred == onnx_pred)
        pytorch_correct += int(pytorch_pred == true_label)
        onnx_correct += int(onnx_pred == true_label)

        difference = float(
            np.max(np.abs(pytorch_logits - onnx_logits))
        )

        max_logit_difference = max(
            max_logit_difference,
            difference
        )

        total += 1

print("=== EFFICIENTNET-B0 ONNX VERIFICATION ===")

print(
    f"PyTorch accuracy: "
    f"{pytorch_correct}/{total} "
    f"({100 * pytorch_correct / total:.2f}%)"
)

print(
    f"ONNX accuracy: "
    f"{onnx_correct}/{total} "
    f"({100 * onnx_correct / total:.2f}%)"
)

print(
    f"Prediction agreement: "
    f"{matching_predictions}/{total} "
    f"({100 * matching_predictions / total:.2f}%)"
)

print(
    "Maximum absolute logit difference:",
    f"{max_logit_difference:.8f}"
)

result = {
    "model": "efficientnet_b0",
    "onnx_path": str(ONNX_PATH),
    "total": total,
    "pytorch_correct": pytorch_correct,
    "onnx_correct": onnx_correct,
    "matching_predictions": matching_predictions,
    "max_absolute_logit_difference": max_logit_difference,
    "class_mapping": "numerical",
    "preprocessing": "provisional_imagenet_resize256_centercrop224"
}

output_path = RESULTS / "efficientnet_b0_onnx_verification.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4)

print("\nSaved:", output_path)

assert total == 818
assert matching_predictions == total, (
    "ONNX and PyTorch predictions differ."
)
assert onnx_correct == pytorch_correct, (
    "ONNX accuracy differs from PyTorch."
)

print("\nPASS: ONNX predictions match PyTorch.")

=== EFFICIENTNET-B0 ONNX VERIFICATION ===
PyTorch accuracy: 815/818 (99.63%)
ONNX accuracy: 815/818 (99.63%)
Prediction agreement: 818/818 (100.00%)
Maximum absolute logit difference: 0.00014734

Saved: /content/FlowerCompetition/results/efficientnet_b0_onnx_verification.json

PASS: ONNX predictions match PyTorch.


### 58. Benchmark EfficientNet-B0 ONNX CPU Inference

Measure single-image ONNX Runtime inference latency for EfficientNet-B0 using the validation dataset.

Report mean, median, and P95 latency, and compare against the previous PyTorch CPU benchmark. Exclude image loading and preprocessing from timed measurements.


In [ ]:
import time
import json
import numpy as np
import onnxruntime as ort
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
RESULTS = PROJECT / "results"
ONNX_PATH = PROJECT / "onnx" / "efficientnet_b0_fp32.onnx"

# Configure ONNX Runtime CPU execution
options = ort.SessionOptions()
options.intra_op_num_threads = 4
options.inter_op_num_threads = 1
options.graph_optimization_level = (
    ort.GraphOptimizationLevel.ORT_ENABLE_ALL
)

session = ort.InferenceSession(
    str(ONNX_PATH),
    sess_options=options,
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# Preload validation tensors so image loading and
# preprocessing do not affect inference measurements
inputs = []

for image_tensor, _ in validation_loader:
    inputs.append(
        np.ascontiguousarray(
            image_tensor.numpy(),
            dtype=np.float32
        )
    )

print("=== EFFICIENTNET-B0 ONNX CPU BENCHMARK ===")
print("ONNX Runtime version:", ort.__version__)
print("CPU threads: 4")
print("Validation images:", len(inputs))

# Warm-up
for i in range(30):
    session.run(
        [output_name],
        {input_name: inputs[i % len(inputs)]}
    )

latencies = []

for input_array in inputs:
    start = time.perf_counter()

    session.run(
        [output_name],
        {input_name: input_array}
    )

    elapsed_ms = (
        time.perf_counter() - start
    ) * 1000

    latencies.append(elapsed_ms)

mean_ms = float(np.mean(latencies))
median_ms = float(np.median(latencies))
p95_ms = float(np.percentile(latencies, 95))

print("\n=== ONNX CPU LATENCY ===")
print(f"Mean:   {mean_ms:.3f} ms")
print(f"Median: {median_ms:.3f} ms")
print(f"P95:    {p95_ms:.3f} ms")

previous_pytorch_ms = 71.175
speedup = previous_pytorch_ms / mean_ms

print("\n=== PYTORCH VS ONNX ===")
print(f"PyTorch CPU: {previous_pytorch_ms:.3f} ms")
print(f"ONNX CPU:    {mean_ms:.3f} ms")
print(f"Speedup:     {speedup:.2f}x")

result = {
    "model": "efficientnet_b0",
    "backend": "onnxruntime_cpu",
    "cpu_threads": 4,
    "validation_images": len(inputs),
    "mean_latency_ms": mean_ms,
    "median_latency_ms": median_ms,
    "p95_latency_ms": p95_ms,
    "pytorch_reference_mean_ms": previous_pytorch_ms,
    "speedup_vs_pytorch": speedup,
    "onnx_size_mib": ONNX_PATH.stat().st_size / 1024**2,
    "validation_accuracy": 815 / 818,
    "preprocessing_verified": False
}

output_path = RESULTS / "efficientnet_b0_onnx_cpu_benchmark.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4)

print("\nSaved:", output_path)

assert len(inputs) == 818

=== EFFICIENTNET-B0 ONNX CPU BENCHMARK ===
ONNX Runtime version: 1.31.1
CPU threads: 4
Validation images: 818

=== ONNX CPU LATENCY ===
Mean:   41.842 ms
Median: 39.814 ms
P95:    57.531 ms

=== PYTORCH VS ONNX ===
PyTorch CPU: 71.175 ms
ONNX CPU:    41.842 ms
Speedup:     1.70x

Saved: /content/FlowerCompetition/results/efficientnet_b0_onnx_cpu_benchmark.json


### 59. Fair ONNX CPU Benchmark: EfficientNet-B0 vs MobileNetV3

Benchmark both ONNX models under identical CPU thread settings, input tensors, warm-up conditions, and timing procedures.

Measure inference-only latency using alternating model execution and save the comparison results.

In [ ]:
import time
import json
import numpy as np
import onnxruntime as ort
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

model_paths = {
    "efficientnet_b0": PROJECT / "onnx" / "efficientnet_b0_fp32.onnx",
    "mobilenetv3": PROJECT / "onnx" / "mobilenetv3_fp32.onnx"
}

options = ort.SessionOptions()
options.intra_op_num_threads = 4
options.inter_op_num_threads = 1
options.graph_optimization_level = (
    ort.GraphOptimizationLevel.ORT_ENABLE_ALL
)

sessions = {}

for name, path in model_paths.items():
    assert path.exists(), f"Missing ONNX model: {path}"

    sessions[name] = ort.InferenceSession(
        str(path),
        sess_options=options,
        providers=["CPUExecutionProvider"]
    )

# Identical input tensors for both models
rng = np.random.default_rng(42)

inputs = [
    rng.standard_normal((1, 3, 224, 224)).astype(np.float32)
    for _ in range(200)
]

WARMUP = 30
REPEATS = 200

print("=== FAIR ONNX CPU BENCHMARK ===")
print("ONNX Runtime:", ort.__version__)
print("CPU threads: 4")
print("Warm-up iterations:", WARMUP)
print("Measurement iterations:", REPEATS)

# Warm up both sessions
for name, session in sessions.items():
    input_name = session.get_inputs()[0].name

    for i in range(WARMUP):
        session.run(
            None,
            {input_name: inputs[i % len(inputs)]}
        )

latencies = {
    "efficientnet_b0": [],
    "mobilenetv3": []
}

# Alternate execution order to reduce order bias
for i in range(REPEATS):

    order = (
        ["efficientnet_b0", "mobilenetv3"]
        if i % 2 == 0
        else ["mobilenetv3", "efficientnet_b0"]
    )

    for name in order:
        session = sessions[name]
        input_name = session.get_inputs()[0].name

        start = time.perf_counter()

        session.run(
            None,
            {input_name: inputs[i]}
        )

        elapsed_ms = (
            time.perf_counter() - start
        ) * 1000

        latencies[name].append(elapsed_ms)

results = {}

print("\n=== RESULTS ===")

for name, values in latencies.items():

    mean_ms = float(np.mean(values))
    median_ms = float(np.median(values))
    p95_ms = float(np.percentile(values, 95))

    results[name] = {
        "mean_ms": mean_ms,
        "median_ms": median_ms,
        "p95_ms": p95_ms,
        "measurements": len(values)
    }

    print(f"\n{name}")
    print(f"Mean:   {mean_ms:.3f} ms")
    print(f"Median: {median_ms:.3f} ms")
    print(f"P95:    {p95_ms:.3f} ms")

speed_ratio = (
    results["efficientnet_b0"]["mean_ms"]
    / results["mobilenetv3"]["mean_ms"]
)

print("\n=== SPEED COMPARISON ===")
print(f"EfficientNet-B0 / MobileNetV3: {speed_ratio:.2f}x")

output_path = RESULTS / "efficientnet_vs_mobilenet_fair_cpu.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "backend": "onnxruntime_cpu",
            "cpu_threads": 4,
            "warmup": WARMUP,
            "repeats": REPEATS,
            "results": results,
            "efficientnet_to_mobilenet_latency_ratio": speed_ratio
        },
        f,
        indent=4
    )

print("\nSaved:", output_path)

=== FAIR ONNX CPU BENCHMARK ===
ONNX Runtime: 1.31.1
CPU threads: 4
Warm-up iterations: 30
Measurement iterations: 200

=== RESULTS ===

efficientnet_b0
Mean:   72.676 ms
Median: 72.503 ms
P95:    93.773 ms

mobilenetv3
Mean:   38.131 ms
Median: 38.268 ms
P95:    54.013 ms

=== SPEED COMPARISON ===
EfficientNet-B0 / MobileNetV3: 1.91x

Saved: /content/FlowerCompetition/results/efficientnet_vs_mobilenet_fair_cpu.json


### 60. Inspect Shared Model Registry

Inspect the existing shared model registry and loader implementation before integrating EfficientNet-B0.

Preserve all existing MobileNetV3 configurations and avoid modifying shared scripts until the integration requirements are confirmed.

In [ ]:
import json
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")

registry_path = PROJECT / "model_registry.json"
loader_path = PROJECT / "shared_model_loader.py"

print("=== CURRENT MODEL REGISTRY ===")

with open(registry_path, "r", encoding="utf-8") as f:
    registry = json.load(f)

print(json.dumps(registry, indent=2, ensure_ascii=False))

print("\n=== SHARED MODEL LOADER ===")

with open(loader_path, "r", encoding="utf-8") as f:
    loader_code = f.read()

print(loader_code)

print("\n=== EFFICIENTNET FILE CHECK ===")

for relative_path in [
    "efficientnet_best.pth",
    "onnx/efficientnet_b0_fp32.onnx",
    "results/efficientnet_b0_onnx_verification.json",
    "results/efficientnet_vs_mobilenet_fair_cpu.json"
]:
    path = PROJECT / relative_path
    print(
        "FOUND" if path.exists() else "MISSING",
        relative_path
    )

=== CURRENT MODEL REGISTRY ===
{
  "mobilenetv3_pytorch": {
    "architecture": "mobilenet_v3_large",
    "backend": "pytorch",
    "checkpoint": "mobilenetv3_finetuned_best.pth",
    "num_classes": 102,
    "image_size": 224,
    "preprocessing": "imagenet_resize256_centercrop224",
    "class_mapping": "checkpoint",
    "enabled": true
  },
  "mobilenetv3_onnx": {
    "architecture": "mobilenet_v3_large",
    "backend": "onnxruntime",
    "checkpoint": "onnx/mobilenetv3_fp32.onnx",
    "num_classes": 102,
    "image_size": 224,
    "preprocessing": "imagenet_resize256_centercrop224",
    "class_mapping": "class_to_idx.json",
    "enabled": true
  }
}

=== SHARED MODEL LOADER ===
from pathlib import Path
import json

import torch
import torchvision.models as models
import onnxruntime as ort


PROJECT_DIR = Path(__file__).resolve().parent
REGISTRY_PATH = PROJECT_DIR / "model_registry.json"


def load_registry():
    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        return js

### 61. Inspect Registry Structure for EfficientNet-B0 Integration

Inspect the existing model registry entries, class mapping, and loader architecture handling without modifying project files.

In [ ]:
import json
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")

with open(PROJECT / "model_registry.json", encoding="utf-8") as f:
    registry = json.load(f)

print("=== REGISTRY STRUCTURE ===")
print("Root type:", type(registry).__name__)

for key, value in registry.items():
    print(f"\nEntry: {key}")
    if isinstance(value, dict):
        print(json.dumps(value, indent=2, ensure_ascii=False))
    else:
        print(repr(value))

print("\n=== NUMERICAL CLASS MAPPING ===")

mapping_path = PROJECT / "class_to_idx.json"

with open(mapping_path, encoding="utf-8") as f:
    mapping = json.load(f)

expected = {str(i): i - 1 for i in range(1, 103)}

print("Mapping entries:", len(mapping))
print("First five:", list(mapping.items())[:5])
print("Matches numerical mapping:", mapping == expected)

print("\n=== LOADER ARCHITECTURE HANDLING ===")

loader_code = (PROJECT / "shared_model_loader.py").read_text(
    encoding="utf-8"
)

for line_number, line in enumerate(loader_code.splitlines(), 1):
    if any(term in line for term in [
        "architecture",
        "mobilenet_v3_large",
        "efficientnet",
        "self.backend",
        "self.config"
    ]):
        print(f"{line_number}: {line.strip()}")

print("\nInspection complete. No files modified.")

=== REGISTRY STRUCTURE ===
Root type: dict

Entry: mobilenetv3_pytorch
{
  "architecture": "mobilenet_v3_large",
  "backend": "pytorch",
  "checkpoint": "mobilenetv3_finetuned_best.pth",
  "num_classes": 102,
  "image_size": 224,
  "preprocessing": "imagenet_resize256_centercrop224",
  "class_mapping": "checkpoint",
  "enabled": true
}

Entry: mobilenetv3_onnx
{
  "architecture": "mobilenet_v3_large",
  "backend": "onnxruntime",
  "checkpoint": "onnx/mobilenetv3_fp32.onnx",
  "num_classes": 102,
  "image_size": 224,
  "preprocessing": "imagenet_resize256_centercrop224",
  "class_mapping": "class_to_idx.json",
  "enabled": true
}

=== NUMERICAL CLASS MAPPING ===
Mapping entries: 102
First five: [('1', 0), ('2', 1), ('3', 2), ('4', 3), ('5', 4)]
Matches numerical mapping: True

=== LOADER ARCHITECTURE HANDLING ===
38: self.config = registry[model_name]
40: if not self.config.get("enabled", False):
43: self.backend = self.config["backend"]
44: self.architecture = self.config["architecture

### 62. Register EfficientNet-B0 in the Shared Model System

Add the verified EfficientNet-B0 ONNX model to the shared model registry.

Preserve existing MobileNetV3 entries, back up the original registry, and reuse the verified numerical class mapping. Mark the preprocessing configuration as provisional until the teammate's training settings are confirmed.

In [ ]:
import json
import shutil
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")

registry_path = PROJECT / "model_registry.json"
backup_path = PROJECT / "model_registry_before_efficientnet.json"

onnx_path = PROJECT / "onnx" / "efficientnet_b0_fp32.onnx"
mapping_path = PROJECT / "class_to_idx.json"

assert onnx_path.exists(), "EfficientNet ONNX file missing."
assert mapping_path.exists(), "Class mapping file missing."

# Preserve the existing registry
if not backup_path.exists():
    shutil.copy2(registry_path, backup_path)

with open(registry_path, "r", encoding="utf-8") as f:
    registry = json.load(f)

original_mobilenet_entries = {
    key: value
    for key, value in registry.items()
    if key.startswith("mobilenetv3")
}

# Add EfficientNet-B0 ONNX
registry["efficientnet_b0_onnx"] = {
    "architecture": "efficientnet_b0",
    "backend": "onnxruntime",
    "checkpoint": "onnx/efficientnet_b0_fp32.onnx",
    "num_classes": 102,
    "image_size": 224,
    "preprocessing": "imagenet_resize256_centercrop224",
    "class_mapping": "class_to_idx.json",
    "enabled": True
}

# Ensure MobileNetV3 configurations are unchanged
for key, original_value in original_mobilenet_entries.items():
    assert registry[key] == original_value

with open(registry_path, "w", encoding="utf-8") as f:
    json.dump(
        registry,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=== UPDATED MODEL REGISTRY ===")

for name, config in registry.items():
    print(
        f"{name}: "
        f"{config['architecture']} | "
        f"{config['backend']} | "
        f"enabled={config['enabled']}"
    )

print("\nBackup:", backup_path)
print("Updated registry:", registry_path)
print("Existing MobileNetV3 entries preserved: PASS")
print("EfficientNet preprocessing: provisional, not yet verified")

=== UPDATED MODEL REGISTRY ===
mobilenetv3_pytorch: mobilenet_v3_large | pytorch | enabled=True
mobilenetv3_onnx: mobilenet_v3_large | onnxruntime | enabled=True
efficientnet_b0_onnx: efficientnet_b0 | onnxruntime | enabled=True

Backup: /content/FlowerCompetition/model_registry_before_efficientnet.json
Updated registry: /content/FlowerCompetition/model_registry.json
Existing MobileNetV3 entries preserved: PASS
EfficientNet preprocessing: provisional, not yet verified


### 63. Verify Shared Model Integration

Load EfficientNet-B0 and MobileNetV3 through the shared model loader.

Verify model outputs, numerical class mappings, and consistency with the standalone EfficientNet-B0 ONNX session.

Ensure that adding EfficientNet-B0 has not broken the existing MobileNetV3 integration.

In [ ]:
import sys
import json
import numpy as np
import torch
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from shared_model_loader import SharedModel

print("=== SHARED MODEL INTEGRATION TEST ===")

model_names = [
    "mobilenetv3_onnx",
    "efficientnet_b0_onnx"
]

expected_mapping = {
    str(i): i - 1
    for i in range(1, 103)
}

# Deterministic test input
torch.manual_seed(42)
test_input = torch.randn(1, 3, 224, 224)

loaded_models = {}

for model_name in model_names:

    print(f"\n--- {model_name} ---")

    shared_model = SharedModel(
        model_name,
        device="cpu"
    )

    logits = shared_model.predict_logits(test_input)

    assert logits.shape == (1, 102), (
        f"Unexpected output shape: {logits.shape}"
    )

    assert torch.isfinite(logits).all(), (
        "Model output contains NaN or Inf."
    )

    assert shared_model.class_to_idx == expected_mapping, (
        "Class mapping mismatch."
    )

    loaded_models[model_name] = shared_model

    print("Model loading: PASS")
    print("Output shape:", tuple(logits.shape))
    print("Numerical class mapping: PASS")
    print("Output values finite: PASS")

# Verify EfficientNet shared output against standalone ONNX
efficientnet_shared = loaded_models["efficientnet_b0_onnx"]

shared_logits = (
    efficientnet_shared.predict_logits(test_input)
    .numpy()
)

standalone_logits = session.run(
    [session.get_outputs()[0].name],
    {
        session.get_inputs()[0].name:
        test_input.numpy()
    }
)[0]

max_difference = float(
    np.max(
        np.abs(shared_logits - standalone_logits)
    )
)

print("\n=== EFFICIENTNET OUTPUT CONSISTENCY ===")
print(f"Maximum logit difference: {max_difference:.8f}")

assert np.allclose(
    shared_logits,
    standalone_logits,
    rtol=1e-4,
    atol=1e-4
), "Shared and standalone EfficientNet outputs differ."

print("Shared vs standalone ONNX: PASS")

print("\n=== FINAL STATUS ===")
print("MobileNetV3 integration: PASS")
print("EfficientNet-B0 integration: PASS")
print("Existing checkpoint files: unchanged")

=== SHARED MODEL INTEGRATION TEST ===

--- mobilenetv3_onnx ---
Model loading: PASS
Output shape: (1, 102)
Numerical class mapping: PASS
Output values finite: PASS

--- efficientnet_b0_onnx ---
Model loading: PASS
Output shape: (1, 102)
Numerical class mapping: PASS
Output values finite: PASS

=== EFFICIENTNET OUTPUT CONSISTENCY ===
Maximum logit difference: 0.00000000
Shared vs standalone ONNX: PASS

=== FINAL STATUS ===
MobileNetV3 integration: PASS
EfficientNet-B0 integration: PASS
Existing checkpoint files: unchanged


### 64. Validate EfficientNet-B0 Using the Shared Evaluation Pipeline

Run the registered EfficientNet-B0 ONNX model through the shared evaluation pipeline on all 818 validation images.

Verify that the shared pipeline reproduces the standalone validation accuracy and save the results without modifying existing MobileNetV3 outputs.

In [ ]:
import sys
import json
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from evaluate_shared import evaluate_model

print("=== EFFICIENTNET-B0 SHARED EVALUATION ===")

output_path = (
    PROJECT / "results" / "efficientnet_b0_shared.json"
)

result = evaluate_model(
    model_name="efficientnet_b0_onnx",
    data_dir=str(PROJECT / "data" / "valid"),
    output_path=str(output_path)
)

print("\n=== SAVED RESULTS ===")

with open(output_path, "r", encoding="utf-8") as f:
    saved_result = json.load(f)

print(json.dumps(saved_result, indent=2, ensure_ascii=False))

print("\n=== ACCURACY VERIFICATION ===")

# The shared evaluator's output format may differ.
# Display the result before asserting specific field names.
print("Returned result type:", type(result).__name__)
print("Result saved:", output_path)

print("\nExpected standalone accuracy: 815/818 = 99.63%")
print("Check that the shared evaluation matches this result.")

=== EFFICIENTNET-B0 SHARED EVALUATION ===

=== SAVED RESULTS ===
{
  "model": "efficientnet_b0_onnx",
  "architecture": "efficientnet_b0",
  "backend": "onnxruntime",
  "dataset": "/content/FlowerCompetition/data/valid",
  "images": 818,
  "correct": 815,
  "accuracy": 0.9963325183374083,
  "cpu_mean_latency_ms": 15.568065040347358,
  "cpu_median_latency_ms": 15.14983149991167,
  "cpu_p95_latency_ms": 18.571508149534562,
  "model_size_mib": 15.77647876739502,
  "cpu_threads": 1,
  "batch_size": 1
}

=== ACCURACY VERIFICATION ===
Returned result type: dict
Result saved: /content/FlowerCompetition/results/efficientnet_b0_shared.json

Expected standalone accuracy: 815/818 = 99.63%
Check that the shared evaluation matches this result.


### 65. Audit EfficientNet-B0 Training Artifacts and Dataset Overlap

Inspect available EfficientNet-B0 training artifacts and verify whether training and validation datasets contain duplicate images.

This step does not modify models, checkpoints, or datasets.

In [ ]:
from pathlib import Path
import hashlib

PROJECT = Path("/content/FlowerCompetition")

print("=== AVAILABLE TRAINING ARTIFACTS ===")

for path in sorted(PROJECT.rglob("*")):
    if not path.is_file():
        continue

    if path.suffix.lower() in {".ipynb", ".py", ".json"}:
        if any(term in path.name.lower() for term in [
            "efficientnet", "train", "config"
        ]):
            print(path.relative_to(PROJECT))

print("\n=== DATASET DUPLICATE AUDIT ===")

train_dir = PROJECT / "data" / "train"
valid_dir = PROJECT / "data" / "valid"

extensions = {".jpg", ".jpeg", ".png", ".webp"}

def image_hashes(folder):
    hashes = {}

    for path in folder.rglob("*"):
        if path.is_file() and path.suffix.lower() in extensions:
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
            hashes.setdefault(digest, []).append(
                str(path.relative_to(folder))
            )

    return hashes

train_hashes = image_hashes(train_dir)
valid_hashes = image_hashes(valid_dir)

overlap = set(train_hashes) & set(valid_hashes)

print("Unique training hashes:", len(train_hashes))
print("Unique validation hashes:", len(valid_hashes))
print("Exact duplicate hashes across splits:", len(overlap))

for digest in list(overlap)[:10]:
    print("TRAIN:", train_hashes[digest])
    print("VALID:", valid_hashes[digest])

if not overlap:
    print("Exact file overlap: PASS")
else:
    print("Exact file overlap: REVIEW REQUIRED")

print("\nNote: SHA-256 detects identical files only.")
print("Resized or recompressed duplicates require further checks.")
print("Original training preprocessing still requires teammate confirmation.")

=== AVAILABLE TRAINING ARTIFACTS ===
model_registry_before_efficientnet.json
results/efficientnet_b0_cpu_benchmark.json
results/efficientnet_b0_onnx_cpu_benchmark.json
results/efficientnet_b0_onnx_verification.json
results/efficientnet_b0_shared.json
results/efficientnet_b0_validation_errors.json
results/efficientnet_vs_mobilenet_fair_cpu.json

=== DATASET DUPLICATE AUDIT ===
Unique training hashes: 6549
Unique validation hashes: 818
Exact duplicate hashes across splits: 1
TRAIN: ['31/image_08077.jpg']
VALID: ['31/image_08067.jpg']
Exact file overlap: REVIEW REQUIRED

Note: SHA-256 detects identical files only.
Resized or recompressed duplicates require further checks.
Original training preprocessing still requires teammate confirmation.


### 66. Evaluate Models Excluding Exact Train–Validation Duplicates

Identify validation images whose SHA-256 hashes match training images.

Evaluate both registered ONNX models on the remaining validation images using the shared class mapping and preprocessing.

Preserve all original datasets and model files.

In [ ]:
import sys
import hashlib
import torch
from pathlib import Path
from PIL import Image
from torchvision import transforms

PROJECT = Path("/content/FlowerCompetition")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from shared_model_loader import SharedModel

train_dir = PROJECT / "data" / "train"
valid_dir = PROJECT / "data" / "valid"

extensions = {".jpg", ".jpeg", ".png", ".webp"}

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

train_hashes = {
    sha256_file(p)
    for p in train_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
}

valid_images = sorted(
    p for p in valid_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
)

clean_images = [
    p for p in valid_images
    if sha256_file(p) not in train_hashes
]

excluded = len(valid_images) - len(clean_images)

print("=== DATASET AUDIT ===")
print("Original validation images:", len(valid_images))
print("Excluded duplicate images:", excluded)
print("Remaining validation images:", len(clean_images))

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

models_to_evaluate = [
    "mobilenetv3_onnx",
    "efficientnet_b0_onnx"
]

print("\n=== CLEAN VALIDATION RESULTS ===")

for model_name in models_to_evaluate:

    model = SharedModel(model_name, device="cpu")

    correct = 0

    for image_path in clean_images:

        true_class = image_path.parent.name
        true_idx = model.class_to_idx[true_class]

        with Image.open(image_path) as image:
            tensor = preprocess(
                image.convert("RGB")
            ).unsqueeze(0)

        logits = model.predict_logits(tensor)
        prediction = int(logits.argmax(dim=1).item())

        correct += int(prediction == true_idx)

    total = len(clean_images)
    accuracy = 100 * correct / total

    print(f"\nModel: {model_name}")
    print(f"Correct: {correct}/{total}")
    print(f"Accuracy: {accuracy:.4f}%")

print("\nOriginal datasets and checkpoints: unchanged")

=== DATASET AUDIT ===
Original validation images: 818
Excluded duplicate images: 1
Remaining validation images: 817

=== CLEAN VALIDATION RESULTS ===

Model: mobilenetv3_onnx
Correct: 792/817
Accuracy: 96.9400%

Model: efficientnet_b0_onnx
Correct: 814/817
Accuracy: 99.6328%

Original datasets and checkpoints: unchanged


### 67. Save Clean Validation Audit Report

Record the original and duplicate-excluded validation accuracy for MobileNetV3-Large and EfficientNet-B0.

Document the identified exact duplicate, the limitations of SHA-256 duplicate detection, and the outstanding preprocessing verification.

Preserve all existing datasets, models, and evaluation results.

In [ ]:
import json
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
output_path = PROJECT / "results" / "clean_validation_audit.json"

report = {
    "audit_type": "exact_train_validation_duplicate_exclusion",
    "method": "SHA-256",
    "original_validation_images": 818,
    "excluded_images": 1,
    "clean_validation_images": 817,
    "exact_duplicate": {
        "train": "31/image_08077.jpg",
        "valid": "31/image_08067.jpg"
    },
    "models": {
        "mobilenetv3_onnx": {
            "original_correct": 793,
            "original_total": 818,
            "clean_correct": 792,
            "clean_total": 817,
            "original_accuracy": 793 / 818,
            "clean_accuracy": 792 / 817
        },
        "efficientnet_b0_onnx": {
            "original_correct": 815,
            "original_total": 818,
            "clean_correct": 814,
            "clean_total": 817,
            "original_accuracy": 815 / 818,
            "clean_accuracy": 814 / 817
        }
    },
    "limitations": [
        "SHA-256 identifies byte-identical files only.",
        "Near-duplicate images have not yet been audited.",
        "EfficientNet-B0 original training preprocessing is unverified.",
        "Training data provenance has not been independently verified."
    ],
    "datasets_modified": False,
    "checkpoints_modified": False
}

output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

with open(output_path, "r", encoding="utf-8") as f:
    verified = json.load(f)

assert verified["excluded_images"] == 1
assert verified["models"]["efficientnet_b0_onnx"]["clean_correct"] == 814
assert verified["models"]["mobilenetv3_onnx"]["clean_correct"] == 792

print("=== CLEAN VALIDATION AUDIT ===")
print("Report saved:", output_path)
print("JSON verification: PASS")
print("Original validation images: 818")
print("Clean validation images: 817")

for name, metrics in verified["models"].items():
    print(
        f"{name}: "
        f"{metrics['clean_correct']}/{metrics['clean_total']} "
        f"({metrics['clean_accuracy'] * 100:.4f}%)"
    )

print("Existing datasets and checkpoints: unchanged")

=== CLEAN VALIDATION AUDIT ===
Report saved: /content/FlowerCompetition/results/clean_validation_audit.json
JSON verification: PASS
Original validation images: 818
Clean validation images: 817
mobilenetv3_onnx: 792/817 (96.9400%)
efficientnet_b0_onnx: 814/817 (99.6328%)
Existing datasets and checkpoints: unchanged


### 68. Back Up the Updated Shared Model System

Create a new Google Drive backup containing the integrated MobileNetV3 and EfficientNet-B0 models, shared inference scripts, model registry, class mapping, and evaluation reports.

Preserve all existing backups and verify the ZIP archive integrity.

In [ ]:
import zipfile
import hashlib
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

DRIVE.mkdir(parents=True, exist_ok=True)

backup_path = DRIVE / "FlowerCompetition_IntegratedModels_Backup.zip"

# Do not overwrite an existing backup
if backup_path.exists():
    raise FileExistsError(
        f"Backup already exists: {backup_path}"
    )

files_to_include = [
    "model_registry.json",
    "model_registry_before_efficientnet.json",
    "class_to_idx.json",
    "shared_model_loader.py",
    "evaluate_shared.py",
    "predict_shared.py",
    "predict_onnx.py",
    "predict_selective_tta.py",
    "mobilenetv3_finetuned_best.pth",
    "efficientnet_best.pth",
    "onnx/mobilenetv3_fp32.onnx",
    "onnx/mobilenetv3_fp32.onnx.data",
    "onnx/efficientnet_b0_fp32.onnx",
]

# Include all evaluation reports
results_dir = PROJECT / "results"

if results_dir.exists():
    files_to_include.extend(
        str(p.relative_to(PROJECT))
        for p in results_dir.rglob("*")
        if p.is_file()
    )

files_to_include = sorted(set(files_to_include))

missing = [
    name for name in files_to_include
    if not (PROJECT / name).is_file()
]

if missing:
    print("Missing files:")
    for name in missing:
        print(" -", name)
    raise FileNotFoundError(
        "Backup stopped because required files are missing."
    )

with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for relative_path in files_to_include:
        archive.write(
            PROJECT / relative_path,
            arcname=relative_path
        )

with zipfile.ZipFile(backup_path, "r") as archive:
    corrupted_file = archive.testzip()
    archived_files = archive.namelist()

assert corrupted_file is None
assert set(archived_files) == set(files_to_include)

sha256 = hashlib.sha256()

with open(backup_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

print("=== INTEGRATED MODEL BACKUP ===")
print("Backup:", backup_path)
print("Files included:", len(archived_files))
print(
    "Archive size:",
    f"{backup_path.stat().st_size / (1024**2):.2f} MiB"
)
print("ZIP integrity: PASS")
print("SHA-256:", sha256.hexdigest())
print("Previous backups: unchanged")

=== INTEGRATED MODEL BACKUP ===
Backup: /content/drive/MyDrive/Flower_Competition/FlowerCompetition_IntegratedModels_Backup.zip
Files included: 38
Archive size: 68.39 MiB
ZIP integrity: PASS
SHA-256: b6ed211e8c4973c9c3e150384d5005226c1a950228849f1c8fde04a6db6159f8
Previous backups: unchanged


### 69. Audit Final Submission Readiness

Inspect the integrated model backup for required inference scripts, model weights, ONNX external data, configuration files, and Python dependencies.

Identify missing deployment artifacts before preparing the final competition submission package.

In [ ]:
import ast
import json
import zipfile
from pathlib import Path

PROJECT = Path("/content/FlowerCompetition")
BACKUP = Path(
    "/content/drive/MyDrive/Flower_Competition/"
    "FlowerCompetition_IntegratedModels_Backup.zip"
)

print("=== BACKUP CONTENT AUDIT ===")

with zipfile.ZipFile(BACKUP, "r") as z:
    archived = set(z.namelist())

required = [
    "model_registry.json",
    "class_to_idx.json",
    "shared_model_loader.py",
    "evaluate_shared.py",
    "predict_shared.py",
    "predict_onnx.py",
    "predict_selective_tta.py",
    "mobilenetv3_finetuned_best.pth",
    "efficientnet_best.pth",
    "onnx/mobilenetv3_fp32.onnx",
    "onnx/mobilenetv3_fp32.onnx.data",
    "onnx/efficientnet_b0_fp32.onnx",
    "results/clean_validation_audit.json",
    "results/efficientnet_b0_shared.json"
]

missing = []

for name in required:
    status = "PASS" if name in archived else "MISSING"
    print(f"{status:8} {name}")
    if status == "MISSING":
        missing.append(name)

print("\n=== PYTHON DEPENDENCIES ===")

python_files = [
    "shared_model_loader.py",
    "evaluate_shared.py",
    "predict_shared.py",
    "predict_onnx.py",
    "predict_selective_tta.py"
]

imports = set()

for name in python_files:
    path = PROJECT / name
    if not path.exists():
        continue

    tree = ast.parse(path.read_text(encoding="utf-8"))

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                imports.add(alias.name.split(".")[0])
        elif isinstance(node, ast.ImportFrom) and node.module:
            imports.add(node.module.split(".")[0])

print("Imported modules:")
print(", ".join(sorted(imports)))

print("\n=== DEPLOYMENT SUPPORT FILES ===")

for name in [
    "requirements.txt",
    "README.md",
    "run_inference.py",
    "submission.py"
]:
    path = PROJECT / name
    print(
        f"{'FOUND' if path.exists() else 'NOT FOUND':10} {name}"
    )

print("\n=== REGISTRY PATH CHECK ===")

with open(PROJECT / "model_registry.json", encoding="utf-8") as f:
    registry = json.load(f)

for model_name, config in registry.items():
    checkpoint = config["checkpoint"]
    mapping = config["class_mapping"]

    checkpoint_ok = checkpoint in archived
    mapping_ok = (
        mapping == "checkpoint"
        or mapping in archived
    )

    print(
        f"{model_name}: "
        f"checkpoint={'PASS' if checkpoint_ok else 'MISSING'}, "
        f"mapping={'PASS' if mapping_ok else 'MISSING'}"
    )

print("\n=== FINAL AUDIT SUMMARY ===")
print("Required files missing:", len(missing))
print("Backup file count:", len(archived))
print("Audit complete. No files modified.")

=== BACKUP CONTENT AUDIT ===
PASS     model_registry.json
PASS     class_to_idx.json
PASS     shared_model_loader.py
PASS     evaluate_shared.py
PASS     predict_shared.py
PASS     predict_onnx.py
PASS     predict_selective_tta.py
PASS     mobilenetv3_finetuned_best.pth
PASS     efficientnet_best.pth
PASS     onnx/mobilenetv3_fp32.onnx
PASS     onnx/mobilenetv3_fp32.onnx.data
PASS     onnx/efficientnet_b0_fp32.onnx
PASS     results/clean_validation_audit.json
PASS     results/efficientnet_b0_shared.json

=== PYTHON DEPENDENCIES ===
Imported modules:
PIL, argparse, json, numpy, onnxruntime, pathlib, shared_model_loader, sys, time, torch, torchvision

=== DEPLOYMENT SUPPORT FILES ===
FOUND      requirements.txt
FOUND      README.md
NOT FOUND  run_inference.py
NOT FOUND  submission.py

=== REGISTRY PATH CHECK ===
mobilenetv3_pytorch: checkpoint=PASS, mapping=PASS
mobilenetv3_onnx: checkpoint=PASS, mapping=PASS
efficientnet_b0_onnx: checkpoint=PASS, mapping=PASS

=== FINAL AUDIT SUMMARY ==

### 70. Verify Deployment Documentation and Dependencies

Inspect the project's requirements and README files.

Check whether they are included in the integrated backup and whether the documented dependencies and inference instructions match the current shared model system.

In [ ]:
from pathlib import Path
import zipfile

PROJECT = Path("/content/FlowerCompetition")

BACKUP = Path(
    "/content/drive/MyDrive/Flower_Competition/"
    "FlowerCompetition_IntegratedModels_Backup.zip"
)

with zipfile.ZipFile(BACKUP, "r") as archive:
    archived_files = set(archive.namelist())

print("=== DOCUMENTATION BACKUP CHECK ===")

for filename in ["requirements.txt", "README.md"]:

    path = PROJECT / filename

    print(f"\n--- {filename} ---")
    print("Exists in project:", path.is_file())
    print("Included in backup:", filename in archived_files)

    if path.is_file():
        content = path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        print("\nCONTENT:")
        print(content[:12000])

        if len(content) > 12000:
            print("\n[Remaining content omitted]")

print("\n=== AVAILABLE INFERENCE SCRIPTS ===")

for filename in [
    "predict_shared.py",
    "predict_onnx.py",
    "predict_selective_tta.py"
]:
    path = PROJECT / filename
    print(f"{filename}: {'FOUND' if path.is_file() else 'MISSING'}")

print("\nInspection complete. No files modified.")

=== DOCUMENTATION BACKUP CHECK ===

--- requirements.txt ---
Exists in project: True
Included in backup: False

CONTENT:
numpy
onnxruntime
Pillow
torch
torchvision


--- README.md ---
Exists in project: True
Included in backup: False

CONTENT:
# MobileNetV3-Large ONNX Deployment

Flower classification model with 102 classes.

Validation accuracy: 96.94% (793/818)
CPU inference: approximately 7.08 ms/image
Benchmark: Google Colab CPU, one thread, batch size 1.

## Installation

pip install -r requirements.txt

## Prediction

python predict_onnx.py --image /path/to/flower.jpg

The script prints the predicted class label.

## Required files

- onnx/mobilenetv3_fp32.onnx
- onnx/mobilenetv3_fp32.onnx.data
- class_to_idx.json
- predict_onnx.py

Keep the ONNX model and its .data file together.


=== AVAILABLE INFERENCE SCRIPTS ===
predict_shared.py: FOUND
predict_onnx.py: FOUND
predict_selective_tta.py: FOUND

Inspection complete. No files modified.


### 71. Update Integrated Model Documentation

Update the project README to document MobileNetV3-Large, EfficientNet-B0, shared inference, dependencies, validation accuracy, and evaluation limitations.

Preserve the original README before making changes.

In [ ]:
from pathlib import Path
import shutil

PROJECT = Path("/content/FlowerCompetition")

readme_path = PROJECT / "README.md"
backup_path = PROJECT / "README_mobilenet_original.md"

# Preserve original documentation
if readme_path.exists() and not backup_path.exists():
    shutil.copy2(readme_path, backup_path)

lines = [
    "# Oxford Flowers 102 - Shared Model System",
    "",
    "Flower classification using 102 classes.",
    "",
    "## Available Models",
    "",
    "| Model | Backend | Validation Accuracy | ONNX Size |",
    "|---|---|---|---|",
    "| MobileNetV3-Large | ONNX Runtime | 96.94% (793/818) | 16.80 MiB |",
    "| EfficientNet-B0 | ONNX Runtime | 99.63% (815/818) | 15.78 MiB |",
    "",
    "A PyTorch MobileNetV3-Large checkpoint is also available.",
    "",
    "## Installation",
    "",
    "Install dependencies using:",
    "",
    "    pip install -r requirements.txt",
    "",
    "## Registered Models",
    "",
    "- mobilenetv3_pytorch",
    "- mobilenetv3_onnx",
    "- efficientnet_b0_onnx",
    "",
    "Model configurations are stored in model_registry.json.",
    "",
    "## Shared Inference",
    "",
    "The shared loader is implemented in shared_model_loader.py.",
    "",
    "Example:",
    "",
    "    from shared_model_loader import SharedModel",
    "    model = SharedModel('efficientnet_b0_onnx', device='cpu')",
    "    logits = model.predict_logits(preprocessed_tensor)",
    "",
    "The input must be a preprocessed tensor with shape [N, 3, 224, 224].",
    "The variable preprocessed_tensor must be created before this example is run.",
    "",
    "## Image Preprocessing",
    "",
    "Current shared evaluation preprocessing:",
    "",
    "1. Convert to RGB.",
    "2. Resize the shorter side to 256 pixels.",
    "3. Center crop to 224 x 224 pixels.",
    "4. Convert to tensor.",
    "5. Apply ImageNet normalization.",
    "",
    "Mean: [0.485, 0.456, 0.406]",
    "Std: [0.229, 0.224, 0.225]",
    "",
    "EfficientNet-B0 original training preprocessing is not yet verified.",
    "",
    "## Shared Evaluation",
    "",
    "Use evaluate_shared.py to evaluate registered models.",
    "",
    "Example:",
    "",
    "    from evaluate_shared import evaluate_model",
    "    result = evaluate_model(",
    "        model_name='efficientnet_b0_onnx',",
    "        data_dir='data/valid',",
    "        output_path='results/efficientnet_b0_shared.json'",
    "    )",
    "    print(result)",
    "",
    "## Required Model Files",
    "",
    "MobileNetV3 ONNX:",
    "- onnx/mobilenetv3_fp32.onnx",
    "- onnx/mobilenetv3_fp32.onnx.data",
    "",
    "Keep both MobileNetV3 ONNX files together.",
    "",
    "EfficientNet-B0 ONNX:",
    "- onnx/efficientnet_b0_fp32.onnx",
    "",
    "Shared configuration:",
    "- model_registry.json",
    "- class_to_idx.json",
    "- shared_model_loader.py",
    "",
    "## Validation Audit",
    "",
    "Original validation dataset: 818 images.",
    "One byte-identical train-validation image pair was detected.",
    "",
    "After excluding the duplicate validation image:",
    "",
    "- MobileNetV3-Large: 792/817 (96.94%)",
    "- EfficientNet-B0: 814/817 (99.63%)",
    "",
    "See results/clean_validation_audit.json.",
    "",
    "SHA-256 cannot identify all visually similar images.",
    "Training data provenance remains to be independently verified.",
    "",
    "## CPU Benchmark",
    "",
    "The shared EfficientNet-B0 evaluation measured approximately",
    "15.57 ms/image on Google Colab CPU with one thread.",
    "",
    "CPU timings depend on hardware, runtime settings and system load.",
    "These are not official competition benchmark results.",
    "",
    "## Additional Scripts",
    "",
    "- predict_shared.py",
    "- predict_onnx.py",
    "- predict_selective_tta.py",
    "- evaluate_shared.py",
    "- shared_model_loader.py",
    "",
    "## Submission Status",
    "",
    "The final competition submission format requires confirmation",
    "against the official competition requirements.",
    ""
]

readme_path.write_text(
    "\n".join(lines),
    encoding="utf-8"
)

content = readme_path.read_text(encoding="utf-8")

required_terms = [
    "MobileNetV3-Large",
    "EfficientNet-B0",
    "model_registry.json",
    "shared_model_loader.py",
    "clean_validation_audit.json",
    "requirements.txt"
]

print("=== README UPDATE ===")

for term in required_terms:
    assert term in content
    print("PASS:", term)

print("\nREADME updated:", readme_path)
print("Original backup:", backup_path)
print("README size:", readme_path.stat().st_size, "bytes")
print("Documentation verification: PASS")
print("Models and checkpoints: unchanged")

=== README UPDATE ===
PASS: MobileNetV3-Large
PASS: EfficientNet-B0
PASS: model_registry.json
PASS: shared_model_loader.py
PASS: clean_validation_audit.json
PASS: requirements.txt

README updated: /content/FlowerCompetition/README.md
Original backup: /content/FlowerCompetition/README_mobilenet_original.md
README size: 2893 bytes
Documentation verification: PASS
Models and checkpoints: unchanged


### 72. Create and Verify Integrated Deployment Package

Create a versioned deployment ZIP containing both ONNX models, required model data, shared inference scripts, configuration files, documentation, and evaluation reports.

Verify the archive contents, integrity, and SHA-256 checksum without modifying previous backups.

In [ ]:
from pathlib import Path
import zipfile
import hashlib
import json

PROJECT = Path("/content/FlowerCompetition")
DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

DRIVE.mkdir(parents=True, exist_ok=True)

package_path = (
    DRIVE / "FlowerCompetition_IntegratedDeployment_v1.zip"
)

# Never overwrite an existing package.
if package_path.exists():
    raise FileExistsError(
        f"Package already exists: {package_path}"
    )

required_files = [
    "README.md",
    "requirements.txt",
    "model_registry.json",
    "class_to_idx.json",
    "shared_model_loader.py",
    "evaluate_shared.py",
    "predict_shared.py",
    "predict_onnx.py",
    "predict_selective_tta.py",
    "mobilenetv3_finetuned_best.pth",
    "efficientnet_best.pth",
    "onnx/mobilenetv3_fp32.onnx",
    "onnx/mobilenetv3_fp32.onnx.data",
    "onnx/efficientnet_b0_fp32.onnx",
    "results/clean_validation_audit.json",
    "results/efficientnet_b0_shared.json",
    "results/efficientnet_b0_onnx_verification.json",
    "results/efficientnet_vs_mobilenet_fair_cpu.json",
]

missing = [
    name for name in required_files
    if not (PROJECT / name).is_file()
]

if missing:
    raise FileNotFoundError(
        f"Missing required files: {missing}"
    )

# Validate registry references before packaging.
registry = json.loads(
    (PROJECT / "model_registry.json").read_text(
        encoding="utf-8"
    )
)

for name, config in registry.items():
    checkpoint = PROJECT / config["checkpoint"]

    assert checkpoint.is_file(), (
        f"Missing checkpoint for {name}: {checkpoint}"
    )

    mapping = config["class_mapping"]

    if mapping != "checkpoint":
        assert (PROJECT / mapping).is_file(), (
            f"Missing mapping for {name}: {mapping}"
        )

# Include all available evaluation results.
result_files = [
    str(path.relative_to(PROJECT))
    for path in (PROJECT / "results").rglob("*")
    if path.is_file()
]

files_to_include = sorted(
    set(required_files + result_files)
)

# Create ZIP.
with zipfile.ZipFile(
    package_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for relative_path in files_to_include:
        archive.write(
            PROJECT / relative_path,
            arcname=relative_path
        )

# Verify ZIP integrity and contents.
with zipfile.ZipFile(package_path, "r") as archive:
    corrupted = archive.testzip()
    archived_files = set(archive.namelist())

assert corrupted is None, (
    f"Corrupted archive entry: {corrupted}"
)

assert archived_files == set(files_to_include), (
    "Archive contents mismatch."
)

# Calculate SHA-256.
digest = hashlib.sha256()

with open(package_path, "rb") as file:
    for chunk in iter(
        lambda: file.read(1024 * 1024),
        b""
    ):
        digest.update(chunk)

print("=== INTEGRATED DEPLOYMENT PACKAGE ===")

print("Package:", package_path)
print("Files included:", len(archived_files))

print(
    "Package size:",
    f"{package_path.stat().st_size / (1024**2):.2f} MiB"
)

print("ZIP integrity: PASS")
print("Required files: PASS")
print("Registry paths: PASS")

print("README included:", "README.md" in archived_files)
print(
    "Requirements included:",
    "requirements.txt" in archived_files
)

print("SHA-256:", digest.hexdigest())
print("Previous backups: unchanged")

=== INTEGRATED DEPLOYMENT PACKAGE ===
Package: /content/drive/MyDrive/Flower_Competition/FlowerCompetition_IntegratedDeployment_v1.zip
Files included: 39
Package size: 68.39 MiB
ZIP integrity: PASS
Required files: PASS
Registry paths: PASS
README included: True
Requirements included: True
SHA-256: cc1fc71dc86caf42c018a2505b04220cf914ce6093736b2ffaf7783bd2c11783
Previous backups: unchanged


### 73. Verify Isolated Deployment

Extract the integrated deployment package into an isolated directory.

Verify both ONNX models using the packaged model loader, class mapping, and model weights without relying on the original project directory.

### 73.1. Install ONNX Runtime

Install the missing ONNX Runtime dependency in the new Colab session and verify that it can be imported successfully.

In [ ]:
import sys
import subprocess

print("=== INSTALL ONNX RUNTIME ===")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "onnxruntime"
])

import onnxruntime as ort

print("\n=== DEPENDENCY VERIFICATION ===")
print("ONNX Runtime version:", ort.__version__)
print("Available providers:", ort.get_available_providers())
print("ONNX Runtime installation: PASS")

=== INSTALL ONNX RUNTIME ===

=== DEPENDENCY VERIFICATION ===
ONNX Runtime version: 1.31.1
Available providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
ONNX Runtime installation: PASS


In [ ]:
import sys
import os
import zipfile
import subprocess
from pathlib import Path

PACKAGE = Path(
    "/content/drive/MyDrive/Flower_Competition/"
    "FlowerCompetition_IntegratedDeployment_v1.zip"
)

TEST_DIR = Path("/content/deployment_test_v1")

assert PACKAGE.is_file(), "Deployment package not found."

TEST_DIR.mkdir(parents=True, exist_ok=True)

# Extract the deployment package
with zipfile.ZipFile(PACKAGE, "r") as archive:
    assert archive.testzip() is None
    archive.extractall(TEST_DIR)

print("=== DEPLOYMENT RESTORE ===")
print("Package found: PASS")
print("ZIP extraction: PASS")

# Verify required files
required = [
    "model_registry.json",
    "class_to_idx.json",
    "shared_model_loader.py",
    "onnx/mobilenetv3_fp32.onnx",
    "onnx/mobilenetv3_fp32.onnx.data",
    "onnx/efficientnet_b0_fp32.onnx"
]

for filename in required:
    assert (TEST_DIR / filename).is_file(), filename
    print("PASS:", filename)

# Test both models in a separate Python process
test_script = """
import torch
from shared_model_loader import SharedModel

torch.manual_seed(42)

test_input = torch.randn(1, 3, 224, 224)

expected_mapping = {
    str(i): i - 1 for i in range(1, 103)
}

for name in ["mobilenetv3_onnx", "efficientnet_b0_onnx"]:

    model = SharedModel(name, device="cpu")

    assert model.class_to_idx == expected_mapping

    logits = model.predict_logits(test_input)

    assert logits.shape == (1, 102)
    assert torch.isfinite(logits).all()

    predicted_index = int(logits.argmax(dim=1).item())

    print(
        name,
        "PASS",
        "Predicted index:",
        predicted_index
    )

print("Isolated inference: PASS")
"""

result = subprocess.run(
    [sys.executable, "-c", test_script],
    cwd=str(TEST_DIR),
    capture_output=True,
    text=True,
    env={
        **os.environ,
        "PYTHONPATH": str(TEST_DIR)
    }
)

print("\n=== INFERENCE OUTPUT ===")
print(result.stdout)

if result.stderr:
    print("=== STDERR ===")
    print(result.stderr[-2000:])

assert result.returncode == 0, (
    f"Deployment test failed: {result.returncode}"
)

print("\n=== FINAL STATUS ===")
print("Deployment package: PASS")
print("Original backups: unchanged")

=== DEPLOYMENT RESTORE ===
Package found: PASS
ZIP extraction: PASS
PASS: model_registry.json
PASS: class_to_idx.json
PASS: shared_model_loader.py
PASS: onnx/mobilenetv3_fp32.onnx
PASS: onnx/mobilenetv3_fp32.onnx.data
PASS: onnx/efficientnet_b0_fp32.onnx

=== INFERENCE OUTPUT ===
mobilenetv3_onnx PASS Predicted index: 50
efficientnet_b0_onnx PASS Predicted index: 46
Isolated inference: PASS


=== FINAL STATUS ===
Deployment package: PASS
Original backups: unchanged


### 74. Verify Real-Image Inference from the Deployment Package

Extract one validation image from the full project backup and run both ONNX models using the isolated deployment directory.

Verify preprocessing, predicted class labels, and inference outputs without modifying the original backups.

In [ ]:
from pathlib import Path
import zipfile
import subprocess
import sys
import os

DRIVE = Path("/content/drive/MyDrive/Flower_Competition")
TEST_DIR = Path("/content/deployment_test_v1")
FULL_BACKUP = DRIVE / "FlowerCompetition_FullBackup.zip"

assert FULL_BACKUP.is_file(), "Full project backup missing."

print("=== FIND VALIDATION IMAGE ===")

with zipfile.ZipFile(FULL_BACKUP, "r") as archive:
    candidates = [
        name for name in archive.namelist()
        if "/valid/" in f"/{name}"
        and name.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print("Validation images found:", len(candidates))
    assert candidates, "No validation images found in backup."

    image_name = sorted(candidates)[0]
    image_bytes = archive.read(image_name)

image_path = Path("/content/flower_test_image.jpg")
image_path.write_bytes(image_bytes)

true_class = Path(image_name).parent.name

print("Selected image:", image_name)
print("True class:", true_class)

test_script = """
import sys
import torch
from PIL import Image
from torchvision import transforms
from shared_model_loader import SharedModel

image_path = sys.argv[1]
true_class = sys.argv[2]

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

with Image.open(image_path) as image:
    tensor = preprocess(image.convert("RGB")).unsqueeze(0)

for name in ["mobilenetv3_onnx", "efficientnet_b0_onnx"]:
    model = SharedModel(name, device="cpu")

    logits = model.predict_logits(tensor)
    probabilities = torch.softmax(logits, dim=1)

    confidence, index = probabilities.max(dim=1)
    reverse_mapping = {
        value: key
        for key, value in model.class_to_idx.items()
    }

    predicted_class = reverse_mapping[int(index.item())]

    print(
        f"{name}: "
        f"predicted={predicted_class}, "
        f"true={true_class}, "
        f"confidence={confidence.item():.4f}, "
        f"correct={predicted_class == true_class}"
    )

print("Real-image inference: PASS")
"""

result = subprocess.run(
    [
        sys.executable,
        "-c",
        test_script,
        str(image_path),
        true_class
    ],
    cwd=str(TEST_DIR),
    capture_output=True,
    text=True,
    env={
        **os.environ,
        "PYTHONPATH": str(TEST_DIR)
    }
)

print("\n=== REAL IMAGE TEST ===")
print(result.stdout)

if result.stderr:
    print(result.stderr[-3000:])

assert result.returncode == 0, "Real-image inference failed."

print("\nDeployment real-image test: PASS")
print("Original backups: unchanged")

=== FIND VALIDATION IMAGE ===
Validation images found: 818
Selected image: FlowerCompetition/data/valid/1/image_06739.jpg
True class: 1

=== REAL IMAGE TEST ===
mobilenetv3_onnx: predicted=1, true=1, confidence=0.9981, correct=True
efficientnet_b0_onnx: predicted=1, true=1, confidence=0.9628, correct=True
Real-image inference: PASS


Deployment real-image test: PASS
Original backups: unchanged


### 75. Inspect Competition Submission Requirements

Search the available project files for submission instructions, evaluation scripts, required filenames, and model packaging constraints.

Do not modify any project files or existing backups.

In [ ]:
from pathlib import Path
import zipfile

DRIVE = Path("/content/drive/MyDrive/Flower_Competition")
FULL_BACKUP = DRIVE / "FlowerCompetition_FullBackup.zip"
DEPLOYMENT = Path("/content/deployment_test_v1")

keywords = [
    "submission",
    "submit",
    "competition",
    "inference",
    "evaluation",
    "rules",
    "instruction",
    "readme"
]

print("=== FULL PROJECT BACKUP ===")

with zipfile.ZipFile(FULL_BACKUP, "r") as archive:
    candidates = [
        name for name in archive.namelist()
        if not name.endswith("/")
        and any(
            keyword in Path(name).name.lower()
            for keyword in keywords
        )
    ]

    for name in candidates[:80]:
        print(name)

    print("Matching files:", len(candidates))

print("\n=== CURRENT DEPLOYMENT FILES ===")

for path in sorted(DEPLOYMENT.iterdir()):
    print(path.name)

print("\n=== DOCUMENTATION FILES ===")

for path in DEPLOYMENT.rglob("*"):
    if path.is_file() and path.suffix.lower() in {
        ".md", ".txt", ".pdf"
    }:
        print(path.relative_to(DEPLOYMENT))

print("\nAudit complete. No files modified.")

=== FULL PROJECT BACKUP ===
Matching files: 0

=== CURRENT DEPLOYMENT FILES ===
README.md
__pycache__
class_to_idx.json
efficientnet_best.pth
evaluate_shared.py
mobilenetv3_finetuned_best.pth
model_registry.json
onnx
predict_onnx.py
predict_selective_tta.py
predict_shared.py
requirements.txt
results
shared_model_loader.py

=== DOCUMENTATION FILES ===
README.md
requirements.txt

Audit complete. No files modified.


### Step 76 — Restore and Inspect the Shared Inference System

Restore the latest verified deployment package from Google Drive.

Inspect the model registry and shared prediction interface before integrating additional team models or creating the final competition entry point.

This step does not modify model weights or existing backups.

In [ ]:
from pathlib import Path
import json
import zipfile
import inspect

DRIVE_DIR = Path("/content/drive/MyDrive/Flower_Competition")
PACKAGE = DRIVE_DIR / "FlowerCompetition_IntegratedDeployment_v1.zip"
PROJECT_DIR = Path("/content/FlowerCompetition")

print("=== STEP 76: SHARED INFERENCE AUDIT ===")

assert PACKAGE.is_file(), "Deployment ZIP not found in Google Drive."

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(PACKAGE, "r") as archive:
    assert archive.testzip() is None, "ZIP integrity check failed."
    archive.extractall(PROJECT_DIR)

print("Deployment restore: PASS")

registry_path = PROJECT_DIR / "model_registry.json"
registry = json.loads(registry_path.read_text(encoding="utf-8"))

print("\n=== MODEL REGISTRY ===")
print(json.dumps(registry, indent=2, ensure_ascii=False))

print("\n=== INFERENCE FILES ===")

for filename in [
    "shared_model_loader.py",
    "predict_shared.py",
    "predict_onnx.py",
    "evaluate_shared.py",
    "requirements.txt",
]:
    path = PROJECT_DIR / filename
    print(f"{filename}: {'PASS' if path.is_file() else 'MISSING'}")

print("\n=== PREDICTION INTERFACE ===")

predict_file = PROJECT_DIR / "predict_shared.py"

if predict_file.is_file():
    source = predict_file.read_text(encoding="utf-8")
    print(source[:7000])

print("\n=== FINAL STATUS ===")
print("Inspection complete.")
print("Original Drive backups: unchanged")

=== STEP 76: SHARED INFERENCE AUDIT ===
Deployment restore: PASS

=== MODEL REGISTRY ===
{
  "mobilenetv3_pytorch": {
    "architecture": "mobilenet_v3_large",
    "backend": "pytorch",
    "checkpoint": "mobilenetv3_finetuned_best.pth",
    "num_classes": 102,
    "image_size": 224,
    "preprocessing": "imagenet_resize256_centercrop224",
    "class_mapping": "checkpoint",
    "enabled": true
  },
  "mobilenetv3_onnx": {
    "architecture": "mobilenet_v3_large",
    "backend": "onnxruntime",
    "checkpoint": "onnx/mobilenetv3_fp32.onnx",
    "num_classes": 102,
    "image_size": 224,
    "preprocessing": "imagenet_resize256_centercrop224",
    "class_mapping": "class_to_idx.json",
    "enabled": true
  },
  "efficientnet_b0_onnx": {
    "architecture": "efficientnet_b0",
    "backend": "onnxruntime",
    "checkpoint": "onnx/efficientnet_b0_fp32.onnx",
    "num_classes": 102,
    "image_size": 224,
    "preprocessing": "imagenet_resize256_centercrop224",
    "class_mapping": "class_to

### Step 77 — Create a Unified Prediction Entry Point

Create a lightweight `predict.py` entry point that uses the existing shared inference system.

Use EfficientNet-B0 ONNX as the default model while allowing any registered model to be selected with the `--model` argument.

Preserve the existing prediction scripts and model weights. Test the new entry point with a real validation image.

### Step 77.2 — Install ONNX Runtime and Verify Both Models

Install the missing ONNX Runtime dependency in the current Colab environment.

Verify that the unified prediction entry point successfully performs inference using both EfficientNet-B0 and MobileNetV3 ONNX models.

Preserve all model weights, source files, and Google Drive backups.

In [ ]:
import sys
import subprocess
import json
from pathlib import Path

PROJECT_DIR = Path("/content/FlowerCompetition")
IMAGE_PATH = Path("/content/flower_test_image.jpg")

print("=== STEP 77.2: DEPENDENCY INSTALLATION ===")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "onnxruntime"
])

import onnxruntime as ort

print("ONNX Runtime version:", ort.__version__)
print("Installation: PASS")

assert (PROJECT_DIR / "predict.py").is_file()
assert IMAGE_PATH.is_file()

print("\n=== UNIFIED INFERENCE TEST ===")

for model_name in [
    "efficientnet_b0_onnx",
    "mobilenetv3_onnx"
]:
    result = subprocess.run(
        [
            sys.executable,
            "predict.py",
            "--image",
            str(IMAGE_PATH),
            "--model",
            model_name,
            "--top-k",
            "3"
        ],
        cwd=str(PROJECT_DIR),
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"Test failed: {model_name}")

    prediction = json.loads(result.stdout)

    assert prediction["model"] == model_name
    assert len(prediction["predictions"]) == 3

    top = prediction["predictions"][0]

    print("\nModel:", model_name)
    print("Predicted class:", top["class_name"])
    print("Confidence:", round(top["confidence"], 4))
    print("Correct:", top["class_name"] == "1")

    assert top["class_name"] == "1"

    print("Status: PASS")

print("\n=== FINAL STATUS ===")
print("EfficientNet-B0: PASS")
print("MobileNetV3: PASS")
print("Unified predict.py: PASS")
print("Original backups: unchanged")

=== STEP 77.2: DEPENDENCY INSTALLATION ===
ONNX Runtime version: 1.31.1
Installation: PASS

=== UNIFIED INFERENCE TEST ===

Model: efficientnet_b0_onnx
Predicted class: 1
Confidence: 0.9628
Correct: True
Status: PASS

Model: mobilenetv3_onnx
Predicted class: 1
Confidence: 0.9981
Correct: True
Status: PASS

=== FINAL STATUS ===
EfficientNet-B0: PASS
MobileNetV3: PASS
Unified predict.py: PASS
Original backups: unchanged


### Step 78 — Create Updated Deployment Backup v2

Create a versioned deployment archive containing the unified prediction entry point, shared inference system, ONNX models, model registry, and dependencies.

Verify archive integrity and required files before copying it to Google Drive.

Preserve all previous deployment packages and model checkpoints.

In [ ]:
from pathlib import Path
import zipfile
import hashlib
import shutil

PROJECT_DIR = Path("/content/FlowerCompetition")
DRIVE_DIR = Path("/content/drive/MyDrive/Flower_Competition")

PACKAGE_NAME = "FlowerCompetition_IntegratedDeployment_v2.zip"
LOCAL_PACKAGE = Path("/content") / PACKAGE_NAME
DRIVE_PACKAGE = DRIVE_DIR / PACKAGE_NAME

print("=== STEP 78: DEPLOYMENT BACKUP V2 ===")

assert PROJECT_DIR.is_dir(), "Project directory missing."
assert DRIVE_DIR.is_dir(), "Google Drive folder missing."
assert not DRIVE_PACKAGE.exists(), (
    "Version 2 backup already exists. "
    "Do not overwrite it."
)

required_files = [
    "predict.py",
    "predict_shared.py",
    "shared_model_loader.py",
    "evaluate_shared.py",
    "model_registry.json",
    "class_to_idx.json",
    "requirements.txt",
    "README.md",
    "onnx/mobilenetv3_fp32.onnx",
    "onnx/mobilenetv3_fp32.onnx.data",
    "onnx/efficientnet_b0_fp32.onnx",
]

for filename in required_files:
    assert (PROJECT_DIR / filename).is_file(), (
        f"Missing required file: {filename}"
    )

print("Required files: PASS")

# Package deployment files, not training data.
exclude_dirs = {
    "__pycache__",
    ".git",
    ".venv",
    "data",
    "datasets",
    "checkpoints",
}

files_to_package = sorted(
    [
        path
        for path in PROJECT_DIR.rglob("*")
        if path.is_file()
        and not any(
            part in exclude_dirs
            for part in path.relative_to(PROJECT_DIR).parts
        )
        and path.suffix.lower() not in {
            ".zip", ".ipynb"
        }
    ],
    key=lambda path: str(path.relative_to(PROJECT_DIR))
)

with zipfile.ZipFile(
    LOCAL_PACKAGE,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in files_to_package:
        archive.write(
            path,
            arcname=str(path.relative_to(PROJECT_DIR))
        )

with zipfile.ZipFile(LOCAL_PACKAGE, "r") as archive:
    assert archive.testzip() is None

    names = set(archive.namelist())

    for filename in required_files:
        assert filename in names, (
            f"Missing from ZIP: {filename}"
        )

print("ZIP integrity: PASS")
print("Files packaged:", len(files_to_package))
print(
    "Package size:",
    round(LOCAL_PACKAGE.stat().st_size / 1024**2, 2),
    "MiB"
)

shutil.copy2(LOCAL_PACKAGE, DRIVE_PACKAGE)

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(block)
    return digest.hexdigest()

assert sha256(LOCAL_PACKAGE) == sha256(DRIVE_PACKAGE)

print("\n=== GOOGLE DRIVE BACKUP ===")
print("Saved:", DRIVE_PACKAGE)
print("SHA256:", sha256(DRIVE_PACKAGE))
print("Backup verification: PASS")
print("Previous backups: unchanged")

=== STEP 78: DEPLOYMENT BACKUP V2 ===
Required files: PASS
ZIP integrity: PASS
Files packaged: 40
Package size: 68.39 MiB

=== GOOGLE DRIVE BACKUP ===
Saved: /content/drive/MyDrive/Flower_Competition/FlowerCompetition_IntegratedDeployment_v2.zip
SHA256: 142f071c381d3fab7034955d6492b6c6c9a30016caec10f3ec5d685b00cc9ad2
Backup verification: PASS
Previous backups: unchanged


### Step 79 — Inspect the CPU Benchmark Environment

Inspect the current CPU environment, ONNX Runtime version, available execution providers, and model input shapes.

Establish a reproducible baseline before comparing inference latency across team models.

Do not modify model weights or deployment backups.

In [ ]:
from pathlib import Path
import json
import os
import platform
import onnxruntime as ort

PROJECT_DIR = Path("/content/FlowerCompetition")

print("=== STEP 79: CPU BENCHMARK ENVIRONMENT ===")

print("Platform:", platform.platform())
print("Processor:", platform.processor() or "Not reported")
print("Logical CPU cores:", os.cpu_count())
print("ONNX Runtime:", ort.__version__)
print("Available providers:", ort.get_available_providers())

registry = json.loads(
    (PROJECT_DIR / "model_registry.json").read_text(
        encoding="utf-8"
    )
)

print("\n=== REGISTERED ONNX MODELS ===")

for name, config in registry.items():
    if config["backend"] != "onnxruntime":
        continue

    model_path = PROJECT_DIR / config["checkpoint"]
    assert model_path.is_file(), f"Missing model: {model_path}"

    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1

    session = ort.InferenceSession(
        str(model_path),
        sess_options=options,
        providers=["CPUExecutionProvider"]
    )

    inputs = session.get_inputs()
    outputs = session.get_outputs()

    print(f"\nModel: {name}")
    print("Architecture:", config["architecture"])
    print("Input:", [(x.name, x.shape) for x in inputs])
    print("Output:", [(x.name, x.shape) for x in outputs])
    print("Model size (MiB):", round(
        model_path.stat().st_size / 1024**2, 2
    ))
    print("Execution provider:", session.get_providers())
    print("CPU threads: 1")

print("\n=== FINAL STATUS ===")
print("Benchmark environment inspection: PASS")
print("Model weights: unchanged")
print("Google Drive backups: unchanged")

=== STEP 79: CPU BENCHMARK ENVIRONMENT ===
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Processor: x86_64
Logical CPU cores: 2
ONNX Runtime: 1.31.1
Available providers: ['AzureExecutionProvider', 'CPUExecutionProvider']

=== REGISTERED ONNX MODELS ===

Model: mobilenetv3_onnx
Architecture: mobilenet_v3_large
Input: [('input', [1, 3, 224, 224])]
Output: [('logits', [1, 102])]
Model size (MiB): 0.32
Execution provider: ['CPUExecutionProvider']
CPU threads: 1

Model: efficientnet_b0_onnx
Architecture: efficientnet_b0
Input: [('input', [1, 3, 224, 224])]
Output: [('logits', [1, 102])]
Model size (MiB): 15.78
Execution provider: ['CPUExecutionProvider']
CPU threads: 1

=== FINAL STATUS ===
Benchmark environment inspection: PASS
Model weights: unchanged
Google Drive backups: unchanged


### Step 80 — Fair CPU Inference Benchmark

Compare MobileNetV3-Large and EfficientNet-B0 ONNX models under identical CPU conditions.

Use single-threaded ONNX Runtime sessions, fixed synthetic input tensors, warm-up iterations, and alternating model execution.

Report mean, median, P95 latency, and total model size including external ONNX weights.

This benchmark measures model inference only, excluding image loading and preprocessing.

In [ ]:
from pathlib import Path
import json
import time
import statistics
import numpy as np
import onnxruntime as ort

PROJECT_DIR = Path("/content/FlowerCompetition")
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILES = {
    "mobilenetv3_onnx": "onnx/mobilenetv3_fp32.onnx",
    "efficientnet_b0_onnx": "onnx/efficientnet_b0_fp32.onnx",
}

WARMUP = 10
RUNS = 50

print("=== STEP 80: FAIR CPU BENCHMARK ===")

options = ort.SessionOptions()
options.intra_op_num_threads = 1
options.inter_op_num_threads = 1

sessions = {}

for name, relative_path in MODEL_FILES.items():
    model_path = PROJECT_DIR / relative_path

    sessions[name] = ort.InferenceSession(
        str(model_path),
        sess_options=options,
        providers=["CPUExecutionProvider"]
    )

    assert sessions[name].get_providers() == [
        "CPUExecutionProvider"
    ]

rng = np.random.default_rng(42)

inputs = [
    rng.standard_normal(
        (1, 3, 224, 224)
    ).astype(np.float32)
    for _ in range(RUNS)
]

latencies = {
    name: []
    for name in MODEL_FILES
}

print("\n=== WARM-UP ===")

for name, session in sessions.items():
    input_name = session.get_inputs()[0].name

    for i in range(WARMUP):
        session.run(
            None,
            {input_name: inputs[i]}
        )

    print(name, "warm-up: PASS")

print("\n=== ALTERNATING BENCHMARK ===")

names = list(MODEL_FILES)

for i in range(RUNS):

    # Reverse execution order every iteration.
    order = names if i % 2 == 0 else names[::-1]

    for name in order:
        session = sessions[name]
        input_name = session.get_inputs()[0].name

        start = time.perf_counter()

        output = session.run(
            None,
            {input_name: inputs[i]}
        )

        elapsed_ms = (
            time.perf_counter() - start
        ) * 1000

        assert output[0].shape == (1, 102)

        latencies[name].append(elapsed_ms)

def model_size_mib(model_path):
    total = model_path.stat().st_size

    external_data = Path(str(model_path) + ".data")

    if external_data.exists():
        total += external_data.stat().st_size

    return total / (1024 ** 2)

results = {
    "environment": {
        "onnxruntime": ort.__version__,
        "provider": "CPUExecutionProvider",
        "threads": 1,
        "warmup": WARMUP,
        "runs": RUNS,
        "input_shape": [1, 3, 224, 224],
        "preprocessing_included": False,
    },
    "models": {}
}

print("\n=== RESULTS ===")

for name, relative_path in MODEL_FILES.items():
    values = latencies[name]

    model_path = PROJECT_DIR / relative_path

    stats = {
        "mean_ms": round(statistics.mean(values), 3),
        "median_ms": round(statistics.median(values), 3),
        "p95_ms": round(
            float(np.percentile(values, 95)), 3
        ),
        "min_ms": round(min(values), 3),
        "max_ms": round(max(values), 3),
        "model_size_mib": round(
            model_size_mib(model_path), 3
        ),
    }

    results["models"][name] = stats

    print("\nModel:", name)

    for key, value in stats.items():
        print(f"  {key}: {value}")

output_path = (
    RESULTS_DIR / "fair_cpu_benchmark_step80.json"
)

output_path.write_text(
    json.dumps(results, indent=2),
    encoding="utf-8"
)

print("\n=== FINAL STATUS ===")
print("Benchmark: PASS")
print("Results saved:", output_path)
print("Model weights: unchanged")
print("Drive backups: unchanged")

=== STEP 80: FAIR CPU BENCHMARK ===

=== WARM-UP ===
mobilenetv3_onnx warm-up: PASS
efficientnet_b0_onnx warm-up: PASS

=== ALTERNATING BENCHMARK ===

=== RESULTS ===

Model: mobilenetv3_onnx
  mean_ms: 9.14
  median_ms: 8.723
  p95_ms: 12.999
  min_ms: 7.911
  max_ms: 13.607
  model_size_mib: 16.805

Model: efficientnet_b0_onnx
  mean_ms: 17.831
  median_ms: 16.856
  p95_ms: 24.293
  min_ms: 15.745
  max_ms: 26.138
  model_size_mib: 15.776

=== FINAL STATUS ===
Benchmark: PASS
Results saved: /content/FlowerCompetition/results/fair_cpu_benchmark_step80.json
Model weights: unchanged
Drive backups: unchanged


### Step 81 — End-to-End CPU Latency Benchmark

Measure image loading, preprocessing, and ONNX inference latency using a real validation image.

Compare MobileNetV3-Large and EfficientNet-B0 under identical single-threaded CPU conditions.

Reuse initialized ONNX Runtime sessions and alternate model execution order.

Report preprocessing, inference, and total latency separately. Exclude model initialization and session loading from the timed region.

In [ ]:
from pathlib import Path
from PIL import Image
from torchvision import transforms
import numpy as np
import onnxruntime as ort
import time
import json
import statistics
import zipfile

PROJECT_DIR = Path("/content/FlowerCompetition")
IMAGE_PATH = Path("/content/flower_test_image.jpg")
BACKUP = Path(
    "/content/drive/MyDrive/Flower_Competition/"
    "FlowerCompetition_FullBackup.zip"
)

print("=== STEP 81: END-TO-END CPU BENCHMARK ===")

# Restore the test image only if the runtime was reset.
if not IMAGE_PATH.is_file():
    with zipfile.ZipFile(BACKUP, "r") as archive:
        matches = [
            name for name in archive.namelist()
            if name.endswith("data/valid/1/image_06739.jpg")
        ]
        assert len(matches) == 1
        IMAGE_PATH.write_bytes(archive.read(matches[0]))

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

model_files = {
    "mobilenetv3_onnx": PROJECT_DIR / "onnx/mobilenetv3_fp32.onnx",
    "efficientnet_b0_onnx": PROJECT_DIR / "onnx/efficientnet_b0_fp32.onnx"
}

options = ort.SessionOptions()
options.intra_op_num_threads = 1
options.inter_op_num_threads = 1

sessions = {
    name: ort.InferenceSession(
        str(path),
        sess_options=options,
        providers=["CPUExecutionProvider"]
    )
    for name, path in model_files.items()
}

def run_once(name):
    session = sessions[name]

    start = time.perf_counter()

    with Image.open(IMAGE_PATH) as image:
        tensor = transform(
            image.convert("RGB")
        ).unsqueeze(0)

    array = tensor.numpy()

    preprocessing_end = time.perf_counter()

    outputs = session.run(
        None,
        {session.get_inputs()[0].name: array}
    )

    inference_end = time.perf_counter()

    assert outputs[0].shape == (1, 102)

    return {
        "preprocessing_ms": (
            preprocessing_end - start
        ) * 1000,
        "inference_ms": (
            inference_end - preprocessing_end
        ) * 1000,
        "total_ms": (
            inference_end - start
        ) * 1000
    }

WARMUP = 10
RUNS = 50
names = list(model_files)

print("\n=== WARM-UP ===")

for name in names:
    for _ in range(WARMUP):
        run_once(name)
    print(name, "PASS")

measurements = {
    name: {
        "preprocessing_ms": [],
        "inference_ms": [],
        "total_ms": []
    }
    for name in names
}

print("\n=== ALTERNATING BENCHMARK ===")

for i in range(RUNS):
    order = names if i % 2 == 0 else names[::-1]

    for name in order:
        result = run_once(name)

        for key, value in result.items():
            measurements[name][key].append(value)

results = {
    "environment": {
        "provider": "CPUExecutionProvider",
        "threads": 1,
        "warmup": WARMUP,
        "runs": RUNS,
        "image": IMAGE_PATH.name,
        "includes": [
            "image_loading",
            "preprocessing",
            "onnx_inference"
        ],
        "excludes": [
            "model_initialization",
            "onnx_session_loading"
        ]
    },
    "models": {}
}

print("\n=== RESULTS ===")

for name in names:
    results["models"][name] = {}

    print("\nModel:", name)

    for metric, values in measurements[name].items():
        summary = {
            "mean": round(statistics.mean(values), 3),
            "median": round(statistics.median(values), 3),
            "p95": round(
                float(np.percentile(values, 95)), 3
            )
        }

        results["models"][name][metric] = summary

        print(
            f"{metric}: "
            f"mean={summary['mean']} ms, "
            f"median={summary['median']} ms, "
            f"P95={summary['p95']} ms"
        )

output_path = (
    PROJECT_DIR / "results/end_to_end_benchmark_step81.json"
)

output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(
    json.dumps(results, indent=2),
    encoding="utf-8"
)

print("\n=== FINAL STATUS ===")
print("End-to-end benchmark: PASS")
print("Results saved:", output_path)
print("Model weights: unchanged")
print("Drive backups: unchanged")

=== STEP 81: END-TO-END CPU BENCHMARK ===

=== WARM-UP ===
mobilenetv3_onnx PASS
efficientnet_b0_onnx PASS

=== ALTERNATING BENCHMARK ===

=== RESULTS ===

Model: mobilenetv3_onnx
preprocessing_ms: mean=5.592 ms, median=5.576 ms, P95=5.96 ms
inference_ms: mean=8.705 ms, median=8.511 ms, P95=10.302 ms
total_ms: mean=14.297 ms, median=14.085 ms, P95=16.675 ms

Model: efficientnet_b0_onnx
preprocessing_ms: mean=5.674 ms, median=5.473 ms, P95=7.144 ms
inference_ms: mean=17.77 ms, median=17.356 ms, P95=19.43 ms
total_ms: mean=23.445 ms, median=22.888 ms, P95=27.368 ms

=== FINAL STATUS ===
End-to-end benchmark: PASS
Results saved: /content/FlowerCompetition/results/end_to_end_benchmark_step81.json
Model weights: unchanged
Drive backups: unchanged


### Step 82 — Preserve CPU Benchmark Results

Back up the reproducible CPU inference and end-to-end latency benchmark reports to Google Drive.

Verify file integrity using SHA256 checksums.

Preserve the existing deployment packages and model checkpoints without modification.

In [ ]:
from pathlib import Path
import hashlib
import shutil
import json

PROJECT_DIR = Path("/content/FlowerCompetition")
SOURCE_DIR = PROJECT_DIR / "results"

BACKUP_DIR = Path(
    "/content/drive/MyDrive/Flower_Competition/"
    "benchmark_results_v1"
)

FILES = [
    "fair_cpu_benchmark_step80.json",
    "end_to_end_benchmark_step81.json"
]

print("=== STEP 82: BENCHMARK BACKUP ===")

BACKUP_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()

for filename in FILES:
    source = SOURCE_DIR / filename
    destination = BACKUP_DIR / filename

    assert source.is_file(), f"Missing source: {filename}"

    # Avoid silently overwriting an existing report.
    if destination.exists():
        assert sha256(source) == sha256(destination), (
            f"Different backup already exists: {filename}"
        )
        print(filename, "already backed up: PASS")
    else:
        shutil.copy2(source, destination)

        assert sha256(source) == sha256(destination)

        print(filename, "backup: PASS")

    # Confirm that the saved file is valid JSON.
    json.loads(destination.read_text(encoding="utf-8"))

print("\n=== FINAL STATUS ===")
print("Benchmark backup: PASS")
print("Backup folder:", BACKUP_DIR)
print("Deployment ZIP files: unchanged")
print("Model weights: unchanged")

=== STEP 82: BENCHMARK BACKUP ===
fair_cpu_benchmark_step80.json backup: PASS
end_to_end_benchmark_step81.json backup: PASS

=== FINAL STATUS ===
Benchmark backup: PASS
Backup folder: /content/drive/MyDrive/Flower_Competition/benchmark_results_v1
Deployment ZIP files: unchanged
Model weights: unchanged


### Step 84 — Isolated Deployment Test

Extract the verified deployment v2 archive into a separate directory.

Run the unified prediction entry point using the default EfficientNet-B0 model and the alternative MobileNetV3 model.

Verify prediction correctness, class mapping, and independence from the original project directory.

Do not modify model weights or Google Drive backups.

### Step 84.1 — Fix Python Import Isolation

Rerun the deployment smoke test using Python's `-E` option instead of `-I`.

Verify that both ONNX models can be loaded and executed directly from the extracted deployment package without importing project modules from the original project directory.

Do not modify the deployment archive or model weights.

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import json

TEST_DIR = Path("/content/isolated_deployment_v2")
IMAGE_PATH = Path("/content/flower_test_image.jpg")

print("=== STEP 84.1: FIXED ISOLATED DEPLOYMENT TEST ===")

assert (TEST_DIR / "predict.py").is_file()
assert (TEST_DIR / "predict_shared.py").is_file()
assert IMAGE_PATH.is_file()

env = os.environ.copy()
env.pop("PYTHONPATH", None)

models = [
    "efficientnet_b0_onnx",
    "mobilenetv3_onnx"
]

for model_name in models:

    command = [
        sys.executable,
        "-E",
        str(TEST_DIR / "predict.py"),
        "--image",
        str(IMAGE_PATH),
        "--model",
        model_name,
        "--top-k",
        "3"
    ]

    result = subprocess.run(
        command,
        cwd=str(TEST_DIR),
        env=env,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(
            f"Deployment test failed: {model_name}"
        )

    prediction = json.loads(result.stdout)

    assert prediction["model"] == model_name
    assert len(prediction["predictions"]) == 3
    assert prediction["predictions"][0]["class_name"] == "1"

    print(f"\nModel: {model_name}")
    print(
        "Predicted class:",
        prediction["predictions"][0]["class_name"]
    )
    print(
        "Confidence:",
        round(
            prediction["predictions"][0]["confidence"],
            4
        )
    )
    print("Status: PASS")

print("\n=== FINAL STATUS ===")
print("Isolated deployment: PASS")
print("Unified predict.py: PASS")
print("Model weights: unchanged")
print("Google Drive backups: unchanged")

=== STEP 84.1: FIXED ISOLATED DEPLOYMENT TEST ===

Model: efficientnet_b0_onnx
Predicted class: 1
Confidence: 0.9628
Status: PASS

Model: mobilenetv3_onnx
Predicted class: 1
Confidence: 0.9981
Status: PASS

=== FINAL STATUS ===
Isolated deployment: PASS
Unified predict.py: PASS
Model weights: unchanged
Google Drive backups: unchanged


In [ ]:
from pathlib import Path
import torch

MODEL_PATH = Path(
    "/content/drive/MyDrive/Flower_Competition/"
    "teammate_models/efficientnetv2_best.pth"
)

assert MODEL_PATH.is_file(), "Model dosyası bulunamadı."

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

print("=== STEP 85: TEAMMATE MODEL CHECK ===")
print("Architecture:", checkpoint["architecture"])
print("Image size:", checkpoint["image_size"])
print("Validation accuracy:", checkpoint["valid_accuracy"])
print("Number of classes:", len(checkpoint["class_to_idx"]))
print("State dict parameters:", len(checkpoint["model_state_dict"]))
print("Model checkpoint: PASS")

=== STEP 85: TEAMMATE MODEL CHECK ===
Architecture: efficientnet_v2_s
Image size: 224
Validation accuracy: 98.77750611246944
Number of classes: 102
State dict parameters: 782
Model checkpoint: PASS


### Step 86 — Load and Validate Teammate EfficientNetV2-S

Load the teammate's EfficientNetV2-S checkpoint on CPU.

Verify architecture compatibility, strict weight loading, class mapping, and output dimensions.

Run a single forward pass in evaluation mode without modifying or retraining the model.

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import models

MODEL_PATH = Path(
    "/content/drive/MyDrive/Flower_Competition/"
    "teammate_models/efficientnetv2_best.pth"
)

print("=== STEP 86: EFFICIENTNETV2-S VALIDATION ===")

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

assert checkpoint["architecture"] == "efficientnet_v2_s"
assert checkpoint["image_size"] == 224

class_to_idx = checkpoint["class_to_idx"]

assert len(class_to_idx) == 102
assert {
    str(i): i - 1 for i in range(1, 103)
} == {
    str(k): int(v) for k, v in class_to_idx.items()
}, "Class mapping mismatch!"

print("Architecture: PASS")
print("Class mapping: PASS")

model = models.efficientnet_v2_s(weights=None)

in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features,
    102
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.cpu().eval()

print("Strict checkpoint loading: PASS")

dummy_input = torch.randn(
    1, 3, 224, 224,
    dtype=torch.float32
)

with torch.inference_mode():
    output = model(dummy_input)

assert output.shape == (1, 102)
assert torch.isfinite(output).all()

print("Output shape:", tuple(output.shape))
print("Forward pass: PASS")

print("\n=== FINAL STATUS ===")
print("EfficientNetV2-S PyTorch loading: PASS")
print("Model weights: unchanged")
print("Training performed: NO")

=== STEP 86: EFFICIENTNETV2-S VALIDATION ===
Architecture: PASS
Class mapping: PASS
Strict checkpoint loading: PASS
Output shape: (1, 102)
Forward pass: PASS

=== FINAL STATUS ===
EfficientNetV2-S PyTorch loading: PASS
Model weights: unchanged
Training performed: NO


### Step 87 — Real Image Prediction Test

Evaluate the teammate's EfficientNetV2-S model on a real Oxford Flowers 102 validation image.

Apply the checkpoint-compatible preprocessing pipeline, verify the predicted class, and report the confidence score.

This is a smoke test, not a full validation accuracy evaluation.

In [ ]:
from pathlib import Path
import zipfile
import torch
from torchvision import transforms
from PIL import Image

print("=== STEP 87: REAL IMAGE PREDICTION ===")

IMAGE_PATH = Path("/content/flower_test_image.jpg")

BACKUP = Path(
    "/content/drive/MyDrive/Flower_Competition/"
    "FlowerCompetition_FullBackup.zip"
)

# Restore the known validation image if necessary.
if not IMAGE_PATH.is_file():
    assert BACKUP.is_file(), "Full backup not found."

    with zipfile.ZipFile(BACKUP, "r") as archive:
        matches = [
            name for name in archive.namelist()
            if name.endswith(
                "data/valid/1/image_06739.jpg"
            )
        ]

        assert len(matches) == 1
        IMAGE_PATH.write_bytes(
            archive.read(matches[0])
        )

# Use the preprocessing established for this checkpoint.
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

with Image.open(IMAGE_PATH) as image:
    image_tensor = preprocess(
        image.convert("RGB")
    ).unsqueeze(0)

model.eval()

with torch.inference_mode():
    logits = model(image_tensor)
    probabilities = torch.softmax(logits, dim=1)

confidence, predicted_index = probabilities.max(dim=1)

predicted_index = int(predicted_index.item())
confidence = float(confidence.item())

idx_to_class = {
    int(index): str(class_name)
    for class_name, index in class_to_idx.items()
}

predicted_class = idx_to_class[predicted_index]
expected_class = "1"

print("Image:", IMAGE_PATH.name)
print("Expected class:", expected_class)
print("Predicted class:", predicted_class)
print("Confidence:", round(confidence, 4))

print("\n=== FINAL STATUS ===")

if predicted_class == expected_class:
    print("Real image prediction: PASS")
else:
    print("Real image prediction: INCORRECT")
    print("Check preprocessing before integration.")

print("Model weights: unchanged")
print("Training performed: NO")

=== STEP 87: REAL IMAGE PREDICTION ===
Image: flower_test_image.jpg
Expected class: 1
Predicted class: 1
Confidence: 0.9209

=== FINAL STATUS ===
Real image prediction: PASS
Model weights: unchanged
Training performed: NO


### Step 88 — Export EfficientNetV2-S to ONNX

Export the validated EfficientNetV2-S PyTorch model to ONNX using a fixed input shape of [1, 3, 224, 224].

Validate the exported ONNX graph and compare its output against the original PyTorch model on the same input.

Do not retrain or modify the checkpoint.

### Step 88.1 — Install ONNX Dependencies

Install the ONNX and ONNX Runtime packages required for model export and validation.

Verify that both libraries can be imported successfully before continuing with the EfficientNetV2-S conversion.

In [ ]:
import sys
import subprocess

print("=== STEP 88.1: INSTALL ONNX DEPENDENCIES ===")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "onnx", "onnxruntime"
])

import onnx
import onnxruntime as ort

print("ONNX version:", onnx.__version__)
print("ONNX Runtime version:", ort.__version__)
print("Available providers:", ort.get_available_providers())

print("\n=== FINAL STATUS ===")
print("ONNX dependencies: PASS")

=== STEP 88.1: INSTALL ONNX DEPENDENCIES ===
ONNX version: 1.23.2
ONNX Runtime version: 1.31.1
Available providers: ['AzureExecutionProvider', 'CPUExecutionProvider']

=== FINAL STATUS ===
ONNX dependencies: PASS


In [ ]:
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort

PROJECT_DIR = Path("/content/FlowerCompetition")
ONNX_DIR = PROJECT_DIR / "onnx"
ONNX_DIR.mkdir(parents=True, exist_ok=True)

ONNX_PATH = ONNX_DIR / "efficientnetv2_s_fp32.onnx"

print("=== STEP 88: EFFICIENTNETV2-S ONNX EXPORT ===")

model = model.cpu().eval()

dummy_input = torch.randn(
    1, 3, 224, 224,
    dtype=torch.float32
)

# Export model.
torch.onnx.export(
    model,
    dummy_input,
    str(ONNX_PATH),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamo=False
)

print("ONNX export: PASS")

# Validate ONNX graph.
onnx_model = onnx.load(
    str(ONNX_PATH),
    load_external_data=True
)

onnx.checker.check_model(onnx_model)

print("ONNX graph validation: PASS")

# Create CPU-only ONNX Runtime session.
options = ort.SessionOptions()
options.intra_op_num_threads = 1
options.inter_op_num_threads = 1

session = ort.InferenceSession(
    str(ONNX_PATH),
    sess_options=options,
    providers=["CPUExecutionProvider"]
)

# Compare PyTorch and ONNX outputs.
with torch.inference_mode():
    torch_output = model(
        dummy_input
    ).numpy()

onnx_output = session.run(
    None,
    {"input": dummy_input.numpy()}
)[0]

assert torch_output.shape == (1, 102)
assert onnx_output.shape == (1, 102)

max_diff = float(
    np.max(np.abs(torch_output - onnx_output))
)

same_prediction = (
    torch_output.argmax(axis=1)
    == onnx_output.argmax(axis=1)
).all()

print("\n=== OUTPUT COMPARISON ===")
print("PyTorch shape:", torch_output.shape)
print("ONNX shape:", onnx_output.shape)
print("Maximum absolute difference:", max_diff)
print("Prediction parity:", bool(same_prediction))

assert np.allclose(
    torch_output,
    onnx_output,
    rtol=1e-3,
    atol=1e-3
), "ONNX output differs significantly from PyTorch!"

assert same_prediction

# Include external data files in total model size.
related_files = [
    p for p in ONNX_DIR.iterdir()
    if p.name == ONNX_PATH.name
    or p.name.startswith(ONNX_PATH.name + ".")
]

total_size = sum(
    p.stat().st_size for p in related_files
)

print("\n=== EXPORTED FILES ===")

for path in sorted(related_files):
    print(
        path.name,
        round(path.stat().st_size / 1024**2, 3),
        "MiB"
    )

print("Total size:", round(total_size / 1024**2, 3), "MiB")

print("\n=== FINAL STATUS ===")
print("EfficientNetV2-S ONNX export: PASS")
print("PyTorch/ONNX parity: PASS")
print("Original checkpoint: unchanged")
print("Training performed: NO")

=== STEP 88: EFFICIENTNETV2-S ONNX EXPORT ===


/tmp/ipykernel_931/1357896945.py:23: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX export: PASS
ONNX graph validation: PASS

=== OUTPUT COMPARISON ===
PyTorch shape: (1, 102)
ONNX shape: (1, 102)
Maximum absolute difference: 2.86102294921875e-06
Prediction parity: True

=== EXPORTED FILES ===
efficientnetv2_s_fp32.onnx 77.324 MiB
Total size: 77.324 MiB

=== FINAL STATUS ===
EfficientNetV2-S ONNX export: PASS
PyTorch/ONNX parity: PASS
Original checkpoint: unchanged
Training performed: NO


### Step 89 — EfficientNetV2-S CPU Inference Benchmark

Benchmark the exported EfficientNetV2-S ONNX model using the same single-threaded CPU configuration as the previous models.

Measure mean, median, and P95 inference latency over 50 runs after 10 warm-up iterations.

Save the results locally without modifying model weights or existing deployment backups.

In [ ]:
from pathlib import Path
import time
import json
import statistics
import numpy as np
import onnxruntime as ort

PROJECT_DIR = Path("/content/FlowerCompetition")
ONNX_PATH = PROJECT_DIR / "onnx/efficientnetv2_s_fp32.onnx"

print("=== STEP 89: EFFICIENTNETV2-S CPU BENCHMARK ===")

assert ONNX_PATH.is_file(), "ONNX model not found."

options = ort.SessionOptions()
options.intra_op_num_threads = 1
options.inter_op_num_threads = 1

session = ort.InferenceSession(
    str(ONNX_PATH),
    sess_options=options,
    providers=["CPUExecutionProvider"]
)

assert session.get_providers() == ["CPUExecutionProvider"]

input_name = session.get_inputs()[0].name

rng = np.random.default_rng(42)

WARMUP = 10
RUNS = 50

inputs = [
    rng.standard_normal((1, 3, 224, 224)).astype(np.float32)
    for _ in range(RUNS)
]

print("\n=== WARM-UP ===")

for i in range(WARMUP):
    session.run(None, {input_name: inputs[i]})

print("Warm-up: PASS")

print("\n=== BENCHMARK ===")

latencies = []

for i in range(RUNS):
    start = time.perf_counter()

    output = session.run(
        None,
        {input_name: inputs[i]}
    )

    elapsed = (time.perf_counter() - start) * 1000

    assert output[0].shape == (1, 102)
    latencies.append(elapsed)

results = {
    "model": "efficientnetv2_s_onnx",
    "provider": "CPUExecutionProvider",
    "threads": 1,
    "runs": RUNS,
    "warmup": WARMUP,
    "mean_ms": round(statistics.mean(latencies), 3),
    "median_ms": round(statistics.median(latencies), 3),
    "p95_ms": round(float(np.percentile(latencies, 95)), 3),
    "min_ms": round(min(latencies), 3),
    "max_ms": round(max(latencies), 3),
    "model_size_mib": round(
        ONNX_PATH.stat().st_size / 1024**2, 3
    )
}

print("\n=== RESULTS ===")

for key, value in results.items():
    print(f"{key}: {value}")

output_path = (
    PROJECT_DIR / "results/efficientnetv2_cpu_benchmark_step89.json"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

output_path.write_text(
    json.dumps(results, indent=2),
    encoding="utf-8"
)

print("\n=== FINAL STATUS ===")
print("CPU benchmark: PASS")
print("Results saved:", output_path)
print("Model weights: unchanged")
print("Existing deployment backups: unchanged")

=== STEP 89: EFFICIENTNETV2-S CPU BENCHMARK ===

=== WARM-UP ===
Warm-up: PASS

=== BENCHMARK ===

=== RESULTS ===
model: efficientnetv2_s_onnx
provider: CPUExecutionProvider
threads: 1
runs: 50
warmup: 10
mean_ms: 108.595
median_ms: 115.526
p95_ms: 141.212
min_ms: 73.484
max_ms: 275.516
model_size_mib: 77.324

=== FINAL STATUS ===
CPU benchmark: PASS
Results saved: /content/FlowerCompetition/results/efficientnetv2_cpu_benchmark_step89.json
Model weights: unchanged
Existing deployment backups: unchanged


### Step 90 — Register EfficientNetV2-S as an Experimental Model

Add the validated EfficientNetV2-S ONNX model to the shared model registry.

Preserve existing model configurations and the default EfficientNet-B0 inference entry point.

Mark EfficientNetV2-S as experimental because it currently exceeds the reference CPU latency and model size limits.

In [ ]:
from pathlib import Path
import json
import shutil

PROJECT_DIR = Path("/content/FlowerCompetition")

REGISTRY_PATH = PROJECT_DIR / "model_registry.json"
BACKUP_PATH = PROJECT_DIR / "model_registry_before_v2s.json"
ONNX_PATH = PROJECT_DIR / "onnx/efficientnetv2_s_fp32.onnx"

print("=== STEP 90: REGISTER EFFICIENTNETV2-S ===")

assert REGISTRY_PATH.is_file()
assert ONNX_PATH.is_file()

registry = json.loads(
    REGISTRY_PATH.read_text(encoding="utf-8")
)

# Preserve the original registry before modification.
if not BACKUP_PATH.exists():
    shutil.copy2(REGISTRY_PATH, BACKUP_PATH)

model_name = "efficientnetv2_s_onnx"

new_config = {
    "architecture": "efficientnet_v2_s",
    "backend": "onnxruntime",
    "checkpoint": "onnx/efficientnetv2_s_fp32.onnx",
    "num_classes": 102,
    "image_size": 224,
    "preprocessing": "imagenet_resize256_centercrop224",
    "class_mapping": "class_to_idx.json",
    "enabled": True,
    "experimental": True
}

if model_name in registry:
    assert registry[model_name] == new_config, (
        "Existing registration differs. Review before overwriting."
    )
    print("Model already registered: PASS")
else:
    registry[model_name] = new_config

    REGISTRY_PATH.write_text(
        json.dumps(registry, indent=4),
        encoding="utf-8"
    )

print("\n=== REGISTERED MODELS ===")

for name in registry:
    print("-", name)

assert "efficientnet_b0_onnx" in registry
assert "mobilenetv3_onnx" in registry
assert model_name in registry

predict_path = PROJECT_DIR / "predict.py"

predict_code = predict_path.read_text(encoding="utf-8")

assert 'DEFAULT_MODEL="efficientnet_b0_onnx"' in predict_code or (
    "DEFAULT_MODEL = \"efficientnet_b0_onnx\"" in predict_code
), "Default model changed unexpectedly!"

print("\n=== FINAL STATUS ===")
print("EfficientNetV2-S registration: PASS")
print("Default model: efficientnet_b0_onnx")
print("Existing model configurations: preserved")
print("Google Drive backups: unchanged")

=== STEP 90: REGISTER EFFICIENTNETV2-S ===

=== REGISTERED MODELS ===
- mobilenetv3_pytorch
- mobilenetv3_onnx
- efficientnet_b0_onnx
- efficientnetv2_s_onnx

=== FINAL STATUS ===
EfficientNetV2-S registration: PASS
Default model: efficientnet_b0_onnx
Existing model configurations: preserved
Google Drive backups: unchanged


### Step 90.1 — Restore Missing Shared Project Files

Restore missing shared inference system files from the verified deployment v2 backup.

Preserve the newly exported EfficientNetV2-S ONNX model and avoid overwriting existing files.

Verify that the model registry and shared prediction modules are available before registering the teammate model.

In [ ]:
from pathlib import Path
import zipfile
import json

PROJECT_DIR = Path("/content/FlowerCompetition")

BACKUP_PATH = Path(
    "/content/drive/MyDrive/Flower_Competition/"
    "FlowerCompetition_IntegratedDeployment_v2.zip"
)

V2S_PATH = (
    PROJECT_DIR / "onnx/efficientnetv2_s_fp32.onnx"
)

print("=== STEP 90.1: RESTORE SHARED PROJECT ===")

assert BACKUP_PATH.is_file(), "Deployment v2 backup not found."
assert V2S_PATH.is_file(), "EfficientNetV2-S ONNX file not found."

v2s_size_before = V2S_PATH.stat().st_size

restored = []
preserved = []

with zipfile.ZipFile(BACKUP_PATH, "r") as archive:

    assert archive.testzip() is None

    for member in archive.infolist():

        if member.is_dir():
            continue

        relative = Path(member.filename)

        # Reject unsafe archive paths.
        assert not relative.is_absolute()
        assert ".." not in relative.parts

        destination = PROJECT_DIR / relative

        if destination.exists():
            preserved.append(str(relative))
            continue

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with archive.open(member) as source:
            with destination.open("xb") as target:
                import shutil
                shutil.copyfileobj(source, target)

        restored.append(str(relative))

assert V2S_PATH.stat().st_size == v2s_size_before

required = [
    "model_registry.json",
    "class_to_idx.json",
    "shared_model_loader.py",
    "predict_shared.py",
    "predict.py"
]

for filename in required:
    assert (PROJECT_DIR / filename).is_file(), filename

registry = json.loads(
    (PROJECT_DIR / "model_registry.json").read_text(
        encoding="utf-8"
    )
)

print("Files restored:", len(restored))
print("Existing files preserved:", len(preserved))

print("\n=== REGISTERED MODELS ===")
for name in registry:
    print("-", name)

print("\n=== FINAL STATUS ===")
print("Shared project restoration: PASS")
print("EfficientNetV2-S ONNX preserved: PASS")
print("Google Drive backup: unchanged")

=== STEP 90.1: RESTORE SHARED PROJECT ===
Files restored: 40
Existing files preserved: 0

=== REGISTERED MODELS ===
- mobilenetv3_pytorch
- mobilenetv3_onnx
- efficientnet_b0_onnx

=== FINAL STATUS ===
Shared project restoration: PASS
EfficientNetV2-S ONNX preserved: PASS
Google Drive backup: unchanged


### Step 90.2 — Register EfficientNetV2-S in the Shared Model Registry

Register the validated EfficientNetV2-S ONNX model as an experimental option.

Preserve the existing model configurations and keep EfficientNet-B0 as the default inference model.

Verify that the registration is compatible with the shared model loader.

In [ ]:
from pathlib import Path
import json
import shutil

PROJECT_DIR = Path("/content/FlowerCompetition")
REGISTRY_PATH = PROJECT_DIR / "model_registry.json"
BACKUP_PATH = PROJECT_DIR / "model_registry_before_v2s.json"
ONNX_PATH = PROJECT_DIR / "onnx/efficientnetv2_s_fp32.onnx"

print("=== STEP 90.2: REGISTER EFFICIENTNETV2-S ===")

assert REGISTRY_PATH.is_file()
assert ONNX_PATH.is_file()

registry = json.loads(
    REGISTRY_PATH.read_text(encoding="utf-8")
)

# Save the original registry once.
if not BACKUP_PATH.exists():
    shutil.copy2(REGISTRY_PATH, BACKUP_PATH)

model_name = "efficientnetv2_s_onnx"

new_config = {
    "architecture": "efficientnet_v2_s",
    "backend": "onnxruntime",
    "checkpoint": "onnx/efficientnetv2_s_fp32.onnx",
    "num_classes": 102,
    "image_size": 224,
    "preprocessing": "imagenet_resize256_centercrop224",
    "class_mapping": "class_to_idx.json",
    "enabled": True,
    "experimental": True
}

if model_name in registry:
    assert registry[model_name] == new_config, (
        "Existing model configuration differs."
    )
    print("Model already registered: PASS")
else:
    registry[model_name] = new_config

    REGISTRY_PATH.write_text(
        json.dumps(registry, indent=4),
        encoding="utf-8"
    )

print("\n=== REGISTERED MODELS ===")

for name in registry:
    print("-", name)

assert model_name in registry
assert "efficientnet_b0_onnx" in registry
assert "mobilenetv3_onnx" in registry

# Check that the default model has not changed.
predict_code = (
    PROJECT_DIR / "predict.py"
).read_text(encoding="utf-8")

assert (
    'DEFAULT_MODEL="efficientnet_b0_onnx"' in predict_code
    or 'DEFAULT_MODEL = "efficientnet_b0_onnx"' in predict_code
), "Default model changed unexpectedly!"

print("\n=== FINAL STATUS ===")
print("EfficientNetV2-S registration: PASS")
print("Default model: efficientnet_b0_onnx")
print("Original models: preserved")
print("Google Drive backups: unchanged")

=== STEP 90.2: REGISTER EFFICIENTNETV2-S ===
Model already registered: PASS

=== REGISTERED MODELS ===
- mobilenetv3_pytorch
- mobilenetv3_onnx
- efficientnet_b0_onnx
- efficientnetv2_s_onnx

=== FINAL STATUS ===
EfficientNetV2-S registration: PASS
Default model: efficientnet_b0_onnx
Original models: preserved
Google Drive backups: unchanged


### Step 91 — Verify EfficientNetV2-S Shared Inference

Run the registered EfficientNetV2-S ONNX model through the unified prediction interface.

Verify successful model loading, correct class mapping, and top-3 predictions on a known validation image.

Keep EfficientNet-B0 as the default model and do not modify model weights.

In [ ]:
from pathlib import Path
import subprocess
import sys
import json

PROJECT_DIR = Path("/content/FlowerCompetition")
IMAGE_PATH = Path("/content/flower_test_image.jpg")

print("=== STEP 91: SHARED EFFICIENTNETV2-S INFERENCE ===")

assert (PROJECT_DIR / "predict.py").is_file()
assert (PROJECT_DIR / "model_registry.json").is_file()
assert (
    PROJECT_DIR / "onnx/efficientnetv2_s_fp32.onnx"
).is_file()
assert IMAGE_PATH.is_file()

command = [
    sys.executable,
    "predict.py",
    "--image", str(IMAGE_PATH),
    "--model", "efficientnetv2_s_onnx",
    "--top-k", "3"
]

result = subprocess.run(
    command,
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True
)

print("\n=== INFERENCE OUTPUT ===")

if result.returncode != 0:
    print("STDERR:")
    print(result.stderr[-4000:])
    print("\nShared inference: FAILED")
    print("No files were modified.")
else:
    print(result.stdout)

    prediction = json.loads(result.stdout)

    assert prediction["model"] == "efficientnetv2_s_onnx"
    assert len(prediction["predictions"]) == 3

    top1 = prediction["predictions"][0]

    print("Predicted class:", top1["class_name"])
    print("Confidence:", round(top1["confidence"], 4))

    assert top1["class_name"] == "1"

    print("\n=== FINAL STATUS ===")
    print("Shared EfficientNetV2-S inference: PASS")
    print("Class mapping: PASS")
    print("Model weights: unchanged")
    print("Google Drive backups: unchanged")

=== STEP 91: SHARED EFFICIENTNETV2-S INFERENCE ===

=== INFERENCE OUTPUT ===
{
    "model": "efficientnetv2_s_onnx",
    "image": "/content/flower_test_image.jpg",
    "predictions": [
        {
            "class_name": "1",
            "class_index": 0,
            "confidence": 0.9208700060844421
        },
        {
            "class_name": "53",
            "class_index": 52,
            "confidence": 0.006595445331186056
        },
        {
            "class_name": "64",
            "class_index": 63,
            "confidence": 0.003263617167249322
        }
    ]
}

Predicted class: 1
Confidence: 0.9209

=== FINAL STATUS ===
Shared EfficientNetV2-S inference: PASS
Class mapping: PASS
Model weights: unchanged
Google Drive backups: unchanged


### Step 92 — Create Integrated Deployment Backup v3

Create a new deployment archive containing all three integrated ONNX models, the shared prediction system, and the model registry.

Verify archive integrity, required files, and SHA256 checksums.

Preserve all previous backups without overwriting them.

In [ ]:
from pathlib import Path
import zipfile
import hashlib
import shutil

PROJECT_DIR = Path("/content/FlowerCompetition")
DRIVE_DIR = Path("/content/drive/MyDrive/Flower_Competition")

PACKAGE_NAME = "FlowerCompetition_IntegratedDeployment_v3.zip"
LOCAL_ZIP = Path("/content") / PACKAGE_NAME
DRIVE_ZIP = DRIVE_DIR / PACKAGE_NAME

print("=== STEP 92: DEPLOYMENT BACKUP V3 ===")

assert PROJECT_DIR.is_dir()
assert DRIVE_DIR.is_dir()

required_files = [
    "predict.py",
    "predict_shared.py",
    "shared_model_loader.py",
    "evaluate_shared.py",
    "model_registry.json",
    "class_to_idx.json",
    "requirements.txt",
    "README.md",
    "onnx/mobilenetv3_fp32.onnx",
    "onnx/mobilenetv3_fp32.onnx.data",
    "onnx/efficientnet_b0_fp32.onnx",
    "onnx/efficientnetv2_s_fp32.onnx",
]

for filename in required_files:
    assert (PROJECT_DIR / filename).is_file(), (
        f"Missing required file: {filename}"
    )

assert not DRIVE_ZIP.exists(), (
    "V3 backup already exists. Refusing to overwrite."
)

exclude_dirs = {
    "__pycache__",
    ".git",
    ".venv",
    "data",
    "datasets",
    "checkpoints"
}

files_to_package = sorted(
    [
        path
        for path in PROJECT_DIR.rglob("*")
        if path.is_file()
        and not any(
            part in exclude_dirs
            for part in path.relative_to(PROJECT_DIR).parts
        )
        and path.suffix.lower() not in {".zip", ".ipynb"}
    ],
    key=lambda path: str(path.relative_to(PROJECT_DIR))
)

with zipfile.ZipFile(
    LOCAL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for path in files_to_package:
        archive.write(
            path,
            arcname=str(path.relative_to(PROJECT_DIR))
        )

with zipfile.ZipFile(LOCAL_ZIP, "r") as archive:
    assert archive.testzip() is None

    packaged = set(archive.namelist())

    for filename in required_files:
        assert filename in packaged

print("Required files: PASS")
print("ZIP integrity: PASS")
print("Files packaged:", len(files_to_package))
print(
    "Package size:",
    round(LOCAL_ZIP.stat().st_size / 1024**2, 2),
    "MiB"
)

def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()

shutil.copy2(LOCAL_ZIP, DRIVE_ZIP)

assert sha256(LOCAL_ZIP) == sha256(DRIVE_ZIP)

print("\n=== GOOGLE DRIVE BACKUP ===")
print("Saved:", DRIVE_ZIP)
print("SHA256:", sha256(DRIVE_ZIP))
print("Backup verification: PASS")
print("Previous backups: unchanged")

=== STEP 92: DEPLOYMENT BACKUP V3 ===
Required files: PASS
ZIP integrity: PASS
Files packaged: 43
Package size: 132.54 MiB

=== GOOGLE DRIVE BACKUP ===
Saved: /content/drive/MyDrive/Flower_Competition/FlowerCompetition_IntegratedDeployment_v3.zip
SHA256: e00a4fc04ab8a5fc6907e846d3423050a3ba5f6e9610a40b9d06f8ce126aa1b3
Backup verification: PASS
Previous backups: unchanged


### Step 93 — Restore the Integrated Project and Verify Test Data

Restore the latest integrated deployment backup from Google Drive.

Verify the MobileNetV3 model, shared evaluation code, and availability of the held-out test dataset.

Do not retrain models or modify checkpoints.

In [2]:
from pathlib import Path

for base in [Path("/content/drive"), Path("/content")]:
    for folder in [
        base / "MyDrive/Flower_Competition",
        base / "drive/MyDrive/Flower_Competition"
    ]:
        if folder.is_dir():
            print("FOUND:", folder)
            print("ZIP files:", [p.name for p in folder.glob("*.zip")])

FOUND: /content/drive/MyDrive/Flower_Competition
ZIP files: ['FlowerCompetition_FullBackup.zip', 'MobileNetV3_ONNX_Deployment.zip', 'FlowerCompetition_SharedSystem_Backup.zip', 'MobileNetV3_Accuracy_Experiments_Backup.zip', 'FlowerCompetition_IntegratedModels_Backup.zip', 'FlowerCompetition_IntegratedDeployment_v1.zip', 'FlowerCompetition_IntegratedDeployment_v2.zip', 'FlowerCompetition_IntegratedDeployment_v3.zip']
FOUND: /content/drive/MyDrive/Flower_Competition
ZIP files: ['FlowerCompetition_FullBackup.zip', 'MobileNetV3_ONNX_Deployment.zip', 'FlowerCompetition_SharedSystem_Backup.zip', 'MobileNetV3_Accuracy_Experiments_Backup.zip', 'FlowerCompetition_IntegratedModels_Backup.zip', 'FlowerCompetition_IntegratedDeployment_v1.zip', 'FlowerCompetition_IntegratedDeployment_v2.zip', 'FlowerCompetition_IntegratedDeployment_v3.zip']


### Step 93.3 — Restore Project and Test Dataset

Restore the latest integrated deployment and the original test dataset from existing Google Drive backups.

Verify all required files without retraining or modifying model weights.

In [3]:
from pathlib import Path
import zipfile

DRIVE = Path("/content/drive/MyDrive/Flower_Competition")
PROJECT = Path("/content/FlowerCompetition")

V3 = DRIVE / "FlowerCompetition_IntegratedDeployment_v3.zip"
FULL = DRIVE / "FlowerCompetition_FullBackup.zip"

assert V3.is_file()
assert FULL.is_file()

PROJECT.mkdir(parents=True, exist_ok=True)

print("=== RESTORING V3 DEPLOYMENT ===")

with zipfile.ZipFile(V3, "r") as z:
    assert z.testzip() is None
    z.extractall(PROJECT)

print("Deployment: PASS")

print("\n=== CHECKING FULL BACKUP STRUCTURE ===")

with zipfile.ZipFile(FULL, "r") as z:
    test_files = [
        name for name in z.namelist()
        if "/data/test/" in name
        and name.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print("Test images in backup:", len(test_files))
    print("Example:", test_files[0] if test_files else "NOT FOUND")

    if test_files:
        for name in test_files:
            relative = name.split("/data/test/", 1)[1]
            destination = PROJECT / "data/test" / relative
            destination.parent.mkdir(parents=True, exist_ok=True)

            if not destination.exists():
                with z.open(name) as source:
                    with open(destination, "wb") as target:
                        import shutil
                        shutil.copyfileobj(source, target)

test_dir = PROJECT / "data/test"

images = [
    p for p in test_dir.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
] if test_dir.exists() else []

print("\n=== FINAL STATUS ===")
print("MobileNetV3 ONNX:",
      (PROJECT / "onnx/mobilenetv3_fp32.onnx").is_file())
print("Evaluation script:",
      (PROJECT / "evaluate_shared.py").is_file())
print("Test images restored:", len(images))
print("Expected test images: 819")
print("Test dataset ready:", len(images) == 819)

=== RESTORING V3 DEPLOYMENT ===
Deployment: PASS

=== CHECKING FULL BACKUP STRUCTURE ===
Test images in backup: 819
Example: FlowerCompetition/data/test/8/image_03364.jpg

=== FINAL STATUS ===
MobileNetV3 ONNX: True
Evaluation script: True
Test images restored: 819
Expected test images: 819
Test dataset ready: True


### Step 94 — Evaluate MobileNetV3-Large on the Test Set

Evaluate the trained MobileNetV3-Large ONNX model on all 819 held-out test images.

Use the shared model loader, the original ImageNet preprocessing, and the numeric class mapping.

Report test accuracy, correct predictions, and total samples without modifying model weights.

In [6]:
from pathlib import Path
import sys
import time
import json

from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

PROJECT = Path("/content/FlowerCompetition")
TEST_DIR = PROJECT / "data/test"

sys.path.insert(0, str(PROJECT))

from shared_model_loader import SharedModel

print("=== STEP 94: MOBILENETV3 TEST ACCURACY ===")

model = SharedModel("mobilenetv3_onnx", device="cpu")

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class_to_idx = model.class_to_idx
idx_to_class = {
    int(index): str(name)
    for name, index in class_to_idx.items()
}

image_paths = sorted([
    p for p in TEST_DIR.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
])

assert len(image_paths) == 819
assert len(class_to_idx) == 102

correct = 0
total = 0
errors = []

start = time.perf_counter()

for path in tqdm(image_paths, desc="Evaluating test images"):

    true_class = path.parent.name
    assert true_class in class_to_idx

    with Image.open(path) as image:
        tensor = preprocess(
            image.convert("RGB")
        ).unsqueeze(0)

    logits = model.predict_logits(tensor)

    if hasattr(logits, "detach"):
        logits = logits.detach().cpu().numpy()

    import numpy as np

    predicted_index = int(np.asarray(logits).argmax())
    predicted_class = idx_to_class[predicted_index]

    total += 1

    if predicted_class == true_class:
        correct += 1
    else:
        errors.append({
            "image": str(path.relative_to(PROJECT)),
            "true_class": true_class,
            "predicted_class": predicted_class
        })

elapsed = time.perf_counter() - start
accuracy = 100.0 * correct / total

results = {
    "model": "mobilenetv3_onnx",
    "dataset": "Oxford Flowers 102",
    "split": "test",
    "total_images": total,
    "correct_predictions": correct,
    "incorrect_predictions": total - correct,
    "test_accuracy_percent": accuracy,
    "evaluation_time_seconds": elapsed,
    "errors": errors
}

output_path = PROJECT / "results/mobilenetv3_test_metrics.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

output_path.write_text(
    json.dumps(results, indent=4),
    encoding="utf-8"
)

print("\n=== OFFICIAL TEST RESULTS ===")
print("Test images:", total)
print("Correct:", correct)
print("Incorrect:", total - correct)
print(f"Test accuracy: {accuracy:.4f}%")
print(f"Evaluation time: {elapsed:.2f} seconds")
print("Results saved:", output_path)

print("\n=== FINAL STATUS ===")
print("MobileNetV3 test evaluation: PASS")
print("Model weights: unchanged")

=== STEP 94: MOBILENETV3 TEST ACCURACY ===


Evaluating test images:   0%|          | 0/819 [00:00<?, ?it/s]


=== OFFICIAL TEST RESULTS ===
Test images: 819
Correct: 777
Incorrect: 42
Test accuracy: 94.8718%
Evaluation time: 17.63 seconds
Results saved: /content/FlowerCompetition/results/mobilenetv3_test_metrics.json

=== FINAL STATUS ===
MobileNetV3 test evaluation: PASS
Model weights: unchanged


### Step 94.1 — Install ONNX Runtime

Install the ONNX Runtime CPU package required to evaluate the exported MobileNetV3 ONNX model.

No model retraining or checkpoint modification is required.

In [5]:
%pip install -q onnxruntime

### Step 95 — Back Up MobileNetV3 Test Results

Save the verified MobileNetV3 test metrics to Google Drive.

Preserve the original evaluation output and existing deployment backups.

In [7]:
from pathlib import Path
import shutil
import json

PROJECT = Path("/content/FlowerCompetition")
DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

SOURCE = PROJECT / "results/mobilenetv3_test_metrics.json"
DEST_DIR = DRIVE / "benchmark_results_v1"
DEST = DEST_DIR / "mobilenetv3_test_metrics.json"

assert SOURCE.is_file(), "Test results not found."
assert DRIVE.is_dir(), "Drive folder not accessible."

results = json.loads(SOURCE.read_text(encoding="utf-8"))

assert results["total_images"] == 819
assert results["correct_predictions"] == 777
assert abs(results["test_accuracy_percent"] - 94.8718) < 0.001

DEST_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(SOURCE, DEST)

assert json.loads(DEST.read_text(encoding="utf-8")) == results

print("=== STEP 95: TEST RESULTS BACKUP ===")
print("Saved:", DEST)
print(f"Test accuracy: {results['test_accuracy_percent']:.4f}%")
print("Backup verification: PASS")
print("Existing deployment ZIPs: unchanged")

=== STEP 95: TEST RESULTS BACKUP ===
Saved: /content/drive/MyDrive/Flower_Competition/benchmark_results_v1/mobilenetv3_test_metrics.json
Test accuracy: 94.8718%
Backup verification: PASS
Existing deployment ZIPs: unchanged


### Step 96 — Prepare MobileNetV3 Model Information

Generate a reproducible model information document for teammates.

Include model architecture, checkpoint information, preprocessing, class mapping, held-out test accuracy, CPU inference speed, and model size.

Save the document to Google Drive without modifying existing models.

In [9]:
from pathlib import Path
import json

PROJECT = Path("/content/FlowerCompetition")
DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

metrics = json.loads(
    (PROJECT / "results/mobilenetv3_test_metrics.json").read_text()
)

registry = json.loads(
    (PROJECT / "model_registry.json").read_text()
)

mapping = json.loads(
    (PROJECT / "class_to_idx.json").read_text()
)

onnx_files = [
    PROJECT / "onnx/mobilenetv3_fp32.onnx",
    PROJECT / "onnx/mobilenetv3_fp32.onnx.data"
]

assert all(p.is_file() for p in onnx_files)
assert len(mapping) == 102
assert all(mapping[str(i)] == i - 1 for i in range(1, 103))

size_mib = sum(p.stat().st_size for p in onnx_files) / (1024**2)

lines = [
    "# MobileNetV3-Large — Model Information",
    "",
    "## 1. Architecture",
    "",
    "- Model: torchvision MobileNetV3-Large",
    "- Output classes: 102",
    "- Framework: PyTorch / torchvision",
    "- Deployment: ONNX Runtime (CPU)",
    "- Original checkpoint: mobilenetv3_finetuned_best.pth",
    "- ONNX: onnx/mobilenetv3_fp32.onnx",
    "- External weights: onnx/mobilenetv3_fp32.onnx.data",
    "",
    "## 2. PyTorch Model Loading",
    "",
    "```python",
    "import torch",
    "import torch.nn as nn",
    "from torchvision import models",
    "",
    "model = models.mobilenet_v3_large(weights=None)",
    "model.classifier[3] = nn.Linear(",
    "    model.classifier[3].in_features, 102",
    ")",
    "",
    "# Load the checkpoint according to its saved structure.",
    "```",
    "",
    "## 3. ONNX Model Loading",
    "",
    "```python",
    "from shared_model_loader import SharedModel",
    'model = SharedModel("mobilenetv3_onnx", device="cpu")',
    "```",
    "",
    "## 4. Preprocessing",
    "",
    "- RGB input",
    "- Resize(256)",
    "- CenterCrop(224)",
    "- ToTensor()",
    "- Normalize mean: [0.485, 0.456, 0.406]",
    "- Normalize std: [0.229, 0.224, 0.225]",
    "",
    "## 5. Class Mapping",
    "",
    "- Class 1 -> index 0",
    "- Class 2 -> index 1",
    "- ...",
    "- Class 102 -> index 101",
    "- Full mapping: class_to_idx.json",
    "- Numeric ordering is used, not alphabetical ordering.",
    "",
    "## 6. Verified Results",
    "",
    "- Dataset: Oxford Flowers 102",
    f'- Test images: {metrics["total_images"]}',
    f'- Correct: {metrics["correct_predictions"]}',
    f'- Incorrect: {metrics["incorrect_predictions"]}',
    f'- Test accuracy: {metrics["test_accuracy_percent"]:.4f}%',
    "- Validation accuracy: 96.94% (793/818)",
    "- CPU ONNX inference: approximately 9.14 ms/image",
    "- CPU benchmark: ONNX Runtime, 1 thread",
    f"- ONNX deployment size: {size_mib:.3f} MiB",
    "",
    "CPU benchmark values were obtained in an earlier session.",
    "Hardware and runtime conditions can affect performance.",
    "",
    "## 7. Model Registry",
    "",
    "```json",
    json.dumps(registry["mobilenetv3_onnx"], indent=2),
    "```",
    "",
    "## 8. Evaluation Files",
    "",
    "- mobilenetv3_test_metrics.json",
    "- class_to_idx.json",
    "- shared_model_loader.py",
    "- predict.py",
    "",
    "The original PyTorch checkpoint must be shared separately.",
]

OUTPUT = DRIVE / "MobileNetV3_Model_Info.md"

OUTPUT.write_text(
    "\n".join(lines) + "\n",
    encoding="utf-8"
)

assert OUTPUT.is_file()
assert "94.8718%" in OUTPUT.read_text()

print("=== STEP 96.1: MODEL INFORMATION ===")
print("Document:", OUTPUT)
print(f'Test accuracy: {metrics["test_accuracy_percent"]:.4f}%')
print(f"ONNX size: {size_mib:.3f} MiB")
print("Class mapping: PASS")
print("Document creation: PASS")
print("Existing backups: unchanged")

=== STEP 96.1: MODEL INFORMATION ===
Document: /content/drive/MyDrive/Flower_Competition/MobileNetV3_Model_Info.md
Test accuracy: 94.8718%
ONNX size: 16.805 MiB
Class mapping: PASS
Document creation: PASS
Existing backups: unchanged


In [10]:
import torch.nn as nn
from torchvision import models

model = models.mobilenet_v3_large(weights=None)
model.classifier[3] = nn.Linear(
    model.classifier[3].in_features,
    102
)

### Step 97 — Locate the Original MobileNetV3 Checkpoint

Locate the original trained PyTorch checkpoint in the existing project and Google Drive backups.

Verify its availability without retraining or modifying model weights.

In [11]:
from pathlib import Path
import zipfile

DRIVE = Path("/content/drive/MyDrive/Flower_Competition")
PROJECT = Path("/content/FlowerCompetition")

names = {
    "mobilenetv3_finetuned_best.pth",
    "mobilenet_best.pth"
}

print("=== STEP 97: CHECKPOINT SEARCH ===")

found = []

for root in [PROJECT, DRIVE]:
    for name in names:
        for path in root.rglob(name):
            if path.is_file():
                found.append(path)
                print("FOUND:", path)
                print("Size:", round(path.stat().st_size / 1024**2, 3), "MiB")

print("\n=== SEARCHING EXISTING ZIP BACKUPS ===")

for zip_path in sorted(DRIVE.glob("*.zip")):
    with zipfile.ZipFile(zip_path) as archive:
        matches = [
            name for name in archive.namelist()
            if Path(name).name in names
        ]

        for match in matches:
            print("ZIP:", zip_path.name)
            print("CHECKPOINT:", match)

print("\n=== FINAL STATUS ===")
print("Standalone checkpoints found:", len(found))
print("No files modified.")

=== STEP 97: CHECKPOINT SEARCH ===
FOUND: /content/FlowerCompetition/mobilenetv3_finetuned_best.pth
Size: 16.731 MiB

=== SEARCHING EXISTING ZIP BACKUPS ===
ZIP: FlowerCompetition_FullBackup.zip
CHECKPOINT: FlowerCompetition/mobilenetv3_finetuned_best.pth
ZIP: FlowerCompetition_IntegratedDeployment_v1.zip
CHECKPOINT: mobilenetv3_finetuned_best.pth
ZIP: FlowerCompetition_IntegratedDeployment_v2.zip
CHECKPOINT: mobilenetv3_finetuned_best.pth
ZIP: FlowerCompetition_IntegratedDeployment_v3.zip
CHECKPOINT: mobilenetv3_finetuned_best.pth
ZIP: FlowerCompetition_IntegratedModels_Backup.zip
CHECKPOINT: mobilenetv3_finetuned_best.pth

=== FINAL STATUS ===
Standalone checkpoints found: 1
No files modified.


### Step 98 — Prepare MobileNetV3 Teammate Package

Create a compact package containing the original MobileNetV3 checkpoint, model architecture and preprocessing documentation, class mapping, and verified test metrics.

Save the package to Google Drive without modifying the existing deployment backups.

In [12]:
from pathlib import Path
import zipfile
import json

PROJECT = Path("/content/FlowerCompetition")
DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

OUTPUT = DRIVE / "MobileNetV3_Teammate_Package.zip"

files = {
    "mobilenetv3_finetuned_best.pth":
        PROJECT / "mobilenetv3_finetuned_best.pth",

    "MobileNetV3_Model_Info.md":
        DRIVE / "MobileNetV3_Model_Info.md",

    "class_to_idx.json":
        PROJECT / "class_to_idx.json",

    "mobilenetv3_test_metrics.json":
        PROJECT / "results/mobilenetv3_test_metrics.json",

    "shared_model_loader.py":
        PROJECT / "shared_model_loader.py",

    "model_registry.json":
        PROJECT / "model_registry.json",
}

for name, path in files.items():
    assert path.is_file(), f"Missing: {name}"

with zipfile.ZipFile(
    OUTPUT,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for name, path in files.items():
        archive.write(path, arcname=name)

with zipfile.ZipFile(OUTPUT, "r") as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("=== STEP 98: TEAMMATE PACKAGE ===")
print("Package:", OUTPUT)
print("Files:", len(files))
print("Size:", round(OUTPUT.stat().st_size / 1024**2, 3), "MiB")
print("ZIP integrity: PASS")
print("Existing deployment backups: unchanged")

=== STEP 98: TEAMMATE PACKAGE ===
Package: /content/drive/MyDrive/Flower_Competition/MobileNetV3_Teammate_Package.zip
Files: 6
Size: 15.47 MiB
ZIP integrity: PASS
Existing deployment backups: unchanged


### Step 99 — Final Model Performance Comparison

Consolidate the available accuracy, CPU inference latency, and model size results for all four models.

Distinguish held-out test accuracy from validation accuracy, and preserve benchmark methodology differences.

Export a comparison CSV and JSON report without retraining any model.

In [13]:
from pathlib import Path
import json
import csv

PROJECT = Path("/content/FlowerCompetition")
DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

mobile_metrics = json.loads(
    (RESULTS / "mobilenetv3_test_metrics.json")
    .read_text(encoding="utf-8")
)

models = [
    {
        "model": "EfficientNet-B0",
        "accuracy_percent": 99.02,
        "accuracy_split": "test",
        "accuracy_source": "teammate-reported",
        "cpu_inference_ms": 17.831,
        "cpu_source": "our ONNX benchmark",
        "model_size_mib": 15.776,
        "deployment": "ONNX",
    },
    {
        "model": "MobileNetV3-Large",
        "accuracy_percent": mobile_metrics["test_accuracy_percent"],
        "accuracy_split": "test",
        "accuracy_source": "our verified evaluation (819 images)",
        "cpu_inference_ms": 9.140,
        "cpu_source": "our ONNX benchmark",
        "model_size_mib": 16.805,
        "deployment": "ONNX",
    },
    {
        "model": "EfficientNetV2-S",
        "accuracy_percent": 98.17,
        "accuracy_split": "test",
        "accuracy_source": "teammate-reported",
        "cpu_inference_ms": 108.595,
        "cpu_source": "our ONNX benchmark",
        "model_size_mib": 77.324,
        "deployment": "ONNX",
    },
    {
        "model": "CLIP ViT-B/32",
        "accuracy_percent": 66.54,
        "accuracy_split": "test",
        "accuracy_source": "teammate-reported zero-shot",
        "cpu_inference_ms": 84.30,
        "cpu_source": "teammate-reported CPU benchmark",
        "model_size_mib": None,
        "deployment": "PyTorch / HuggingFace",
    },
]

csv_path = RESULTS / "final_model_comparison.csv"
json_path = RESULTS / "final_model_comparison.json"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=models[0].keys())
    writer.writeheader()
    writer.writerows(models)

json_path.write_text(
    json.dumps(models, indent=4, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 99: MODEL COMPARISON ===")

print(
    f'{"MODEL":<23} {"TEST ACC":>10} '
    f'{"CPU (ms)":>11} {"SIZE (MiB)":>12}'
)

for m in models:
    size = (
        f'{m["model_size_mib"]:.2f}'
        if m["model_size_mib"] is not None
        else "N/A"
    )

    print(
        f'{m["model"]:<23} '
        f'{m["accuracy_percent"]:>9.2f}% '
        f'{m["cpu_inference_ms"]:>10.2f} '
        f'{size:>12}'
    )

print("\n=== IMPORTANT NOTES ===")
print("All accuracy values refer to test results.")
print("CLIP uses zero-shot classification.")
print("CPU timings are not all measured on identical hardware.")
print("CLIP model size is not directly comparable in MiB.")

print("\nCSV:", csv_path)
print("JSON:", json_path)
print("Comparison generation: PASS")
print("Model weights: unchanged")

=== STEP 99: MODEL COMPARISON ===
MODEL                     TEST ACC    CPU (ms)   SIZE (MiB)
EfficientNet-B0             99.02%      17.83        15.78
MobileNetV3-Large           94.87%       9.14        16.80
EfficientNetV2-S            98.17%     108.59        77.32
CLIP ViT-B/32               66.54%      84.30          N/A

=== IMPORTANT NOTES ===
All accuracy values refer to test results.
CLIP uses zero-shot classification.
CPU timings are not all measured on identical hardware.
CLIP model size is not directly comparable in MiB.

CSV: /content/FlowerCompetition/results/final_model_comparison.csv
JSON: /content/FlowerCompetition/results/final_model_comparison.json
Comparison generation: PASS
Model weights: unchanged


### Step 100 — Back Up Final Model Comparison

Save the consolidated four-model comparison results to Google Drive.

Verify the copied files and preserve all existing deployment backups.

In [14]:
from pathlib import Path
import shutil
import hashlib

PROJECT = Path("/content/FlowerCompetition")
DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

SOURCE_DIR = PROJECT / "results"
DEST_DIR = DRIVE / "final_comparison"

DEST_DIR.mkdir(parents=True, exist_ok=True)

files = [
    "final_model_comparison.csv",
    "final_model_comparison.json",
]

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("=== STEP 100: FINAL COMPARISON BACKUP ===")

for name in files:
    source = SOURCE_DIR / name
    destination = DEST_DIR / name

    assert source.is_file(), f"Missing: {name}"

    shutil.copy2(source, destination)

    assert sha256(source) == sha256(destination)

    print(f"{name}: PASS")

print("\n=== FINAL STATUS ===")
print("Backup directory:", DEST_DIR)
print("Comparison backup: PASS")
print("Existing deployment ZIPs: unchanged")

=== STEP 100: FINAL COMPARISON BACKUP ===
final_model_comparison.csv: PASS
final_model_comparison.json: PASS

=== FINAL STATUS ===
Backup directory: /content/drive/MyDrive/Flower_Competition/final_comparison
Comparison backup: PASS
Existing deployment ZIPs: unchanged


### Step 101 — Independent Final Inference Test

Restore the integrated deployment into a clean directory and run the default EfficientNet-B0 model on a held-out test image.

Verify model loading, preprocessing, class mapping, and top-3 predictions without modifying model weights.

In [15]:
from pathlib import Path
import zipfile
import subprocess
import sys
import shutil
import json

DRIVE = Path("/content/drive/MyDrive/Flower_Competition")
PROJECT = Path("/content/FlowerCompetition")

BACKUP = DRIVE / "FlowerCompetition_IntegratedDeployment_v3.zip"
TEST_PROJECT = Path("/content/FlowerCompetition_FinalTest")

assert BACKUP.is_file()

if TEST_PROJECT.exists():
    shutil.rmtree(TEST_PROJECT)

TEST_PROJECT.mkdir(parents=True)

print("=== STEP 101: INDEPENDENT FINAL INFERENCE ===")

with zipfile.ZipFile(BACKUP, "r") as archive:
    assert archive.testzip() is None
    archive.extractall(TEST_PROJECT)

print("Clean deployment restored: PASS")

required = [
    "predict.py",
    "predict_shared.py",
    "shared_model_loader.py",
    "model_registry.json",
    "class_to_idx.json",
    "onnx/efficientnet_b0_fp32.onnx"
]

for name in required:
    assert (TEST_PROJECT / name).is_file(), name

print("Required deployment files: PASS")

test_images = sorted(
    (PROJECT / "data/test").rglob("*.jpg")
)

assert len(test_images) == 819

image_path = test_images[0]
expected_class = image_path.parent.name

command = [
    sys.executable,
    "-E",
    "predict.py",
    "--image", str(image_path),
    "--top-k", "3"
]

result = subprocess.run(
    command,
    cwd=TEST_PROJECT,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("STDERR:")
    print(result.stderr[-4000:])
    raise RuntimeError("Independent inference failed.")

prediction = json.loads(result.stdout)

assert prediction["model"] == "efficientnet_b0_onnx"
assert len(prediction["predictions"]) == 3

top1 = prediction["predictions"][0]

print("\n=== PREDICTION ===")
print("Image:", image_path.name)
print("Expected class:", expected_class)
print("Predicted class:", top1["class_name"])
print("Confidence:", round(top1["confidence"], 4))

print("\n=== FINAL STATUS ===")
print("Independent deployment: PASS")
print("Default model: efficientnet_b0_onnx")
print("Prediction interface: PASS")
print("Model weights: unchanged")

=== STEP 101: INDEPENDENT FINAL INFERENCE ===
Clean deployment restored: PASS
Required deployment files: PASS

=== PREDICTION ===
Image: image_06743.jpg
Expected class: 1
Predicted class: 1
Confidence: 0.9915

=== FINAL STATUS ===
Independent deployment: PASS
Default model: efficientnet_b0_onnx
Prediction interface: PASS
Model weights: unchanged


### Step 102 — Create Lightweight Final Submission

Build a minimal CPU inference package using EfficientNet-B0.

Include the unified prediction interface, model loader, preprocessing logic, class mapping, and the selected ONNX model.

Exclude unused model weights and preserve all existing project backups.

In [16]:
from pathlib import Path
import shutil
import zipfile
import json

PROJECT = Path("/content/FlowerCompetition")
DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

FINAL_DIR = Path("/content/FlowerCompetition_FinalSubmission")
OUTPUT_ZIP = DRIVE / "FlowerCompetition_FinalSubmission_B0.zip"

FINAL_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    "predict.py",
    "predict_shared.py",
    "shared_model_loader.py",
    "class_to_idx.json",
    "onnx/efficientnet_b0_fp32.onnx",
]

for name in required_files:
    source = PROJECT / name
    destination = FINAL_DIR / name

    assert source.is_file(), f"Missing: {name}"

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(source, destination)

# Keep only the selected model in the registry.
original_registry = json.loads(
    (PROJECT / "model_registry.json").read_text(
        encoding="utf-8"
    )
)

assert "efficientnet_b0_onnx" in original_registry

final_registry = {
    "efficientnet_b0_onnx":
        original_registry["efficientnet_b0_onnx"]
}

(FINAL_DIR / "model_registry.json").write_text(
    json.dumps(final_registry, indent=4),
    encoding="utf-8"
)

# Copy the existing dependency list.
requirements = PROJECT / "requirements.txt"

if requirements.is_file():
    shutil.copy2(
        requirements,
        FINAL_DIR / "requirements.txt"
    )

# Create a simple usage guide.
readme = [
    "# Oxford Flowers 102 — Final Inference Package",
    "",
    "Selected model: EfficientNet-B0 (ONNX)",
    "",
    "## Installation",
    "",
    "pip install -r requirements.txt",
    "",
    "## Prediction",
    "",
    "python predict.py --image path/to/flower.jpg",
    "",
    "## Notes",
    "",
    "- CPU inference is supported.",
    "- The model predicts one of 102 flower classes.",
    "- Class mapping is stored in class_to_idx.json.",
]

(FINAL_DIR / "README.md").write_text(
    "\n".join(readme) + "\n",
    encoding="utf-8"
)

# Create the final ZIP.
with zipfile.ZipFile(
    OUTPUT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for path in sorted(FINAL_DIR.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                arcname=str(
                    path.relative_to(FINAL_DIR)
                )
            )

with zipfile.ZipFile(OUTPUT_ZIP, "r") as archive:
    assert archive.testzip() is None

print("=== STEP 102: FINAL SUBMISSION PACKAGE ===")
print("ZIP:", OUTPUT_ZIP)
print(
    "Size:",
    round(OUTPUT_ZIP.stat().st_size / 1024**2, 3),
    "MiB"
)
print("ZIP integrity: PASS")
print("Selected model: EfficientNet-B0")
print("Other model weights: excluded")
print("Existing backups: unchanged")

=== STEP 102: FINAL SUBMISSION PACKAGE ===
ZIP: /content/drive/MyDrive/Flower_Competition/FlowerCompetition_FinalSubmission_B0.zip
Size: 14.647 MiB
ZIP integrity: PASS
Selected model: EfficientNet-B0
Other model weights: excluded
Existing backups: unchanged


### Step 103 — Validate the Final Submission Package

Extract the final EfficientNet-B0 submission ZIP into a clean directory.

Run the prediction interface independently, verify the expected output, and confirm that no unused model weights are included.

In [17]:
from pathlib import Path
import zipfile
import subprocess
import sys
import json
import tempfile

DRIVE = Path("/content/drive/MyDrive/Flower_Competition")
PROJECT = Path("/content/FlowerCompetition")

FINAL_ZIP = DRIVE / "FlowerCompetition_FinalSubmission_B0.zip"

assert FINAL_ZIP.is_file()

test_images = sorted(
    (PROJECT / "data/test").rglob("*.jpg")
)

assert len(test_images) == 819

image_path = test_images[0]
expected_class = image_path.parent.name

print("=== STEP 103: FINAL SUBMISSION VALIDATION ===")

with tempfile.TemporaryDirectory(
    prefix="flower_final_test_"
) as temp:

    clean_dir = Path(temp)

    with zipfile.ZipFile(FINAL_ZIP, "r") as archive:
        assert archive.testzip() is None
        archive.extractall(clean_dir)

    registry = json.loads(
        (clean_dir / "model_registry.json").read_text()
    )

    assert list(registry.keys()) == ["efficientnet_b0_onnx"]

    assert (
        clean_dir / "onnx/efficientnet_b0_fp32.onnx"
    ).is_file()

    assert not list(clean_dir.rglob("*.pth"))

    command = [
        sys.executable,
        "-E",
        "predict.py",
        "--image",
        str(image_path),
        "--top-k",
        "3"
    ]

    result = subprocess.run(
        command,
        cwd=clean_dir,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print("STDERR:")
        print(result.stderr[-4000:])
        raise RuntimeError("Final submission inference failed.")

    prediction = json.loads(result.stdout)

    assert prediction["model"] == "efficientnet_b0_onnx"
    assert len(prediction["predictions"]) == 3

    top1 = prediction["predictions"][0]

    print("ZIP extraction: PASS")
    print("Model registry: PASS")
    print("Unused checkpoints excluded: PASS")

    print("\n=== PREDICTION ===")
    print("Expected class:", expected_class)
    print("Predicted class:", top1["class_name"])
    print("Confidence:", round(top1["confidence"], 4))

    print("\n=== FINAL STATUS ===")
    print("Independent final ZIP inference: PASS")
    print("Selected model: EfficientNet-B0")
    print("Original model weights: unchanged")
    print("Existing backups: unchanged")

=== STEP 103: FINAL SUBMISSION VALIDATION ===
ZIP extraction: PASS
Model registry: PASS
Unused checkpoints excluded: PASS

=== PREDICTION ===
Expected class: 1
Predicted class: 1
Confidence: 0.9915

=== FINAL STATUS ===
Independent final ZIP inference: PASS
Selected model: EfficientNet-B0
Original model weights: unchanged
Existing backups: unchanged


### Steps 104–106 — Update CLIP Results and Back Up Team Deliverables

Update the CLIP CPU latency using the latest teammate-reported measurement (56.58 ms/image).

Save the revised comparison files to Google Drive and prepare a folder for the corrected CLIP submission.

Preserve the existing EfficientNet-B0 inference package and all model weights.

In [3]:
from pathlib import Path
import json
import csv
import shutil

PROJECT = Path("/content/FlowerCompetition")
DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

RESULTS = PROJECT / "results"
COMPARISON_DIR = DRIVE / "final_comparison"
TEAM_DIR = DRIVE / "Team_Final_Deliverables"
CLIP_DIR = TEAM_DIR / "Person3_CLIP"

COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
CLIP_DIR.mkdir(parents=True, exist_ok=True)

json_path = RESULTS / "final_model_comparison.json"
csv_path = RESULTS / "final_model_comparison.csv"

assert json_path.is_file(), "Comparison JSON missing"

models = json.loads(json_path.read_text(encoding="utf-8"))

clip_found = False

for model in models:
    if model["model"] == "CLIP ViT-B/32":
        model["cpu_inference_ms"] = 56.58
        model["cpu_source"] = "teammate-reported updated CPU benchmark"
        clip_found = True

assert clip_found, "CLIP entry not found"

# Save updated JSON and CSV.
json_path.write_text(
    json.dumps(models, indent=4, ensure_ascii=False),
    encoding="utf-8"
)

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=models[0].keys())
    writer.writeheader()
    writer.writerows(models)

# Back up comparison files.
for path in [json_path, csv_path]:
    shutil.copy2(path, COMPARISON_DIR / path.name)

# Look for the corrected CLIP ZIP in common Colab locations.
clip_candidates = [
    Path("/content/CLIP_3Kisi_Teslim(2).zip"),
    DRIVE / "CLIP_3Kisi_Teslim(2).zip",
    DRIVE / "Flower_Competition" / "CLIP_3Kisi_Teslim(2).zip",
]

clip_source = next(
    (p for p in clip_candidates if p.is_file()),
    None
)

if clip_source:
    destination = CLIP_DIR / clip_source.name
    shutil.copy2(clip_source, destination)
    clip_status = f"PASS — {destination}"
else:
    clip_status = "PENDING — upload corrected ZIP to Colab"

print("=== STEPS 104–106: FINAL UPDATE ===")

for model in models:
    print(
        f'{model["model"]:<23} '
        f'Accuracy: {model["accuracy_percent"]:>6.2f}% | '
        f'CPU: {model["cpu_inference_ms"]:>7.2f} ms'
    )

print("\nComparison JSON: PASS")
print("Comparison CSV: PASS")
print("Google Drive comparison backup: PASS")
print("Corrected CLIP ZIP:", clip_status)

print("\nEfficientNet-B0 final ZIP: unchanged")
print("Model weights: unchanged")

=== STEPS 104–106: FINAL UPDATE ===
EfficientNet-B0         Accuracy:  99.02% | CPU:   17.83 ms
MobileNetV3-Large       Accuracy:  94.87% | CPU:    9.14 ms
EfficientNetV2-S        Accuracy:  98.17% | CPU:  108.59 ms
CLIP ViT-B/32           Accuracy:  66.54% | CPU:   56.58 ms

Comparison JSON: PASS
Comparison CSV: PASS
Google Drive comparison backup: PASS
Corrected CLIP ZIP: PENDING — upload corrected ZIP to Colab

EfficientNet-B0 final ZIP: unchanged
Model weights: unchanged


### Step 104 — Restore and Update the Model Comparison

Load the previously backed-up comparison from Google Drive, update the CLIP CPU latency to 56.58 ms/image, and save the revised CSV and JSON files.

No model training or inference is required.

In [2]:
from pathlib import Path
import json
import csv
import shutil

DRIVE = Path("/content/drive/MyDrive/Flower_Competition")
BACKUP_DIR = DRIVE / "final_comparison"
TEAM_DIR = DRIVE / "Team_Final_Deliverables"
CLIP_DIR = TEAM_DIR / "Person3_CLIP"

json_path = BACKUP_DIR / "final_model_comparison.json"
csv_path = BACKUP_DIR / "final_model_comparison.csv"

print("=== STEP 104: RESTORE AND UPDATE ===")

assert json_path.is_file(), (
    f"Drive backup not found: {json_path}\n"
    "Check whether Google Drive is accessible in this session."
)

models = json.loads(json_path.read_text(encoding="utf-8"))

clip_found = False

for model in models:
    if model["model"] == "CLIP ViT-B/32":
        print("Previous CLIP latency:", model["cpu_inference_ms"], "ms")
        model["cpu_inference_ms"] = 56.58
        model["cpu_source"] = "teammate-reported updated CPU benchmark"
        clip_found = True

assert clip_found, "CLIP entry not found"

# Save updated comparison to Google Drive.
json_path.write_text(
    json.dumps(models, indent=4, ensure_ascii=False),
    encoding="utf-8"
)

with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=models[0].keys())
    writer.writeheader()
    writer.writerows(models)

# Restore local comparison files if the Colab project exists.
local_results = Path("/content/FlowerCompetition/results")
local_results.mkdir(parents=True, exist_ok=True)

shutil.copy2(json_path, local_results / json_path.name)
shutil.copy2(csv_path, local_results / csv_path.name)

# Prepare team submission folder.
CLIP_DIR.mkdir(parents=True, exist_ok=True)

print("\n=== UPDATED COMPARISON ===")

for model in models:
    print(
        f'{model["model"]:<23} '
        f'{model["accuracy_percent"]:>6.2f}% | '
        f'{model["cpu_inference_ms"]:>7.2f} ms'
    )

print("\nGoogle Drive backup: PASS")
print("Local comparison restored: PASS")
print("CLIP submission folder: READY")
print("Model weights: unchanged")

=== STEP 104: RESTORE AND UPDATE ===
Previous CLIP latency: 84.3 ms

=== UPDATED COMPARISON ===
EfficientNet-B0          99.02% |   17.83 ms
MobileNetV3-Large        94.87% |    9.14 ms
EfficientNetV2-S         98.17% |  108.59 ms
CLIP ViT-B/32            66.54% |   56.58 ms

Google Drive backup: PASS
Local comparison restored: PASS
CLIP submission folder: READY
Model weights: unchanged


In [4]:
from pathlib import Path
import shutil
import zipfile
import hashlib

source = Path("/content/CLIP_3Kisi_Teslim(2).zip")
destination_dir = Path(
    "/content/drive/MyDrive/Flower_Competition/"
    "Team_Final_Deliverables/Person3_CLIP"
)
destination = destination_dir / source.name

assert source.is_file(), "CLIP ZIP Colab'a yüklenmemiş."

with zipfile.ZipFile(source) as z:
    assert z.testzip() is None, "ZIP dosyası bozuk."

destination_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, destination)

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert sha256(source) == sha256(destination)

print("=== STEP 106: CLIP BACKUP ===")
print("ZIP:", destination)
print("ZIP integrity: PASS")
print("Google Drive backup: PASS")
print("Existing final B0 ZIP: unchanged")

AssertionError: CLIP ZIP Colab'a yüklenmemiş.

In [5]:
from google.colab import files
from pathlib import Path
import shutil
import zipfile
import hashlib

print("CLIP ZIP dosyasını seç:")
uploaded = files.upload()

source = next(
    (Path("/content") / name
     for name in uploaded
     if name.lower().endswith(".zip")),
    None
)

assert source is not None, "ZIP dosyası seçilmedi."

destination_dir = Path(
    "/content/drive/MyDrive/Flower_Competition/"
    "Team_Final_Deliverables/Person3_CLIP"
)
destination_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(source) as z:
    assert z.testzip() is None, "ZIP bütünlük hatası."

destination = destination_dir / source.name
shutil.copy2(source, destination)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(source) == sha256(destination)

print("\n=== STEP 106: CLIP BACKUP ===")
print("File:", destination)
print("ZIP integrity: PASS")
print("Google Drive backup: PASS")
print("EfficientNet-B0 final ZIP: unchanged")

CLIP ZIP dosyasını seç:


Saving CLIP_3Kisi_Teslim.zip to CLIP_3Kisi_Teslim.zip

=== STEP 106: CLIP BACKUP ===
File: /content/drive/MyDrive/Flower_Competition/Team_Final_Deliverables/Person3_CLIP/CLIP_3Kisi_Teslim.zip
ZIP integrity: PASS
Google Drive backup: PASS
EfficientNet-B0 final ZIP: unchanged


In [6]:
from pathlib import Path
import zipfile
import json

zip_path = Path(
    "/content/drive/MyDrive/Flower_Competition/"
    "Team_Final_Deliverables/Person3_CLIP/CLIP_3Kisi_Teslim.zip"
)

with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None
    names = z.namelist()

    print("=== CLIP ZIP CONTENT CHECK ===")
    print("Total entries:", len(names))

    for name in names:
        if any(key in name.lower() for key in [
            "clip.ipynb",
            "generalization_tests.py",
            "generalization_report",
            "comparison",
            "attribution.json"
        ]):
            print(name)

    # Search for the updated CPU benchmark in text files.
    matches = []
    for name in names:
        if name.lower().endswith((".md", ".csv", ".json")):
            try:
                content = z.read(name).decode("utf-8")
                if "56.58" in content or "56,58" in content:
                    matches.append(name)
            except UnicodeError:
                pass

    print("\nUpdated CLIP latency (56.58 ms) found in:")
    print(matches if matches else "NOT FOUND")

    print("\nZIP integrity: PASS")

=== CLIP ZIP CONTENT CHECK ===
Total entries: 24
clip.ipynb
generalization_tests.py
results/GENERALIZATION_REPORT.md
results/comparison_chart.png
results/comparison_table.csv

Updated CLIP latency (56.58 ms) found in:
['results/clip_metrics.json', 'results/GENERALIZATION_REPORT.md', 'results/comparison_table.csv']

ZIP integrity: PASS


In [7]:
from pathlib import Path
import zipfile

DRIVE = Path("/content/drive/MyDrive/Flower_Competition")
FINAL_ZIP = DRIVE / "FlowerCompetition_FinalSubmission_B0.zip"
COMPARISON = DRIVE / "final_comparison"
OUTPUT = DRIVE / "Person2_GitHub_Files.zip"

assert FINAL_ZIP.is_file(), "Final inference ZIP bulunamadı."

code_files = [
    "predict.py",
    "predict_shared.py",
    "shared_model_loader.py",
    "model_registry.json",
    "class_to_idx.json",
]

with zipfile.ZipFile(OUTPUT, "w", zipfile.ZIP_DEFLATED) as out:
    with zipfile.ZipFile(FINAL_ZIP) as source:
        for name in code_files:
            assert name in source.namelist(), f"Eksik: {name}"
            out.writestr(name, source.read(name))

    for name in [
        "final_model_comparison.csv",
        "final_model_comparison.json",
    ]:
        path = COMPARISON / name
        assert path.is_file(), f"Eksik: {path}"
        out.write(path, f"results/{name}")

with zipfile.ZipFile(OUTPUT) as z:
    assert z.testzip() is None
    print("=== PERSON 2 GITHUB PACKAGE ===")
    for name in z.namelist():
        print("✓", name)

print("\nZIP:", OUTPUT)
print("GitHub transfer package: PASS")
print("Existing repository: unchanged")

=== PERSON 2 GITHUB PACKAGE ===
✓ predict.py
✓ predict_shared.py
✓ shared_model_loader.py
✓ model_registry.json
✓ class_to_idx.json
✓ results/final_model_comparison.csv
✓ results/final_model_comparison.json

ZIP: /content/drive/MyDrive/Flower_Competition/Person2_GitHub_Files.zip
GitHub transfer package: PASS
Existing repository: unchanged


In [8]:
from pathlib import Path
import zipfile

DRIVE = Path("/content/drive/MyDrive/Flower_Competition")

targets = ("mobilenet.ipynb", "evaluate.py")

print("=== STEP 115: MISSING FILE SEARCH ===")

found = []

# Search regular files
for path in DRIVE.rglob("*"):
    if path.is_file() and path.name.lower() in targets:
        found.append(("FILE", str(path)))

# Search ZIP archives without extracting them
for archive_path in DRIVE.rglob("*.zip"):
    try:
        with zipfile.ZipFile(archive_path) as archive:
            for name in archive.namelist():
                if Path(name).name.lower() in targets:
                    found.append((
                        "ZIP",
                        f"{archive_path} -> {name}"
                    ))
    except (zipfile.BadZipFile, OSError):
        pass

if found:
    for kind, location in found:
        print(f"[{kind}] {location}")
else:
    print("Neither file was found in the Drive backups.")

print("\nSearch completed.")

=== STEP 115: MISSING FILE SEARCH ===
[ZIP] /content/drive/MyDrive/Flower_Competition/FlowerCompetition_FullBackup.zip -> FlowerCompetition/evaluate.py

Search completed.


In [9]:
from pathlib import Path
import zipfile

DRIVE = Path("/content/drive/MyDrive/Flower_Competition")
BACKUP = DRIVE / "FlowerCompetition_FullBackup.zip"

OUTPUT_DIR = DRIVE / "Person2_GitHub_Recovery"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_NAME = "FlowerCompetition/evaluate.py"
DESTINATION = OUTPUT_DIR / "evaluate.py"

with zipfile.ZipFile(BACKUP) as z:
    assert SOURCE_NAME in z.namelist(), "evaluate.py not found"
    content = z.read(SOURCE_NAME)

DESTINATION.write_bytes(content)

# Verify Python syntax without executing the script.
compile(content, str(DESTINATION), "exec")

print("=== STEP 116: EVALUATE.PY RECOVERY ===")
print("Recovered:", DESTINATION)
print("File size:", len(content), "bytes")
print("Python syntax: PASS")
print("Existing files: unchanged")

=== STEP 116: EVALUATE.PY RECOVERY ===
Recovered: /content/drive/MyDrive/Flower_Competition/Person2_GitHub_Recovery/evaluate.py
File size: 11544 bytes
Python syntax: PASS
Existing files: unchanged


In [10]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive")

print("=== STEP 117: MOBILENET NOTEBOOK SEARCH ===")

notebooks = []

for path in ROOT.rglob("*.ipynb"):
    if any(word in path.name.lower() for word in [
        "mobile", "flower", "competition"
    ]):
        notebooks.append(path)

if notebooks:
    for path in notebooks:
        print("FOUND:", path)
else:
    print("No matching notebook found in Google Drive.")

print("\nSearch completed.")

=== STEP 117: MOBILENET NOTEBOOK SEARCH ===
FOUND: /content/drive/MyDrive/Colab Notebooks/Flower_Competition_01_Data_Analysis.ipynb
FOUND: /content/drive/MyDrive/Colab Notebooks/Flower_Competition_01_Data_Analysis_ipynb_adlı_not_defterinin_kopyası.ipynb
FOUND: /content/drive/MyDrive/Colab Notebooks/mobilenet_optimization_experiments.ipynb

Search completed.


In [11]:
import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "mobilenet_optimization_experiments.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

code_cells = [
    cell for cell in nb["cells"]
    if cell["cell_type"] == "code"
]

all_code = "\n".join(
    "".join(cell["source"])
    for cell in code_cells
).lower()

keywords = [
    "mobilenet_v3_large",
    "mobilenetv3",
    "train",
    "optimizer",
    "epoch",
    "torch.save",
    "onnx",
    "accuracy",
    "inference"
]

print("=== STEP 118: NOTEBOOK INSPECTION ===")
print("Notebook:", notebook_path.name)
print("Total cells:", len(nb["cells"]))
print("Code cells:", len(code_cells))

print("\nKeyword checks:")
for keyword in keywords:
    print(f"{keyword:22s}: {'FOUND' if keyword in all_code else 'NOT FOUND'}")

print("\nNotebook size:", round(notebook_path.stat().st_size / 1024, 2), "KB")

=== STEP 118: NOTEBOOK INSPECTION ===
Notebook: mobilenet_optimization_experiments.ipynb
Total cells: 234
Code cells: 122

Keyword checks:
mobilenet_v3_large    : FOUND
mobilenetv3           : FOUND
train                 : FOUND
optimizer             : FOUND
epoch                 : FOUND
torch.save            : FOUND
onnx                  : FOUND
accuracy              : FOUND
inference             : FOUND

Notebook size: 502.73 KB
